# Polytopia engine (v123)

This notebook holds everything built so far, running in one place:

1. **Save decoder**: reads iPhone `.state` saves (map, players, every action).
2. **Engine replay**: replays a game action by action and checks it against the saved checkpoints.
3. **Combat calculator**: the damage formula the replay confirmed.
4. **Map generator**: uses the numbers measured from real saves, with the wiki filling gaps.

**How to use:** *Runtime → Run all*. It works straight away with a bundled bot-war game (Massive, 16 players, turns 5, 11 and 15). To use your own saves, run the **Upload** cell; it accepts the iPhone text exports or raw `.state` files.

Everything is plain Python with no installs needed. The setup cells write the code to `/content/polytopia/`, so you can open and edit the files there.

## 1. Setup: write the code and data

In [ ]:
import os, sys, base64, zlib, json
ROOT = "/content/polytopia" if os.path.isdir("/content") else os.path.abspath("polytopia")
for d in ("verify", "engine", "sim", "data", "saves"): os.makedirs(os.path.join(ROOT, d), exist_ok=True)
os.chdir(ROOT)
for p in (os.path.join(ROOT, "engine"), os.path.join(ROOT, "verify"), os.path.join(ROOT, "sim")):
    if p not in sys.path: sys.path.insert(0, p)
print("working in", ROOT)

### Save-file decoder (`verify/state_v123.py`)

In [ ]:
%%writefile verify/state_v123.py
"""Python reader for Polytopia .state saves, written for game version 123 (iOS, Oct 2026).

The Go library polytopiamapmodelgo (MIT) documents the layout up to ~version 115; v123 adds/moves header bytes,
so the header is located by structure (map width/height followed by tile (0,0)) instead of fixed offsets.
Handles the double LZ4 wrapping seen in the iOS export (outer block whose content is another compressed save).

Usage: python3 -I state_v123.py <save.state | already-decompressed .decomp> <out.json>
Output schema = state2json's ("polytopia-state-json/1"), so analyze_map.py reads it unchanged."""
import struct, sys, json

def lz4_block_decompress(src, out_len):
    """Minimal LZ4 block decoder (no external lz4 module available here)."""
    dst = bytearray(); i = 0; n = len(src)
    while i < n:
        tok = src[i]; i += 1
        lit = tok >> 4
        if lit == 15:
            while True:
                b = src[i]; i += 1; lit += b
                if b != 255: break
        dst += src[i:i + lit]; i += lit
        if i >= n: break
        off = src[i] | (src[i + 1] << 8); i += 2
        ml = tok & 15
        if ml == 15:
            while True:
                b = src[i]; i += 1; ml += b
                if b != 255: break
        ml += 4; start = len(dst) - off
        for k in range(ml): dst.append(dst[start + k])
    return bytes(dst)

def unwrap(b):
    """Polytopia wrapper: 1 header byte (bits 6-7 = size of the length diff field: 2 or 4 bytes), diff, LZ4 block."""
    sz = (b[0] >> 6) & 3; sz = 4 if sz == 3 else sz
    if sz not in (2, 4): return None
    diff = int.from_bytes(b[1:1 + sz], "little")
    return lz4_block_decompress(b[1 + sz:], len(b) - 1 - sz + diff)

def decompress(raw):
    data = raw
    for _ in range(3):                      # unwrap until the payload starts with version (u32 == u32)
        v1, v2 = struct.unpack_from("<II", data, 0)
        if v1 == v2 and 90 <= v1 <= 200: return data
        data = unwrap(data)
    raise ValueError("could not find a decompressed save payload")

class R:
    def __init__(s, b, o=0): s.b, s.o = b, o
    def u8(s): v = s.b[s.o]; s.o += 1; return v
    def u16(s): v = struct.unpack_from("<H", s.b, s.o)[0]; s.o += 2; return v
    def i16(s): v = struct.unpack_from("<h", s.b, s.o)[0]; s.o += 2; return v
    def u32(s): v = struct.unpack_from("<I", s.b, s.o)[0]; s.o += 4; return v
    def i32(s): v = struct.unpack_from("<i", s.b, s.o)[0]; s.o += 4; return v
    def raw(s, n): v = s.b[s.o:s.o + n]; s.o += n; return v
    def vstr(s):
        n = s.u8()                          # 7-bit-encoded length (short names fit one byte)
        if n & 0x80: n = (n & 0x7f) | (s.u8() << 7)
        return s.raw(n).decode("utf-8", "replace")

UNIT_FMT = "<IBHIIiiiiHHHBBBH"                   # 42 bytes, per polytopiamapmodelgo docs
UNIT_KEYS = ["Id", "Owner", "UnitType", "FollowerUnitId", "LeaderUnitId", "cx", "cy", "hx", "hy", "Health", "PromotionLevel", "Experience", "Moved", "Attacked", "Flipped", "CreatedTurn"]

def read_unit(r):
    vals = struct.unpack_from(UNIT_FMT, r.b, r.o); r.o += struct.calcsize(UNIT_FMT)
    return dict(zip(UNIT_KEYS, vals))

def read_header(r):
    h = {"Version1": r.u32(), "Version2": r.u32(), "TotalActions": r.u16(), "CurrentTurn": r.u32(), "CurrentPlayerIndex": r.u8(),
         "MaxUnitId": r.u32(), "CurrentGameState": r.u8(), "Seed": r.i32(), "TurnLimit": r.u32(), "ScoreLimit": r.u32(),
         "WinByCapital": r.u8(), "Unknown6": list(r.raw(6)), "GameModeBase": r.u8(), "GameModeRules": r.u8()}
    h["MapName"] = r.vstr(); h["SquareSize"] = r.u32()
    h["Disabled"] = [r.u16() for _ in range(r.u16())]; h["Unlocked"] = [r.u16() for _ in range(r.u16())]
    h["Difficulty"] = r.u16(); h["Opponents"] = r.u32()
    # v123: the rest of the settings block changed size; locate width,height (= SquareSize) followed by tile (0,0)
    sq = h["SquareSize"]; pat = struct.pack("<HHII", sq, sq, 0, 0)
    k = r.b.find(pat, r.o)
    if k < 0 or k - r.o > 200: raise ValueError("map dimensions not found after header")
    h["SettingsTail"] = list(r.b[r.o:k]); r.o = k
    h["MapWidth"], h["MapHeight"] = r.u16(), r.u16()
    return h

def read_tile(r, version, ex, ey):
    t = {"WorldCoordinates": [r.u32(), r.u32()]}
    if t["WorldCoordinates"] != [ex, ey]: raise ValueError(f"tile ({ex},{ey}) misaligned at offset {r.o}: got {t['WorldCoordinates']}")
    t["Terrain"] = r.u16(); t["Climate"] = r.u16(); t["Altitude"] = r.i16(); t["Owner"] = r.u8(); t["Capital"] = r.u8()
    t["CapitalCoordinates"] = [r.i32(), r.i32()]
    t["ResourceExists"] = r.u8() == 1; t["ResourceType"] = r.u16() if t["ResourceExists"] else -1
    t["ImprovementExists"] = r.u8() == 1; t["ImprovementType"] = r.u16() if t["ImprovementExists"] else -1
    if t["ImprovementExists"]:
        d = {"Level": r.u16(), "FoundedTurn": r.u16(), "CurrentPopulation": r.i16(), "TotalPopulation": r.u16(), "Production": r.i16(),
             "BaseScore": r.i16(), "BorderSize": r.i16(), "UpgradeCount": r.i16(), "ConnectedPlayerCapital": r.u8()}
        if r.u8() == 1: d["CityName"] = r.vstr()
        d["FoundedTribe"] = r.u8(); d["CityRewards"] = [r.u16() for _ in range(r.u16())]
        d["RebellionFlag"] = r.u16()
        if d["RebellionFlag"]: d["RebellionBuffer"] = list(r.raw(2))
        t["ImprovementData"] = d
    t["Unit"] = None
    if r.u8() == 1:
        t["Unit"] = read_unit(r)
        if r.u8() == 1:
            t["PassengerUnit"] = read_unit(r); assert r.u8() == 0
            t["PassengerEffects"] = [r.u16() for _ in range(r.u16())]; r.raw(5)
        t["UnitEffects"] = [r.u16() for _ in range(r.u16())]; t["UnitDirection"] = list(r.raw(5))
    t["PlayerVisibility"] = list(r.raw(r.u8()))
    t["HasRoad"] = r.u8() == 1; t["HasWaterRoute"] = r.u8() == 1; t["TileSkin"] = r.u16(); t["Unknown"] = list(r.raw(2))
    if version >= 105:
        t["FloodedFlag"] = r.u8()
        if t["FloodedFlag"] == 1: t["FloodedValue"] = r.u32()
    return t

def read_map(r, version):
    h = read_header(r); W, H = h["MapWidth"], h["MapHeight"]
    tiles = [[read_tile(r, version, x, y) for x in range(W)] for y in range(H)]
    return h, tiles

def read_player(r, version=123):
    """Player record (polytopiamapmodelgo layout, aggressions array dropped since v114) + one trailing byte in v123
    (equal to the tribe id in every save seen so far)."""
    p = {"PlayerId": r.u8(), "Name": r.vstr(), "AccountId": r.vstr(), "AutoPlay": r.u8(), "StartTile": [r.i32(), r.i32()], "Tribe": r.u16(),
         "UnknownByte1": r.u8(), "DifficultyHandicap": r.u32()}
    if version < 114:
        p["Aggressions"] = [(r.u8(), r.i32()) for _ in range(r.u16())]
    p.update({"Currency": r.u32(), "Score": r.u32(), "UnknownInt2": r.u32(), "NumCities": r.u16()})
    p["AvailableTech"] = [r.u16() for _ in range(r.u16())]
    p["Encountered"] = [r.u8() for _ in range(r.u16())]
    p["Tasks"] = []
    for _ in range(r.i16()):
        tt = r.i16(); p["Tasks"].append([tt, list(r.raw(6 if tt in (1, 5, 9) else 2))])   # v123: task type 9 also carries a uint32
    p["Kills"], p["Losses"], p["TribesDestroyed"] = r.i32(), r.i32(), r.i32()
    p["OverrideColor"] = list(r.raw(4)); p["OverrideTribe"] = r.u8()
    p["UniqueImprovements"] = [r.u16() for _ in range(r.u16())]
    p["Diplomacy"] = []
    for _ in range(r.u16()):
        d = struct.unpack_from("<BBiBiiii", r.b, r.o); r.o += 23
        p["Diplomacy"].append(dict(zip(["PlayerId", "RelationState", "LastAttackTurn", "EmbassyLevel", "LastPeaceBrokenTurn", "FirstMeet", "EmbassyBuildTurn", "PreviousAttackTurn"], d)))
    p["DiplomacyMessages"] = [list(r.raw(2)) for _ in range(r.u16())]
    p["DestroyedByTribe"] = r.u8(); p["DestroyedTurn"] = r.u32(); p["UnknownBuffer2"] = list(r.raw(4))
    p["EndScore"] = r.i32(); p["PlayerSkin"] = r.u16(); p["UnknownBuffer3"] = list(r.raw(4))
    if version >= 123: p["V123TrailingByte"] = r.u8()
    return p

def read_players(r, version=123):
    return [read_player(r, version) for _ in range(r.u16())]

# Action payload sizes after the uint16 type (polytopiamapmodelgo, older versions). v123 differs: type 20 is 10 bytes
# (seen in every turn-0 save). Types not confirmed for v123 are flagged; parsing stops at the first unknown type and the
# rest is kept raw, so a played save tells us which sizes changed.
ACTION_SIZES = {1: 11, 2: 21, 3: 9, 4: 9, 5: 11, 6: 21, 7: 13, 8: 3, 9: 9, 11: 11, 13: 9, 14: 9, 15: 1, 16: 11, 17: 9, 18: 9,
                20: 1, 21: 9, 24: 9, 25: 9, 27: 10, 28: 3, 29: 10, 30: 10,
                38: 9}   # 38: player + tile, seen from Cymanti (v123; meaning TBD)
ACTION_CONFIRMED_V123 = {1, 2, 3, 5, 6, 7, 8, 11, 13, 14, 15, 16, 18, 20, 21, 24, 27, 28, 30, 38}   # confirmed by exact-end parses of saves 9847d452 and 6e1bf8e0
ACTION_NAMES = {1: "Build", 2: "Attack", 3: "Recover", 5: "Train", 6: "Move", 7: "CaptureCity", 8: "Research", 9: "DestroyImprovement",
                11: "CityReward", 13: "Promote", 14: "ExamineRuins", 15: "EndTurn", 16: "Upgrade", 20: "StartMatch?", 21: "CityLevelUp"}

def read_actions(r):
    n = r.u16(); out = []
    for i in range(n):
        if r.o + 2 > len(r.b): return out, {"stopped_at": i, "reason": "end of data"}
        t = r.u16()
        if t not in ACTION_SIZES: return out, {"stopped_at": i, "reason": f"unknown action type {t}", "raw_tail": r.b[r.o - 2:].hex()}
        out.append({"type": t, "name": ACTION_NAMES.get(t, f"type{t}"), "raw": r.raw(ACTION_SIZES[t]).hex(), "confirmed_v123": t in ACTION_CONFIRMED_V123})
    return out, None

def read_state(data):
    """Full v123 save: initial state, 7 bytes, current state, 2 bytes, actions. The current state's first header field is a
    32-bit hash (also stored in the StartMatch action) rather than the version."""
    r = R(data); version = struct.unpack_from("<I", data, 0)[0]
    h0, t0 = read_map(r, version); p0 = read_players(r, version)
    gap = r.raw(7).hex()
    h1, t1 = read_map(r, version); p1 = read_players(r, version)
    pad = r.raw(2).hex()
    actions, problem = read_actions(r)
    trailer = None
    if problem is None and len(data) - r.o == 9:   # v123 trailer: 3 zero bytes, the state hash, the action count
        z, hsh, cnt = r.raw(3), r.u32(), r.u16(); trailer = {"hash": hsh, "action_count": cnt, "matches_list": cnt == len(actions)}
    return {"trailer": trailer,"version": version, "initial": {"header": h0, "tiles": t0, "players": p0}, "current": {"header": h1, "tiles": t1, "players": p1},
            "actions": actions, "actions_problem": problem, "gap_bytes": gap, "pad_bytes": pad, "unparsed_bytes": len(data) - r.o}

if __name__ == "__main__":
    raw = open(sys.argv[1], "rb").read(); data = decompress(raw); st = read_state(data)
    h = st["initial"]["header"]; cur = st["current"]["header"]
    out = {"format": "polytopia-state-json/2", "gameVersion": st["version"], "mapWidth": h["MapWidth"], "mapHeight": h["MapHeight"],
           "currentTurn": cur["CurrentTurn"], "header": {"MapHeaderInput": h}, "currentHeader": cur,
           "initialTiles": st["initial"]["tiles"], "initialPlayers": st["initial"]["players"],
           "currentTiles": st["current"]["tiles"], "currentPlayers": st["current"]["players"],
           "actions": st["actions"], "actionsProblem": st["actions_problem"], "unparsedBytes": st["unparsed_bytes"]}
    json.dump(out, open(sys.argv[2], "w"))
    print(f"OK v{st['version']} '{h['MapName']}' {h['MapWidth']}x{h['MapHeight']} turn {cur['CurrentTurn']} players {len(st['current']['players'])} "
          f"actions {len(st['actions'])} unparsed {st['unparsed_bytes']}")

### Engine: unit stats, game state, replay (`engine/`)

In [ ]:
%%writefile engine/stats.py
"""Unit stats for the engine. Priority: v123 in-game info cards (data/measured_ui.json) > calculator v116 config > nothing.
Keys are the game's UnitType names (game_enums.json, Polyfish ids)."""
import json, os
HERE = os.path.dirname(os.path.abspath(__file__)); ROOT = os.path.join(HERE, "..")

# calculator v116 table (amigobrewbrew/polytopiacalculatorfirebase-public src/config/v116.json), copied as data
V116 = {
 "Warrior": (10, 2, 2, ["fortify"]), "Rider": (10, 2, 1, ["fortify"]), "Archer": (10, 2, 1, ["fortify"]), "Defender": (15, 1, 3, ["fortify"]),
 "MindBender": (10, 0, 1, ["stiff"]), "Swordsman": (15, 3, 3, []), "Catapult": (10, 4, 0, ["stiff"]), "Knight": (10, 3.5, 1, ["fortify"]),
 "Cloak": (5, 0, 0.5, ["stiff"]), "Dagger": (10, 2, 2, ["surprise", "static"]), "Giant": (40, 5, 4, ["static"]),
 "Raft": (10, 0, 1, ["static", "stiff", "variableHp"]), "Scoutship": (10, 2, 1, ["static", "variableHp"]), "Rammership": (10, 3, 3, ["static", "variableHp"]),
 "Bomber": (10, 3, 2, ["splash", "static", "stiff", "variableHp"]), "Juggernaut": (40, 4, 4, ["stiff", "stomp", "static"]),
 "Dinghy": (5, 0, 0.5, ["static", "stiff"]), "Pirate": (10, 2, 2, ["surprise", "static"]), "Amphibian": (10, 2, 1, []), "Tridention": (10, 2.5, 1, []),
 "Shark": (10, 3.5, 2, ["surprise"]), "Puffer": (10, 4, 0, ["stiff"]), "Jelly": (20, 2, 2, ["stiff", "tentacles", "static"]), "Crab": (40, 4, 4, ["static"]),
 "Polytaur": (15, 3, 1, ["fortify", "static"]), "DragonEgg": (10, 0, 2, ["fortify", "static", "stiff"]), "BabyDragon": (15, 3, 3, ["static"]),
 "FireDragon": (20, 4, 3, ["splash", "static"]), "Mooni": (10, 0, 1, ["stiff"]), "IceArcher": (10, 1, 1, ["freeze", "fortify", "stiff"]),
 "BattleSled": (15, 3, 2, []), "IceFortress": (20, 4, 3, ["static"]), "Gaami": (30, 4, 3, ["static"]), "Hexapod": (5, 3, 1, []), "Kiton": (15, 1, 3, ["poison"]),
 "Phychi": (5, 0.7, 1, ["poison", "surprise", "double attack"]), "Raychi": (10, 3, 2, []), "Boomchi": (10, 3, 3, ["onlyExplode", "static", "stiff"]),
 "LivingIsland": (20, 4, 4, ["stomp", "static", "poison"]), "Shaman": (10, 1, 1, ["static"]), "Exida": (10, 3, 1, ["poison", "poisonSplash"]),
 "Doomux": (20, 3.5, 2, ["explode", "static"]), "Centipede": (20, 4, 3, ["static"]), "Segment": (10, 2, 2, ["onlyExplode", "static", "stiff"]),
 "InsectEgg": (10, 2, 3, ["onlyExplode", "static", "stiff"]), "Larva": (10, 2, 2, ["surprise", "static"]), "Moth": (10, 2, 0.1, ["poison", "static", "stiff"]),
}
# ranges (not in the calculator): info cards + dataset; default 1
RANGE = {"Archer": 2, "Catapult": 3, "Bomber": 3, "Scoutship": 2, "IceArcher": 2, "FireDragon": 2, "AshArcher": 2, "SteamFortress": 2, "Pirate": 1}
UI_NAME = {"Scout": "Scoutship", "Rammer": "Rammership"}   # info-card name -> enum name

def load():
    S = {}
    for k, (hp, a, d, sk) in V116.items():
        S[k] = {"hp": hp, "atk": a, "def": d, "skills": set(sk), "range": RANGE.get(k, 1), "src": "calc-v116"}
    p = os.path.join(ROOT, "data", "measured_ui.json")
    if os.path.exists(p):
        for k, v in json.load(open(p))["units"].items():
            k = UI_NAME.get(k, k); old = S.get(k, {"skills": set()})
            sk = {s.lower().replace("_", "") for s in v["skills"].split()}
            if "variableHp" in old["skills"]: sk.add("variableHp")
            S[k] = {"hp": v["hp"], "atk": v["attack"], "def": v["defense"], "skills": sk, "range": v["range"], "move": v["movement"], "src": "v123-ui"}
    return S

STATS = load()

In [ ]:
%%writefile engine/state.py
"""Game state for the engine, loaded from an ingested save JSON (verify/ingest_saves.py output)."""
import json, os, struct
from dataclasses import dataclass, field
HERE = os.path.dirname(os.path.abspath(__file__)); ROOT = os.path.join(HERE, "..")
E = json.load(open(os.path.join(ROOT, "verify", "game_enums.json")))
UNIT = {int(k): v for k, v in E["UnitType"].items()}
UNIT.update({46: "Mermaid",   # v123 id, measured: Aquarion's start unit (22/22 saves), 10 HP, trainable with no tech
             53: "Shark",   # v123 id, inferred: best replay fit (96 exact HP at turn 15 vs 90 for the others), 10 HP
             48: "Swordsmaid"})   # v123 id, measured (af181c8f): Aquarion only (45/45), first trained right after Smithery, 15 HP fresh
UNIT_ID = {v: k for k, v in UNIT.items()}
TERRAIN = {int(k): v for k, v in E["TerrainType"].items()}
TECH = {int(k): v for k, v in E["TechnologyType"].items()}
TRIBE = {int(k): v for k, v in E["TribeType"].items()}
START = json.load(open(os.path.join(ROOT, "data", "measured_start.json")))
BOATS = {"Raft", "Scoutship", "Rammership", "Bomber", "Juggernaut"}   # Juggernaut: a Giant at sea (af181c8f, 19/19 Juggernauts carry a Giant)

@dataclass
class Unit:
    id: int; owner: int; type: str; x: int; y: int; hp: float; max_hp: float
    promotion: int = 0; xp: int = 0; passenger: str = None; effects: list = field(default_factory=list)

@dataclass
class Player:
    id: int; tribe: str; techs: set; stars: int = 0

class State:
    def __init__(self, d, which="initial"):
        tiles = d["initialTiles"] if which == "initial" else d["currentTiles"]
        players = d["initialPlayers"] if which == "initial" else d["currentPlayers"]
        self.H = len(tiles); self.W = len(tiles[0]); self.tiles = tiles
        self.terrain = {}; self.owner = {}; self.city = {}; self.wall = set(); self.units = {}; self.at = {}; self.flood = {}; self.fungi = {}; self.port = set()
        for y, row in enumerate(tiles):
            for x, t in enumerate(row):
                self.terrain[(x, y)] = TERRAIN.get(t["Terrain"], t["Terrain"]); self.owner[(x, y)] = t["Owner"]
                if t.get("FloodedFlag"): self.flood[(x, y)] = t.get("FloodedValue", 1)   # 1 = Aquarion flood (land), 4 = Cymanti algae (water)
                if t.get("ImprovementExists") and t["ImprovementType"] == 37: self.fungi[(x, y)] = t["Owner"]   # Cymanti Fungi (poisons)
                if t.get("ImprovementExists") and t["ImprovementType"] == 8: self.port.add((x, y))
                if t.get("ImprovementExists") and t["ImprovementType"] == 1:            # village / city
                    self.city[(x, y)] = t["Owner"]
                    if 1 in (t.get("ImprovementData") or {}).get("CityRewards", []): self.wall.add((x, y))
                if t.get("Unit"):
                    self.add_unit_record(t["Unit"], t.get("PassengerUnit"), t.get("UnitEffects"))
        self.players = {}
        for p in players:
            tr = TRIBE.get(p["Tribe"], str(p["Tribe"]))
            techs = {TECH.get(t, t) for t in p["AvailableTech"]}
            if which == "initial":       # start techs are granted by StartMatch (measured_start.json)
                techs |= set(START.get(tr, {}).get("after_match_start", {}).get("techs", []))
            self.players[p["PlayerId"]] = Player(p["PlayerId"], tr, techs, p["Currency"])
        self.max_unit_id = max(self.units) if self.units else 0

    def add_unit_record(self, u, passenger=None, effects=None):
        from stats import STATS
        name = UNIT.get(u["UnitType"], f"unit{u['UnitType']}")
        un = Unit(u["Id"], u["Owner"], name, u["cx"], u["cy"], u["Health"] / 10, 0, u["PromotionLevel"], u["Experience"])
        if passenger:
            un.passenger = UNIT.get(passenger["UnitType"], str(passenger["UnitType"])); un.passenger_id = passenger["Id"]
            un.promotion = passenger["PromotionLevel"]   # MEASURED: the boat record's level is always 0; the passenger keeps veteran status (HP is the boat's)
        un.max_hp = max_hp(un); un.effects = effects or []
        self.units[un.id] = un; self.at[(un.x, un.y)] = un.id

def max_hp(u):
    from stats import STATS
    base = STATS.get(u.passenger if (u.type in BOATS and u.passenger) else u.type, {}).get("hp", 10)
    return base + (5 if u.promotion > 0 else 0)

In [ ]:
%%writefile engine/replay.py
"""Shadow replay: apply a save's action list to its turn-0 state and compare with saved checkpoints.

Usage: python3 -I replay.py <final save json> <checkpoint save json> [more checkpoints...]
Each checkpoint is an earlier export of the same game; its action count says where to stop and compare."""
import sys, os, json, struct, math, collections
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
from stats import STATS
from state import State, Unit, UNIT, TECH, TERRAIN, BOATS, max_hp

FMT = {1: "<BHII", 2: "<BIIIII", 3: "<BII", 4: "<BII", 5: "<BHII", 6: "<BIIIII", 7: "<BIII", 8: "<BH", 9: "<BII", 11: "<BIIH",
       13: "<BII", 14: "<BII", 15: "<B", 16: "<BHII", 17: "<BII", 18: "<BII", 20: "<B", 21: "<BII", 24: "<BII", 25: "<BII",
       27: "<BIIB", 28: "<BBB", 29: "<BIIB", 30: "<BIIB", 38: "<BII"}

LAND_UNITS = {"Warrior", "Rider", "Archer", "Defender", "MindBender", "Swordsman", "Catapult", "Knight", "Cloak", "Dagger", "Giant",
              "Polytaur", "Mooni", "IceArcher", "BattleSled", "IceFortress", "Gaami", "Hexapod", "Kiton", "Shaman", "Exida", "Doomux",
              "Centipede", "Segment", "Larva", "SteamWagon", "SteamFortress", "Pyro", "AshArcher"}   # Aquarion units are amphibious/water: never embark
WATER_UNITS = {"Shark", "Raychi", "Puffer", "Jelly", "Crab"}   # MEASURED: seen only on water, flooded tiles or cities (Mermaid walks on land)
RULES = {"retaliation_uses_bonus": True, "melee_moves_in_on_kill": True, "t18_disband": False, "t24_explode": True, "t38_boost": True,
         "algae_is_land": True, "flood_keeps_form": True, "resync_terrain": True, "kill_move_lands": True, "push_polyfish": True, "ruin_unit": True, "mooni_freeze": True, "water_units_stay": True, "shaman_convert": True, "shaman_damage": True,
         "poison": True, "attack_splash": True, "poison_recover_cures": True,
         "mooni_freezes": ("Water", "Ocean"), "freezers": ("Mooni",), "giant_juggernaut": True,
         "port_kill_embark": True, "algae_keeps_boats": True,
         "push_onto_port": True, "no_cities_no_units": True}
LOG = collections.Counter(); EVENTS = []

def dist(a, b): return max(abs(a[0] - b[0]), abs(a[1] - b[1]))

def stat(u):
    return STATS.get(u.type, {"hp": 10, "atk": 2, "def": 2, "skills": set(), "range": 1})

def defence_bonus(st, u):
    """1 normally; 1.5 on forest/mountain/water with the owner's tech, or in own city with Fortify; 4 behind city walls."""
    p = st.players.get(u.owner); techs = p.techs if p else set(); pos = (u.x, u.y); ter = st.terrain.get(pos)
    sk = stat(u)["skills"]
    if pos in st.city and st.city[pos] == u.owner and "fortify" in sk:
        return 4.0 if pos in st.wall else 1.5
    if ter == "Forest" and "Archery" in techs: return 1.5
    if ter == "Mountain" and "Climbing" in techs: return 1.5
    if ter in ("Water", "Ocean") and "Aquatism" in techs: return 1.5
    return 1.0

def rnd(x): return math.floor(x + 0.5 + 1e-9)

def fight(st, a, d):
    sa, sd = stat(a), stat(d)
    atk, dfn = sa["atk"] + (0.5 if getattr(a, "boosted", False) else 0), sd["def"]
    bonus = defence_bonus(st, d)
    if RULES["poison"] and POISON in d.effects: bonus *= 0.5   # Polyfish (v115 notes): poison halves defence, bonuses included
    af = atk * a.hp / a.max_hp; df = dfn * d.hp / d.max_hp * bonus; tot = af + df
    dmg = rnd(af / tot * atk * 4.5) if tot > 0 else 0
    ret = 0
    d_hp = d.hp - dmg
    if d_hp > 0 and "stiff" not in sd["skills"] and "surprise" not in sa["skills"] and dist((a.x, a.y), (d.x, d.y)) <= sd.get("range", 1):
        dfr = df if RULES["retaliation_uses_bonus"] else dfn * d.hp / d.max_hp
        ret = rnd(dfr / tot * dfn * 4.5) if tot > 0 else 0
    return dmg, ret, bonus

def remove(st, u):
    st.units.pop(u.id, None)
    if st.at.get((u.x, u.y)) == u.id: st.at.pop((u.x, u.y))

def place(st, u, x, y):
    if st.at.get((u.x, u.y)) == u.id: st.at.pop((u.x, u.y))
    u.x, u.y = x, y; st.at[(x, y)] = u.id

def unit_at(st, x, y):
    i = st.at.get((x, y)); return st.units.get(i) if i is not None else None

DIRS = [(0, -1), (1, -1), (1, 0), (1, 1), (0, 1), (-1, 1), (-1, 0), (-1, -1)]
def sgn(v): return (v > 0) - (v < 0)

def push_target(st, u, actor):
    """Polyfish's push rule (functions.rs calculate_pushable_position): keep going the way the unit last moved this turn (or
    toward its last ranged attack target), reversed for an enemy of the acting player; a unit that hasn't moved goes toward the
    map centre. A blocked tile is skipped for the next direction either side, nearer the centre first."""
    x, y = u.x, u.y; cx, cy = st.W // 2, st.H // 2
    atk, frm = getattr(u, "last_attack", None), getattr(u, "last_from", None)
    if atk is not None and stat(u).get("range", 1) > 1: d = (sgn(atk[0] - x), sgn(atk[1] - y))
    elif frm is not None and frm != (x, y): d = (sgn(x - frm[0]), sgn(y - frm[1]))
    else: d = None
    if d is not None and u.owner != actor: d = (-d[0], -d[1])
    if d is None or d == (0, 0):
        d = (sgn(cx - x), sgn(cy - y))
        if d == (0, 0): d = (0, 1)
    k = DIRS.index(d); order = [DIRS[k]]
    for i in range(1, 5):
        a, b = DIRS[(k - i) % 8], DIRS[(k + i) % 8]
        if a == b: order.append(a); continue
        da = (x + a[0] - cx) ** 2 + (y + a[1] - cy) ** 2; db = (x + b[0] - cx) ** 2 + (y + b[1] - cy) ** 2
        if db < da or (db == da and abs(x + b[0] - y - b[1]) < abs(x + a[0] - y - a[1])): order += [b, a]
        else: order += [a, b]
    for dx, dy in order:
        q = (x + dx, y + dy)
        if q not in st.terrain or q in st.at: continue
        if u.type in LAND_UNITS and is_water(st, *q) and not (RULES["push_onto_port"] and q in st.port and st.owner.get(q) == u.owner): continue   # MEASURED (af181c8f t15, t21): its own port takes it, and it boards
        if u.type not in LAND_UNITS and u.type in BOATS and not is_water(st, *q): continue
        return q
    return None

TEMP = [0]
def spawn(st, owner, name, x, y):
    """New unit whose real id the log doesn't give: temporary negative id until its first Move/Attack names it."""
    TEMP[0] -= 1; u = Unit(TEMP[0], owner, name, x, y, 0, 0); u.max_hp = max_hp(u); u.hp = u.max_hp
    old = unit_at(st, x, y)
    if old is not None:                          # forced spawn pushes the occupant to a neighbouring tile
        LOG["spawn on occupied tile (push)"] += 1
        q = push_target(st, old, owner) if RULES["push_polyfish"] else None
        if q is None and not RULES["push_polyfish"]:
            free = [(x + dx, y + dy) for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dx or dy) and (x + dx, y + dy) in st.terrain and (x + dx, y + dy) not in st.at]
            land = [q for q in free if st.terrain[q] not in ("Water", "Ocean")]
            q = (land or free or [None])[0]
        if q is not None: enter(st, old, x, y, *q); EVENTS.append(("push", old.id, (x, y), q))
        else: st.at.pop((x, y), None)
    st.units[u.id] = u; st.at[(x, y)] = u.id
    return u

RUIN = {}     # (owner, x, y) -> [(created turn, unit type, promotion)]: units a checkpoint shows standing where a ruin was examined
ORACLE = {}   # unit id -> (owner, type, promotion) from any saved checkpoint, for units spawned by events the log doesn't spell out

def summon(st, uid, owner, x, y):
    o = ORACLE.get(uid); name = o[1] if o else "Warrior"
    u = Unit(uid, owner, name, x, y, 0, 0); u.max_hp = max_hp(u); u.hp = u.max_hp
    st.units[uid] = u; st.at[(x, y)] = uid; st.max_unit_id = max(st.max_unit_id, uid); LOG["spawned unseen unit"] += 1
    return u

def locate(st, uid, x, y, owner):
    """Return the unit with this id, standing at (x, y). Fixes pushes and spawns the log doesn't record."""
    u = st.units.get(uid)
    if u is not None and (u.x, u.y) == (x, y): return u
    here = unit_at(st, x, y)
    if u is not None and (here is None or here.id == uid):
        place(st, u, x, y); LOG["teleported (push/unlogged move)"] += 1; return u
    if here is not None and here.owner == owner and (here.id < 0 or uid not in ORACLE):
        return rekey(st, here, uid)
    if here is not None:
        # someone else stands where the save says uid is: the engine's occupant must have been pushed away; keep it, but off-map
        LOG["occupant displaced"] += 1; st.at.pop((x, y), None)
    if u is not None:
        place(st, u, x, y); return u
    return summon(st, uid, owner, x, y)

def rekey(st, u, uid):
    """The save names units by id; if the unit on the tile carries a different id in the engine, adopt the save's id."""
    if u.id == uid: return u
    other = st.units.pop(uid, None)
    if other is not None and st.at.get((other.x, other.y)) == uid: st.at.pop((other.x, other.y))
    st.units.pop(u.id, None); u.id = uid; st.units[uid] = u; st.at[(u.x, u.y)] = uid; LOG["rekeyed"] += 1
    return u

def is_water(st, x, y):
    return st.terrain.get((x, y)) in ("Water", "Ocean") and not (RULES["algae_is_land"] and st.flood.get((x, y)) == 4)

def enter(st, u, x0, y0, x1, y1):
    """Step u from (x0, y0) onto (x1, y1): a land unit entering water boards a Raft, a boat reaching land drops its passenger."""
    ter = st.terrain.get((x1, y1)); ter0 = st.terrain.get((x0, y0))
    water = ("Water", "Ocean")
    if RULES["algae_is_land"]:                     # MEASURED (e15f392c): land units stand on Cymanti algae (flood value 4) unboarded
        if st.flood.get((x1, y1)) == 4: ter = "Field"
        if st.flood.get((x0, y0)) == 4: ter0 = "Field"
    if RULES["algae_keeps_boats"] and st.flood.get((x1, y1)) == 4 and u.type in BOATS:
        pass                                     # MEASURED (af181c8f t22, t25): a boat moving onto Cymanti algae stays a boat, passenger aboard
    elif RULES["flood_keeps_form"] and st.flood.get((x1, y1)) == 1 and ter not in water and (x1, y1) not in st.city:
        pass                                     # MEASURED: on Aquarion-flooded land a boat stays a boat and a land unit stays on foot (a city is plain land)
    elif ter in water and ter0 not in water and u.type in LAND_UNITS:
        u.passenger = u.type                         # a land unit stepping into water becomes a Raft; MEASURED (af181c8f): a Giant becomes a Juggernaut
        u.type = "Juggernaut" if RULES["giant_juggernaut"] and u.type == "Giant" else "Raft"
        if RULES.get("boat_new_id", True):            # MEASURED: the boat is a new unit id; the passenger keeps its own id
            u.passenger_id = u.id; TEMP[0] -= 1; st.units.pop(u.id, None)
            if st.at.get((u.x, u.y)) == u.id: st.at[(u.x, u.y)] = TEMP[0]
            u.id = TEMP[0]; st.units[u.id] = u
    elif ter not in water and u.type in BOATS and u.passenger:
        u.type = u.passenger; u.passenger = None    # disembark
        pid = getattr(u, "passenger_id", None)
        if pid is not None and RULES.get("boat_new_id", True) and pid not in st.units:   # MEASURED: the unit gets its old id back
            st.units.pop(u.id, None)
            if st.at.get((u.x, u.y)) == u.id: st.at[(u.x, u.y)] = pid
            u.id = pid; st.units[pid] = u; u.passenger_id = None
    place(st, u, x1, y1)

PREV = [None]
POISON = 1   # UnitEffect enum

def add_effect(u, e):
    if e not in u.effects: u.effects = list(u.effects) + [e]

def mooni_freeze(st, x, y):
    """MEASURED (e15f392c, 4/4 cases): right after a Mooni moves or captures, the water around it is ice, so land units walk on it.
    MEASURED (af181c8f): ocean freezes too (45 of 70 new ice tiles were ocean; 61 of the 70 were next to a Mooni or Gaami)."""
    if not RULES["mooni_freeze"]: return
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            q = (x + dx, y + dy)
            if st.terrain.get(q) in RULES["mooni_freezes"] and q not in st.at: st.terrain[q] = "Ice"

def apply(st, a, turn):
    t = a["type"]; b = bytes.fromhex(a["raw"]); f = FMT.get(t)
    v = struct.unpack(f, b) if f else (b[0],)
    pl = v[0]
    if PREV[0] == 15 and pl != 255:              # a new player's turn: their units' moved/attacked marks reset
        for u in st.units.values():
            if u.owner == pl: u.last_from = None; u.last_attack = None
    PREV[0] = t
    if t == 15 and RULES["poison"]:              # Polyfish process_end_turn_effects: Fungi poisons non-Cymanti units of other owners on it
        for q, owner in st.fungi.items():
            u = unit_at(st, *q)
            if u and u.owner != owner and st.players.get(u.owner) and st.players[u.owner].tribe != "Cymanti": add_effect(u, POISON)
    if t == 6:                                   # Move
        _, x0, y0, x1, y1, uid = v
        u = locate(st, uid, x0, y0, pl)
        enter(st, u, x0, y0, x1, y1); u.last_from = (x0, y0)
        if u.type in RULES["freezers"]: mooni_freeze(st, x1, y1)
    elif t == 2:                                 # Attack
        _, uid, ox, oy, tx, ty = v
        at = locate(st, uid, ox, oy, pl)
        df = unit_at(st, tx, ty)
        if df is None and RULES.get("recover_lost_defender", False):
            near = [u for u in st.units.values() if u.owner != pl and dist((u.x, u.y), (tx, ty)) <= 3 and st.at.get((u.x, u.y)) == u.id]
            if near:
                df = min(near, key=lambda u: dist((u.x, u.y), (tx, ty))); place(st, df, tx, ty); LOG["defender recovered by proximity"] += 1
        if at is None or df is None:
            LOG["attack: missing " + ("attacker" if at is None else "defender")] += 1; return
        if at.type == "MindBender" or (RULES["shaman_convert"] and at.type == "Shaman"):   # Convert (Shaman: Polyfish + e15f392c t9)
            if at.type == "Shaman" and RULES["shaman_damage"]:   # MEASURED (n=1, e15f392c t9): the Shaman also deals its normal damage, no retaliation
                dmg, _, _ = fight(st, at, df); df.hp -= dmg
                if df.hp <= 0: remove(st, df); return
            df.owner = at.owner; LOG["convert"] += 1; EVENTS.append(("convert", turn, at.type, df.type)); return
        dmg, ret, bonus = fight(st, at, df); at.boosted = False; at.last_attack = (tx, ty)
        if RULES["attack_splash"] and "splash" in stat(at)["skills"]:   # Polyfish: enemies next to the target take half their own damage, floored
            for e in [u for u in st.units.values() if u.owner != at.owner and u is not df and dist((u.x, u.y), (tx, ty)) == 1 and st.at.get((u.x, u.y)) == u.id]:
                sd = math.floor(fight(st, at, e)[0] / 2); e.hp -= sd
                if RULES["poison"] and "poison" in stat(at)["skills"]: add_effect(e, POISON)
                if e.hp <= 0: remove(st, e)
                LOG["splash"] += 1
        EVENTS.append(("attack", turn, at.id, at.type, at.hp, df.id, df.type, df.hp, bonus, dmg, ret, st.terrain.get((tx, ty))))
        df.hp -= dmg
        if RULES["poison"]:                      # Polyfish: a poison attacker poisons its target; a poison defender poisons a contact attacker
            if df.hp > 0 and "poison" in stat(at)["skills"]: add_effect(df, POISON)
            if df.hp > 0 and ret > 0 and "poison" in stat(df)["skills"] and dist((ox, oy), (tx, ty)) <= 1: add_effect(at, POISON)
        if df.hp <= 0:
            remove(st, df); at.xp += 1
            if RULES["melee_moves_in_on_kill"] and dist((ox, oy), (tx, ty)) == 1 and "stiff" not in stat(at)["skills"] and stat(at).get("range", 1) == 1:
                if not RULES["kill_move_lands"]: place(st, at, tx, ty)
                elif at.type in LAND_UNITS and is_water(st, tx, ty) and not (RULES["port_kill_embark"] and (tx, ty) in st.port and st.owner.get((tx, ty)) == at.owner):
                    pass                                 # a land unit doesn't follow a kill into water, except onto its own port, where it boards (MEASURED: 4/4 own port, 0/2 enemy port)
                elif RULES["water_units_stay"] and at.type in WATER_UNITS and not (is_water(st, tx, ty) or (tx, ty) in st.flood or (tx, ty) in st.city):
                    pass                                 # MEASURED (e15f392c t9): a Shark doesn't follow a kill onto dry land
                else:
                    enter(st, at, at.x, at.y, tx, ty)    # MEASURED (e15f392c t15): a boat moving in onto land drops its passenger
        else:
            at.hp -= ret
            if at.hp <= 0: remove(st, at)
    elif t == 5:                                 # Train
        _, ut, x, y = v
        spawn(st, pl, UNIT.get(ut, f"unit{ut}"), x, y)
    elif t == 16:                                # Upgrade (boat)
        _, ut, x, y = v
        u = unit_at(st, x, y)
        if u:                                    # an upgraded boat gets a new id in the save: give it a temp id until named
            TEMP[0] -= 1; st.units.pop(u.id, None); u.id = TEMP[0]; st.units[u.id] = u; st.at[(x, y)] = u.id
            u.type = UNIT.get(ut, f"unit{ut}")
    elif t == 13:                                # Promote: +5 max HP, full heal
        _, x, y = v; u = unit_at(st, x, y)
        if u: u.promotion += 1; u.max_hp = max_hp(u); u.hp = u.max_hp
    elif t == 3:                                 # Recover: +4 in own territory, +2 elsewhere
        _, x, y = v; u = unit_at(st, x, y)
        if u and RULES["poison_recover_cures"] and POISON in u.effects: u.effects = [e for e in u.effects if e != POISON]
        elif u: u.hp = min(u.max_hp, u.hp + (4 if st.owner.get((x, y)) == u.owner else 2))
    elif t == 1:                                 # Build (only unit-spawning builds matter for combat)
        _, imp, x, y = v
        if imp == 30: spawn(st, pl, "Polytaur", x, y); LOG["enchant animal"] += 1
        if imp == 8: st.port.add((x, y))
    elif t == 7 and RULES.get("territory", True):  # Capture: the city's tile and the 8 around it become the captor's (unowned or owned by the old owner)
        _, uid, x, y = v; old = st.city.get((x, y))
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                q = (x + dx, y + dy)
                if q in st.terrain and st.owner.get(q) in (0, old, None): st.owner[q] = pl
        st.city[(x, y)] = pl
        if RULES["no_cities_no_units"] and old and old != pl and old not in st.city.values():   # MEASURED (af181c8f t10): Kickoo lost its last city
            for u in [u for u in st.units.values() if u.owner == old]: remove(st, u)          # and its remaining units were gone by the next save
            LOG["tribe destroyed"] += 1
        cu = st.units.get(uid)
        if cu is not None and cu.type in RULES["freezers"]: mooni_freeze(st, x, y)
    elif t == 7:                                 # Capture village/city
        _, uid, x, y = v; st.city[(x, y)] = pl
    elif t == 8:                                 # Research
        _, tech = v
        if pl in st.players: st.players[pl].techs.add(TECH.get(tech, tech))
    elif t == 11:                                # City reward
        _, x, y, rw = v
        if rw == 1: st.wall.add((x, y))
        if rw == 5 and RULES.get("territory", True):   # Border growth: radius 2 around the city
            for dy in range(-2, 3):
                for dx in range(-2, 3):
                    q = (x + dx, y + dy)
                    if q in st.terrain and st.owner.get(q) in (0, None): st.owner[q] = pl
        if rw == 6:                              # Super unit (Giant), forced spawn
            spawn(st, pl, "Giant", x, y)
    elif t == 14 and RULES["ruin_unit"]:         # Examine ruins: the reward isn't logged; a unit reward (veteran Swordsman seen twice)
        _, x, y = v                              # spawns on the ruin tile and pushes the explorer. Read it back from the checkpoints.
        hit = [r for r in RUIN.get((pl, x, y), ()) if r[0] == turn]
        if hit:
            _, name, promo = hit[0]; u = spawn(st, pl, name, x, y)
            if promo: u.promotion = promo; u.max_hp = max_hp(u); u.hp = u.max_hp
            LOG["ruin unit"] += 1
        else: LOG["ruin (no unit)"] += 1
    elif t == 18 and RULES["t18_disband"]:       # Disband (inferred)
        _, x, y = v; u = unit_at(st, x, y)
        if u and u.owner == pl: remove(st, u); LOG["disband"] += 1
    elif t == 38 and RULES["t38_boost"]:         # Boost (Shaman): adjacent friendly units +0.5 attack until they attack
        _, x, y = v
        for u in list(st.units.values()):
            if u.owner == pl and dist((u.x, u.y), (x, y)) == 1: u.boosted = True
        LOG["boost"] += 1
    elif t == 24 and RULES["t24_explode"]:       # Explode: the unit dies; adjacent enemies take half damage (floored)
        _, x, y = v; u = unit_at(st, x, y)
        if u and u.owner == pl:
            sa = stat(u)
            for d in list(st.units.values()):
                if d.owner != pl and dist((d.x, d.y), (x, y)) == 1 and st.at.get((d.x, d.y)) == d.id:
                    af = sa["atk"] * u.hp / u.max_hp; df = stat(d)["def"] * d.hp / d.max_hp * defence_bonus(st, d); tot = af + df
                    dmg = math.floor(rnd(af / tot * sa["atk"] * 4.5) / 2) if tot > 0 else 0
                    d.hp -= dmg
                    if d.hp <= 0: remove(st, d)
            remove(st, u); LOG["explode"] += 1
    else:
        LOG[f"unhandled type {t}"] += 1

def snapshot_units(d):
    out = {}
    for y, row in enumerate(d["currentTiles"]):
        for x, t in enumerate(row):
            u = t.get("Unit")
            if u: out[u["Id"]] = (u["Owner"], UNIT.get(u["UnitType"], u["UnitType"]), x, y, u["Health"] / 10, u["PromotionLevel"])
    return out

def compare(st, d):
    snap = snapshot_units(d); mine = {u.id: (u.owner, u.type, u.x, u.y, u.hp, u.promotion) for u in st.units.values()}
    for tid in [i for i in mine if i < 0]:          # temp ids: pair with the saved unit on the same tile
        m = mine[tid]
        for i, sv in snap.items():
            if (sv[0], sv[2], sv[3]) == (m[0], m[2], m[3]) and i not in mine:
                mine[i] = mine.pop(tid); break
    res = collections.Counter(); bad = []
    for i, s in snap.items():
        m = mine.get(i)
        if m is None: res["missing in engine"] += 1; bad.append(("missing", i, s)); continue
        if m[1] != s[1] and not (m[1] in BOATS and s[1] in BOATS): res["type differs"] += 1; bad.append(("type", i, s, m))
        if (m[2], m[3]) != (s[2], s[3]): res["position differs"] += 1; bad.append(("pos", i, s, m))
        if m[0] != s[0]: res["owner differs"] += 1; bad.append(("owner", i, s, m))
        if abs(m[4] - s[4]) > 1e-6: res["HP differs"] += 1; bad.append(("hp", i, s, m))
        else: res["HP exact"] += 1
    for i in mine:
        if i not in snap: res["extra in engine (should be dead)"] += 1; bad.append(("extra", i, mine[i]))
    return res, bad

def add_ruin_oracle(d):
    for row in d["currentTiles"]:
        for t in row:
            u = t.get("Unit")
            if u and u["CreatedTurn"] == d["currentTurn"] - 1:   # still on its spawn tile: units can't move the turn they appear
                RUIN.setdefault((u["Owner"], u["cx"], u["cy"]), set()).add((u["CreatedTurn"], UNIT.get(u["UnitType"], u["UnitType"]), u["PromotionLevel"]))

def run(final, checkpoints):
    D = json.load(open(final)); st = State(D, "initial"); acts = D["actions"]
    for c in list(checkpoints) + [final]:
        cd = json.load(open(c))
        for i, v in snapshot_units(cd).items(): ORACLE.setdefault(i, (v[0], v[1], v[5]))
        add_ruin_oracle(cd)
    cps = sorted(((len(json.load(open(c))["actions"]), c) for c in checkpoints))
    turn = 0; k = 0; reports = []
    for i, a in enumerate(acts):
        while k < len(cps) and i == cps[k][0]:
            cd = json.load(open(cps[k][1])); res, bad = compare(st, cd); reports.append((cps[k][1], i, res, bad)); k += 1
            if RULES.get("resync_territory", True):     # borders are read back from each checkpoint (they are not under test here)
                for yy, row in enumerate(cd["currentTiles"]):
                    for xx, tt in enumerate(row): st.owner[(xx, yy)] = tt["Owner"]
            if RULES["resync_terrain"]:                  # ice (Mooni freeze, melts again), Aquarion flood and Cymanti algae change outside the log
                st.flood = {}; st.fungi = {}; st.port = set()
                for yy, row in enumerate(cd["currentTiles"]):
                    for xx, tt in enumerate(row):
                        if tt.get("ImprovementExists") and tt["ImprovementType"] == 37: st.fungi[(xx, yy)] = tt["Owner"]
                        if tt.get("ImprovementExists") and tt["ImprovementType"] == 8: st.port.add((xx, yy))
                        st.terrain[(xx, yy)] = TERRAIN.get(tt["Terrain"], tt["Terrain"])
                        if tt.get("FloodedFlag"): st.flood[(xx, yy)] = tt.get("FloodedValue", 1)
        if a["type"] == 15 and bytes.fromhex(a["raw"])[0] == 255: turn += 1
        apply(st, a, turn)
    if k < len(cps) or True:
        res, bad = compare(st, D); reports.append((final, len(acts), res, bad))
    return st, reports

if __name__ == "__main__":
    st, reps = run(sys.argv[1], sys.argv[2:])
    for path, i, res, bad in reps:
        print(os.path.basename(path), "after", i, "actions:", dict(res))
    print("log:", dict(LOG))

### Map generator (`sim/wiki_mapgen.py`)

In [ ]:
%%writefile sim/wiki_mapgen.py
"""Wiki-literal Polytopia map generator.

Every number that the fandom wiki's Map Generation page states is applied as written (marked WIKI below).
Where the wiki is silent, a parameter is still needed to draw a map; each of those is marked PARAM with
where its value came from, so they can be swapped or tested. Nothing here is taken from Polyfish's code
except the four land-ratio PARAMs the wiki gives no number for.

Wiki reading used for the resource table: each column (inner / outer) sums to 100% of LAND tiles in that
zone, so P(resource | terrain) = share / terrain share (e.g. inner fruit on field = 18/48 = 0.375).
Since update 2.0.58 the wiki says terrain follows these percentages "or as close as possible", so terrain
and resources are assigned as exact quotas per tribe biome per zone, not as independent coin flips.

Two modes (generate(..., mode=...)):
  "measured" (default): numbers counted from Jason's real game saves (data/measured_mapgen.json, built by
      verify/build_measured_tables.py) come first; the wiki fills in where the saves are thin. Each measured rate is
      blended with the wiki value as a prior worth PRIOR_N tiles, so a cell with 2,000 tiles follows the game and a
      cell with 10 tiles stays near the wiki. Every place the saves overrule the wiki is marked MEASURED.
  "wiki": the wiki-literal generator, kept so the two can still be compared.

Usage:  python3 -I wiki_mapgen.py --mode measured --type lakes --size 16 --tribes Imperius,Bardur,XinXi,Kickoo --seed 1 --out map.json
        (or import and call generate(...))"""
import math, random, json, argparse, os

MEASURED_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "data", "measured_mapgen.json")
MEASURED = json.load(open(MEASURED_PATH)) if os.path.exists(MEASURED_PATH) else None
PRIOR_N = {"terrain": 50, "resource": 30}     # wiki prior strength, in tiles

# ---------------- WIKI numbers ----------------
BASE = {"mountain": 0.14, "forest": 0.38, "field": 0.48}                                          # WIKI
SHARE = {  # WIKI resource table: share of land tiles in the zone
    "inner": {"fruit": 0.18, "crop": 0.18, "field_empty": 0.12, "animal": 0.19, "forest_empty": 0.19, "metal": 0.11, "mountain_empty": 0.03},
    "outer": {"fruit": 0.06, "crop": 0.06, "field_empty": 0.36, "animal": 0.06, "forest_empty": 0.32, "metal": 0.03, "mountain_empty": 0.11}}
FISH_SHARE = 0.50                                                                                  # WIKI: 50% of shallow water tiles
TRIBE_MOD = {  # WIKI tribe modifier table (multipliers); water modifiers are listed but not applied in the game (WIKI)
    "XinXi": {"mountain": 1.5, "metal": 1.5}, "Imperius": {"animal": 0.5, "fruit": 2.0}, "Bardur": {"fruit": 1.5, "crop": 0.0},
    "Oumaji": {"forest": 0.2, "animal": 0.2, "mountain": 0.5}, "Kickoo": {"mountain": 0.5, "fish": 1.5}, "Hoodrick": {"mountain": 0.5, "forest": 1.5},
    "Luxidoor": {}, "Vengir": {"metal": 2.0, "animal": 0.1, "fruit": 0.1, "fish": 0.1}, "Zebasi": {"mountain": 0.5, "forest": 0.5, "fruit": 0.5},
    "AiMo": {"mountain": 1.5, "crop": 0.1}, "Quetzali": {"fruit": 2.0, "crop": 0.1}, "Yadakk": {"mountain": 0.5, "forest": 0.5, "fruit": 1.5},
    "Aquarion": {"forest": 0.5}, "Elyrion": {"mountain": 0.5, "crop": 1.5}, "Cymanti": {"mountain": 1.2}}      # Cymanti: spores replace crops (WIKI)
GUARANTEE = {"Bardur": "animal", "Imperius": "fruit", "Kickoo": "fish", "Zebasi": "crop", "Elyrion": "animal", "Cymanti": "crop",
             "XinXi": "metal", "Oumaji": "fruit", "Hoodrick": "animal"}                             # WIKI: >=2 identical; others random tier-1
TIER1 = ["fish", "animal", "fruit"]                                                                # WIKI
PRE_TERRAIN = {"drylands": 0.0, "waterworld": 0.10}                                                # WIKI (others 0.30)
VILLAGE_TOTAL = {"drylands": 0.80}                                                                 # WIKI (others 0.60 "after terrain")
SUBURBS = (1, 2)                                                                                   # WIKI: 1-2 per capital, not drylands/waterworld

# ---------------- PARAMs (wiki gives no number) ----------------
PARAMS = {
    "land_ratio": {"drylands": 1.0, "lakes": 0.733, "continents": 0.468, "pangea": 0.513, "archipelago": 0.399, "waterworld": 0.151},
    "land_ratio_src": "MEASURED on Jason's 6 real Massive saves (one per type, 16 players, game v123); wiki gives no numbers. May differ by map size.",
    "village_total_reading": "A (default): 60% of max = TOTAL after terrain; B: 60% of the slots still free after terrain. C9 / Q4 decides",
    "max_villages": "(size//3)^2 grid slots with >=3 spacing, not on the edge (PolyEnv; wiki: 'not deterministic')",
    "capital_spot": "random tile inside the player's domain, >=2 from its border and the map edge (wiki: capitals never within 2 of a domain border)",
    "biome_owner": "each land tile takes the tribe of the nearest capital (Chebyshev; wiki: land is divided between tribes)",
    "shallow_rule": "water edge-adjacent to land = shallow, else ocean (PolyEnv)",
    "resource_area": "inner = distance 1 of any settlement, outer = distance 2; none farther (PolyEnv + wiki inner/outer wording)",
    "ruins_count": {121: 4, 196: 5, 256: 7, 324: 9, 400: 11, 900: 23},
    "ruins_count_src": "PolyEnv table (wiki gives no count); placement uses the wiki rules",
    "starfish_count": "water tiles // 25 (PolyEnv; wiki: any water tile, no count)",
    "noise_smoothing_passes": 2,
}

def dist(a, b, n): return max(abs(a % n - b % n), abs(a // n - b // n))
def nbrs(i, n, r=1):
    x, y = i % n, i // n
    return [(y + dy) * n + x + dx for dy in range(-r, r + 1) for dx in range(-r, r + 1)
            if (dx or dy) and 0 <= x + dx < n and 0 <= y + dy < n]
def orth(i, n):
    x, y = i % n, i // n
    return [(y + dy) * n + x + dx for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)) if 0 <= x + dx < n and 0 <= y + dy < n]

FOREST_RULE = "renormalised"   # "wiki_text" | "renormalised" (fitted to real saves, see ledger C12)

def biome_shares(tribe, rule=None):
    """Mountain first (WIKI). Forest: wiki text multiplies forest and gives the rest to field; real saves fit better when the
    forest multiplier re-weights forest against field instead: forest = fb*mod / (fb*mod + field_b) * (1 - mountain)."""
    rule = rule or FOREST_RULE
    m = TRIBE_MOD.get(tribe, {})
    mountain = BASE["mountain"] * m.get("mountain", 1.0)
    forest_before = BASE["forest"] + (BASE["mountain"] - mountain) * (BASE["forest"] / (1 - BASE["mountain"]))
    if rule == "wiki_text":
        forest = min(forest_before * m.get("forest", 1.0), 1 - mountain)
    else:
        fw = forest_before * m.get("forest", 1.0); field_b = 1 - mountain - forest_before
        forest = fw / (fw + field_b) * (1 - mountain)
    return {"mountain": mountain, "forest": forest, "field": 1 - mountain - forest}

def cond_probs(tribe, zone):
    """P(resource | terrain) from the wiki share table, times the tribe multiplier; field split capped at 100%."""
    s, m = SHARE[zone], TRIBE_MOD.get(tribe, {})
    fruit = s["fruit"] / BASE["field"] * m.get("fruit", 1.0)
    crop = s["crop"] / BASE["field"] * m.get("crop", 1.0)
    if fruit + crop > 1.0: fruit, crop = fruit / (fruit + crop), crop / (fruit + crop)          # PARAM: proportional cap
    return {"field": {"fruit": fruit, "crop": crop},
            "forest": {"animal": min(1.0, s["animal"] / BASE["forest"] * m.get("animal", 1.0))},
            "mountain": {"metal": min(1.0, s["metal"] / BASE["mountain"] * m.get("metal", 1.0))},
            "water": {"fish": min(1.0, FISH_SHARE * m.get("fish", 1.0))}}

# ---------------- MEASURED overrides (27 saves, game v123; ledger C10, AQUACROP, C9) ----------------
GUARANTEE_MEASURED = dict(GUARANTEE, AiMo="animal", Polaris="animal", Aquarion="fish", Quetzali="fruit", Yadakk="fruit")
# MEASURED: every non-Vengir tribe's pair is fixed (e.g. Ai-Mo 20/20 animal, Aquarion 22/22 fish); the wiki says "random tier-1".
LUXIDOOR_RING = [("field", "fruit"), ("field", "fruit"), ("field", "crop"), ("forest", "animal"), ("forest", "animal"),
                 ("mountain", "metal"), ("water", "fish"), ("water", "fish")]
# MEASURED: all 18 Luxidoor capitals had exactly this 8-tile ring (Field 3, Forest 2, Mountain 1, Water 2).
def village_share(map_type, mode):
    if mode == "measured" and MEASURED and map_type in MEASURED["by_map_type"]:
        return MEASURED["by_map_type"][map_type]["settlements_per_slot"]       # MEASURED: settlements (incl. capitals) / (n//3)^2
    return VILLAGE_TOTAL.get(map_type, 0.60)

def _blend(counts, prior, k):
    """Posterior mean: measured counts plus the wiki distribution as k pseudo-tiles. Labels not in prior get prior 0."""
    tot = sum(counts.values()); labels = set(prior) | set(counts)
    return {l: (counts.get(l, 0) + k * prior.get(l, 0.0)) / (tot + k) for l in labels}

def shares_for(tribe, mode="measured"):
    w = biome_shares(tribe)
    if mode != "measured" or not MEASURED or tribe not in MEASURED["terrain_counts"]: return w
    b = _blend(MEASURED["terrain_counts"][tribe], w, PRIOR_N["terrain"])
    return {k: b.get(k, 0.0) for k in ("mountain", "forest", "field")}

def probs_for(tribe, zone, mode="measured"):
    """P(resource | terrain) per zone. Measured mode: counted from the saves (water now includes AquaCrop, which the wiki
    never mentions, and covers deep ocean in the zone too), blended with the wiki prior."""
    w = cond_probs(tribe, zone)
    if mode != "measured" or not MEASURED: return w
    out = {}
    for ter, wter in (("field", "field"), ("forest", "forest"), ("mountain", "mountain"), ("water", "water")):
        prior = dict(w[wter]); prior["none"] = max(0.0, 1 - sum(prior.values()))
        if ter == "water": prior.setdefault("aquacrop", 0.0)
        cnt = dict(MEASURED["resource_counts"].get(f"{tribe}|{zone}|{ter}", {}))
        if ter == "water":                                                      # shallow + ocean pooled
            for k, v in MEASURED["resource_counts"].get(f"{tribe}|{zone}|ocean", {}).items(): cnt[k] = cnt.get(k, 0) + v
        cnt.pop("starfish", None)                                               # starfish placed separately
        if ter == "water":   # the wiki has no AquaCrop at all, so water shrinks toward the all-tribe measured rate instead
            pool = {}
            for key, v in MEASURED["resource_counts"].items():
                _, z, tt = key.split("|")
                if z == zone and tt in ("water", "ocean"):
                    for r, x in v.items():
                        if r != "starfish": pool[r] = pool.get(r, 0) + x
            tot = sum(pool.values()); prior = {r: x / tot for r, x in pool.items()}
            if "fish" in TRIBE_MOD.get(tribe, {}):                              # keep the wiki's fish multiplier as a prior
                f = min(prior["fish"] * TRIBE_MOD[tribe]["fish"], 0.95); scale = (1 - f) / (1 - prior["fish"])
                prior = {r: (f if r == "fish" else x * scale) for r, x in prior.items()}
        b = _blend(cnt, prior, PRIOR_N["resource"]); b.pop("none", None)
        out[ter] = {k: v for k, v in b.items() if v > 0}
    return out

def ring_template(rng, tribe, c, n, map_type=None):
    """MEASURED: one real capital ring of this tribe, rotated/reflected at random -> {tile: (terrain, resource)}.
    Rings from the same map type are used when there are any (a Drylands capital rarely touches water)."""
    tpls = (MEASURED or {}).get("capital_ring_templates", {}).get(tribe)
    if not tpls: return None
    same = [t for t in tpls if t["map_type"] == map_type]
    tpl = rng.choice(same or tpls)["tiles"]; sym = rng.randrange(8); out = {}
    for dx, dy, ter, r in tpl:
        if sym & 1: dx = -dx
        if sym & 2: dy = -dy
        if sym & 4: dx, dy = dy, dx
        out[(c // n + dy) * n + c % n + dx] = ("water" if ter in ("water", "ocean") else ter, r)
    return out

def smooth(field, n, passes, outside):
    for _ in range(passes):
        new = []
        for i in range(n * n):
            x, y = i % n, i // n; tot = 0; c = 0
            for dy in (-1, 0, 1):
                for dx in (-1, 0, 1):
                    xx, yy = x + dx, y + dy
                    tot += field[yy * n + xx] if 0 <= xx < n and 0 <= yy < n else outside; c += 1
            new.append(tot / c)
        field = new
    return field

def quota_assign(rng, tiles, probs):
    """Exact-quota assignment: round(p * len) tiles get each label ('as close as possible', WIKI 2.0.58)."""
    tiles = list(tiles); rng.shuffle(tiles); out = {}; k = 0
    for label, p in probs.items():
        x = p * len(tiles); c = int(x) + (1 if rng.random() < x - int(x) else 0)   # unbiased rounding (small zones)
        for t in tiles[k:k + c]: out[t] = label
        k += c
    return out

def generate(map_type="lakes", size=16, tribes=("Imperius", "Bardur"), seed=0, village_reading="A", mode="measured"):
    """village_reading A: 60% (80% drylands) of the max is the TOTAL after terrain; B: 60% of the slots still free after terrain."""
    rng = random.Random(seed); n = size; N = n * n; P = len(tribes)
    terrain = ["field"] * N; res = [None] * N; imp = [None] * N; cap_of = [0] * N
    # 1. capitals in domains
    g = math.ceil(math.sqrt(P)); span = n // g
    cells = list(range(g * g)); rng.shuffle(cells); caps = []
    for p, cell in enumerate(cells[:P]):
        cx, cy = cell % g, cell // g
        cand = [y * n + x for y in range(cy * span + 2, (cy + 1) * span - 2) for x in range(cx * span + 2, (cx + 1) * span - 2)
                if 2 <= x < n - 2 and 2 <= y < n - 2]
        c = rng.choice(cand); caps.append(c); imp[c] = "Village"; cap_of[c] = p + 1
    settle = list(caps)
    rings = {c: ring_template(rng, tribes[p], c, n, map_type) for p, c in enumerate(caps)} if mode == "measured" else {}
    rings = {c: r for c, r in rings.items() if r}
    def ok_spot(i): x, y = i % n, i // n; return 0 < x < n - 1 and 0 < y < n - 1 and all(dist(i, s, n) >= 3 for s in settle)
    max_v = (n // 3) ** 2
    share = village_share(map_type, mode)
    total = round(share * max_v) if village_reading == "A" else None
    cap_total = total if (mode == "measured" and total) else 10 ** 9      # MEASURED: earlier steps may not overshoot the total
    # 2. suburbs
    if map_type not in ("drylands", "waterworld"):
        for c in caps:
            want = rng.randint(*SUBURBS)
            cand = [i for i in nbrs(c, n, 4) if ok_spot(i)]; rng.shuffle(cand)
            for i in cand:
                if want and ok_spot(i) and len(settle) < cap_total: imp[i] = "Village"; settle.append(i); want -= 1
    # 3. pre-terrain villages up to 30% (10% waterworld, 0 drylands) of max
    pre = min(cap_total, len(settle) + round(PRE_TERRAIN.get(map_type, 0.30) * (max_v - len(settle))))   # 'a third of the REMAINING villages' (WIKI)
    cand = list(range(N)); rng.shuffle(cand)
    for i in cand:
        if len(settle) >= pre: break
        if ok_spot(i): imp[i] = "Village"; settle.append(i)
    # 4. land / water from smoothed noise (PARAM land ratio), settlement tiles forced land
    land = [True] * N
    ratio = PARAMS["land_ratio"][map_type]
    if mode == "measured" and MEASURED: ratio = MEASURED["by_map_type"].get(map_type, {}).get("land_ratio", ratio)   # MEASURED
    if ratio < 1.0:
        f = [rng.random() for _ in range(N)]
        for i in range(N):
            x, y = i % n, i // n
            if map_type == "pangea": f[i] -= 1.2 * max(abs(x - n / 2 + 0.5), abs(y - n / 2 + 0.5)) / n          # PARAM: central mass
        locked = set(settle) | {j for c in caps for j in orth(c, n)}
        ring_water = {j for r in rings.values() for j, (ter, _) in r.items() if ter == "water"}
        locked = (locked | {j for r in rings.values() for j in r}) - ring_water
        if map_type == "lakes": locked |= {i for i in range(N) if i % n in (0, n - 1) or i // n in (0, n - 1)}   # WIKI: land border on map edges
        for i in locked: f[i] = 2.0
        passes = 1 if map_type == "archipelago" else PARAMS["noise_smoothing_passes"] + (1 if map_type == "continents" else 0)
        f = smooth(f, n, passes, 0.5)
        for i in locked: f[i] = 9.0
        for i in ring_water: f[i] = -9.0                                  # MEASURED ring water counts toward the water quota
        order = sorted(range(N), key=lambda i: f[i])
        for i in order[:N - int(round(ratio * N))]: land[i] = False
    # 5. post-terrain villages up to the wiki total (60%, drylands 80%)
    if total is None: total = len(settle) + round(share * (max_v - len(settle)))
    cand = [i for i in range(N) if land[i]]; rng.shuffle(cand)
    for i in cand:
        if len(settle) >= total: break
        if ok_spot(i): imp[i] = "Village"; settle.append(i)
    # 6. biome owner = nearest capital
    owner = [None] * N
    for i in range(N):
        if land[i]:
            d = min(dist(i, c, n) for c in caps); best = [k for k, c in enumerate(caps) if dist(i, c, n) == d]
            owner[i] = tribes[rng.choice(best)]
    # 7. terrain per biome, exact quotas; settlements are fields (WIKI: city tiles forced to land)
    for t in set(tribes):
        tiles = [i for i in range(N) if land[i] and owner[i] == t and imp[i] is None]
        sh = shares_for(t, mode)
        labels = quota_assign(rng, tiles, {"mountain": sh["mountain"], "forest": sh["forest"]})
        for i in tiles: terrain[i] = labels.get(i, "field")
    for i in range(N):
        if not land[i]: terrain[i] = "water"
    # 8. resources: inner / outer zones, exact quotas per biome x terrain
    inner = {j for s in settle for j in nbrs(s, n, 1)} - set(settle)
    outer = {j for s in settle for j in nbrs(s, n, 2)} - inner - set(settle)
    shallow = {i for i in range(N) if not land[i] and any(land[j] for j in orth(i, n))}
    for zone_name, zone in (("inner", inner), ("outer", outer)):
        for t in set(tribes):
            cp = probs_for(t, zone_name, mode)
            for ter in ("field", "forest", "mountain"):
                tiles = [i for i in zone if owner[i] == t and terrain[i] == ter]
                for i, lab in quota_assign(rng, tiles, cp[ter]).items():
                    res[i] = "spores" if (lab == "crop" and t == "Cymanti") else lab
        # water: wiki = fish on shallow water only; measured = fish + aquacrop on all water in the zone (MEASURED)
        wz = zone & shallow if mode == "wiki" else {i for i in zone if not land[i]}
        for t in set(tribes):
            tiles = [i for i in wz if tribes[min(range(P), key=lambda k: dist(i, caps[k], n))] == t]
            for i, lab in quota_assign(rng, tiles, probs_for(t, zone_name, mode)["water"]).items(): res[i] = lab
    # 9. capital guarantee: >=2 identical resources in the 8-tile ring, force-spawned by replacing tiles (WIKI)
    TER_FOR = {"fruit": "field", "crop": "field", "spores": "field", "animal": "forest", "metal": "mountain", "fish": "water"}
    for p, c in enumerate(caps):
        t = tribes[p]; ring = [j for j in nbrs(c, n, 1) if imp[j] is None]
        cnt = {}
        for j in ring:
            if res[j]: cnt[res[j]] = cnt.get(res[j], 0) + 1
        if c in rings:   # MEASURED: the 8-tile ring is copied from one of this tribe's real capitals (rotated/reflected)
            for j, (ter, r) in rings[c].items():
                terrain[j] = ter; land[j] = ter != "water"; res[j] = "spores" if (r == "crop" and t == "Cymanti") else r
            continue
        if t == "Vengir": continue                     # MEASURED: 8 of 19 Vengir capitals had no identical pair (wiki silent)
        want = (GUARANTEE_MEASURED if mode == "measured" else GUARANTEE).get(t)
        if want == "crop" and t == "Cymanti": want = "spores"
        if want is None:
            if any(cnt.get(r, 0) >= 2 for r in TIER1): continue
            want = rng.choice(TIER1)
        need = 2 - cnt.get(want, 0)
        spots = [j for j in ring if res[j] != want and terrain[j] == TER_FOR[want]] + [j for j in ring if res[j] is None]
        spots += [j for j in ring if res[j] not in (want, None)]
        for j in spots:
            if need <= 0: break
            if res[j] == want: continue
            if TER_FOR[want] == "water" and land[j] and map_type != "drylands" and j in orth(c, n): continue   # keep capital's orth land
            terrain[j] = TER_FOR[want]; land[j] = terrain[j] != "water"; res[j] = want; need -= 1
    # 10. ruins (PARAM count, WIKI placement: not shallow water, not next to settlement or ruin)
    nr = PARAMS["ruins_count"].get(N, round(N * 23 / 900)); ruins = []
    cand = list(range(N)); rng.shuffle(cand)
    corners = {0, n - 1, N - n, N - 1}
    for i in cand:
        if len(ruins) >= nr: break
        if i in corners or imp[i] or i in shallow: continue
        if mode == "measured":   # MEASURED (481 real ruins): >=2 from any village, >=3 from capitals and other ruins; resources may sit hidden under a ruin
            if any(dist(i, s, n) < (3 if cap_of[s] else 2) for s in settle) or any(dist(i, j, n) < 3 for j in ruins): continue
        else:
            if res[i] or any(imp[j] == "Village" or j in ruins for j in nbrs(i, n, 1)): continue
        imp[i] = "Ruin"; ruins.append(i)
    # 11. starfish (PARAM count), lighthouses on corners
    water = [i for i in range(N) if not land[i] and res[i] is None and imp[i] is None and i not in corners]
    rng.shuffle(water)
    for i in water[:sum(1 for k in range(N) if not land[k]) // 25]: res[i] = "starfish"
    for i in corners: imp[i] = imp[i] or "Lighthouse"
    ter_out = {"field": "Field", "forest": "Forest", "mountain": "Mountain", "water": None}
    res_out = {"aquacrop": "AquaCrop", "fruit": "Fruit", "crop": "Crop", "animal": "Game", "metal": "Metal", "fish": "Fish", "spores": "Spores", "starfish": "Starfish"}
    tiles = {i: {"x": i % n, "y": i // n, "terrain": ter_out[terrain[i]] or ("Water" if i in shallow else "Ocean"),
                 "resource": res_out.get(res[i]), "improvement": imp[i], "owner": 0, "capital_of": cap_of[i], "seen_by": [],
                 "biome": owner[i]} for i in range(N)}
    return {"source": f"wiki_mapgen seed {seed}", "kind": f"generator ({mode})", "version": "wiki", "size": n, "map_type": map_type,
            "turn_of_map": 0, "tiles": tiles, "players": [{"id": k + 1, "tribe": t} for k, t in enumerate(tribes)], "params": PARAMS}

if __name__ == "__main__":
    a = argparse.ArgumentParser(); a.add_argument("--type", default="lakes"); a.add_argument("--size", type=int, default=16)
    a.add_argument("--tribes", default="Imperius,Bardur"); a.add_argument("--seed", type=int, default=0); a.add_argument("--out")
    a.add_argument("--mode", default="measured", choices=["measured", "wiki"])
    o = a.parse_args(); m = generate(o.type, o.size, o.tribes.split(","), o.seed, mode=o.mode)
    if o.out: json.dump(m, open(o.out, "w"), default=str)
    print("generated", o.type, o.size, "x", o.size, "tribes", o.tribes)

### Data tables and sample saves (compressed)

In [ ]:
BLOB = {"verify/game_enums.json": "eNqNWFtX67YSft+/gsXz6VrNBdicNxLCpZsAxTml7dvEVmw1tuTKMmC6+t/PWCPJ47Sr6RsZbGku3/fNjP/4cnK61pnYdLU4/e/JH19OTk6/xz9OH7USp//pf076n8/C7ERqpVZknPbGa11JBYNx1htvS206+j3vf69lXlj6fdb/XraN1RUZzntDAirb6g+yXPSWTWu1kVCefjn5E62nG2EMSHXMx1ewwjD3nlIB3LMbKcqMe6ZbZfFc5tyNNqKxzLn7VDDHXoUt0VuyfHVngMqNfhODq2mhdKnzjnv7nfNvITqtsqWuaiMKoZqYNxfMAhqZsmheZCZVzsK5MUIktTTSspiWhXyDcpTwFw1Zw2LaGMgEC+nJ5KDkJ6ubiy2xBvOXdyy4GzBV9OHSXadVY03LgDD5nlLbFPHJiXP36vcWr2h8pSczKrUsh8ect4/wJnPmy8T5fIeVGR48H0oTIp1cUPZtISp8O/UBT5zfVyYtRHzSOU6Ja6GMHk2d48tSVtshzy7za5FJy1yaunieC1nqRteFP3c6I3Sr4W0XUFJJO9w+PQulu5Zvw5ME/FqAGWxfhyjXkAcszCi/Rjf2XZu9NxIndQnmFcwOjK/vbBrtMc6Z8yApdLrfQDqkauZ8eBFplw41mbm03nWZcRj2RufZtaxLXUEaoOZcSEqRDTCdTz1lRnCYEyqhGrA0/xr5+g6dd2h+GUlr2e0TKlTSKWEGm6dIX9Emcs/I7VEhewTbhnQRTuVaM0L1sDWx9HNipslawxi1KrvhEZfGO60zI9M949N9VQsj24bx6Rs+oTWj00P7ITOtDafSU1vBb9JbnMs/tsJ+InI5u34SKpeGc+tnqX6WnFm/QAb7PWfVr2KLIsNJ5aHCObXsKiyADEldStu9iHdMwbHM9k++Qlmy3D5DRKwrOAK4KXTNcrv6QFQZwbO70CYT5tbod1vwNtFiPv+ngvpdEHob3ZpU8CQ/69q/6yN4aaVq/l0IB+dNx+cNgYy9ngeK9+LPPEmQsFlThR50GZIUHOuD+QePpsQRg1Az7G7sC6OLv6mDBnstdkLFZ5wnS7BQt6VljpBGcuChlGUL9iaB7yAIQt+tRIhwZCGOOguBJASkawO5Vqs859K8gG1H/+DqfCON4Fai/FVVF3Irw9XEeSR5JtShOK+1VpLr8gKsLUUvT1ybe2nSxqLENlye0czTQeq8NLD1v104twCV5FJ9Jz6g1uH8SxqIdNV+cOF+Lrq0kFy1v0kb56UpgUlm4A3U0/v4ahGa9ozaisgrEZJOmv4C7GziSAGxUiTl15DnIS7S8WWpIZCSnEZZDk1tTk7LfhTgOr/Q1TaiznmdpLq1qPKBzIRN2NlD0ReGPeX8/qHtfVLQBtSe0xW6iuGcOecfXMu8b4aZ6+zSD102iNa58/heNTidRqydUx/XUUCcyw9g3iCQL9nLsjymB9fQFIyMqyaFWjAuuiQwLiZKhNzSGCMzPj7iEN3IMF5+9fPUmzCcl3cCyqd+gGhG8xUWzXJaLlEXuhElUaN4O7gpO94MkrqMsZD0+8FLcHb2IvYpRsx0lisjYMRMHND5s0TNZB+PI1b2XJO70by0xLfqEfdQcOpedQK2p74xySZS3CVvFRJAxHISHClC97emxmYmOEfu1U6WdoAzcSBBt3Yc8QmuJPUI3v0AmHLUuphLHQg/v2ASpUObZ4ONh4Hz7KrMwTtwRsDBbuTns7M5KUe7LcWVtYBDhIfo1RanZdsdA+miNYrvLVNiOc6W3DoLIOFGurofq3U36iAppkM3fGsYIdU33xQXH95mrmWzjVT9ezRPfGcIW5cH6wHKPGaHJFGXGdXcDzBGVzqC2Pm1wGP2cW/zbae3PQtIR+B2hhfxe4sZwNZf43IjOMxXjYUtbgzFqsLBqek42N27GzzWdhzyPpnjF2hmUGmBunWlZDVstolbpnAgPVbkn1CtIOeTQT/W8LEAtz6+yMbUuTS9Ym+vZJjN/JwUqkkC8tD2En8X9CwsbO/Da5TejajqcqwaDlEj+2WkATdTVsPaPfrPxE8gh/oRQ6bhvLRgnnaskNTXN/pdoP1VNln4tEDqcmsQjwv4BBj19lU/m2vTbLCtjXu8FXh8fxrv9P0Y+7Tr1ayNDroIVx0+fhskgdp5AgorCkGdqaU/tRb5ZHlDR4QuQO1HYoWIYkmhDn7T4qTPxYuJCTXwdZeKUrYVb9jLEldjE2XJt1McFAvUKsHFaoEDVcjyGRUIwxWB6UQEDueJV/ICJ+XiaeeFIW5hYYY+huhbqDiclybsBbPwQYHBeS0slAzPN6YdLQJJ3WOQz94IlF084zJsdu4WNn6vdv1XrZGbuGp/CsU/ffFW5CeVxobBckZd5k02chsKR1tju40GmpGFNXInw4s0sL3LHY+D2sCLgIx/h1kWyGZ8L+y5shR/4/hBfl2viuOv11PEPfd7g10X0pHbvFURB6GKuyDU/0ar3MLF7x3VYvYXOZsfLkZr7CvHLklsnCOc29Q52R5PzZMPZ21Vaf6x7w7nwfHXvkUrwxfCsF9iGx0tdT4HDFe0WvJpbaWyDTZl3vLwJJmrWEFo9n8J8BlShEfwhzgiVPyCQOsg9tNjm+g3zO1ooX4U7NPRuacTEkGXYYi+YB0eXw1lgDqRn2MnN1J1vArVeOF/1KYKRJ35mXtU5rs25+hao6zI4esp3nh0JDddvw7wpDzAPnB/5gOx2EmUbTiusYy4VCHqNLvftahXbUpOymdQuXCLwpc//w8OX7XV", "data/measured_start.json": "eNrFmF9zmzgQwN/zKTS8uDfHNQaMSdqnpHeZ3rSZps3MXa+djkcBxeiCJVcSbdM2370SyA6iSNgDuXvwBNBmWf122T/6dgC8E3xOvSfg2wEAHpEX4dRXl1xAxvVz4MXVgry8qxdTytB2cRZNzeWSYLFdPcckO0UkQ8wUSuEaC1gsCvQJFVvpwBTKyxUk91Yc6ldt1+G1QGyxgiLNF8ri+9cKlObK/vfqDninkOPU8+ubixwXlNN1fuupBx+qx9sdx/pW7XGxopnaaBJN66ew2Nqqlfr32sB3EMtfUhsZKKG7g+qnrPVOPpaQYUoM2qGNdmijHbSWDdqkrAxsLDs4h3bO1UuigZxn8a585fusfGex5qp3HrW5nkKWlcygGtioBm6q2suzQHkw6Eb8N2TSi8zU5qAc9FAOBlJ+XhKByXJX1LEDtValecc1k8B3iWxAGR55dis3KvBAl9x/Rh1euMxhDXOwE6LpCE44g2y1hxPmjnyiVWnC85pC2Eb8R3H7Uy7ZP3MHsSNzNwJ9aNoeJdDPpOlcnMOlfDRCsDfUtQK+zfo5pRnD6U0TdnDUATupFmywwyrQEr+B3hLcJyzNq3rZUGYH3xAywSeH+p0DwdfmdBbLpCu4Q3twa1WqSqrgrs078h0iW06GS/5crRHDJR+llgb7Zf1RaquOS3U7hotesSUk+CsUKieM8HE09ZlfR+j3ynXWhBfyA6J0lCr9HxXmoW3mGeb5SIVZq2p5oo34ZfkFZ5Sy3jwVuvJUrb2xbIHcm5siV24KD+NhRaGDargfVUky3NIMjts0X0l7/8W9BXbuLLChazR6g8eYiuY6yw8NV2mNJVrnXVwdSb7WpJK3wht2T0MXtJDTEH/Q0ZNSgkebOge3L4xy8Zmym50TgqtJ3CjbpATLzPm6ROIrLPrDOOkP4+B4h9bld3Q9zrA/VvNyKRgUaDlK97LRZbQvWt4m092//IXIEpuJ+tgS/FVmsgd/Y9kckGR0ZLwm2pBxtJHHD3zscrnCwtZH7nvostFlHrlM25jfYvK2fwxN/v+eIxlpRnpW4NWVJY0nezYdG106km1dxz8wgzdDZ6N68/NmIx4ObkD6hqNRiL+hMOM7JxcH7kqRmTUS3y6wQWT44h26kqJGwEe2mhrtMRT1+qKhzRH90UOfEOx5CuPquM1TmK7Zpy3S6ZEFvKKlst67QAwIhq+QD64ZXYHwCHD4CXHwKApnvwZzsC7gLWL8l8dgUpk5OZwo2vKPgXQCIEO1CpkCgSgZ+W0KOIFrnlPxFEx+YjcBmFey6n0TDtKSMUQEkIsCgUq8Wr5Uwufq/wBM1UTng885lnc55GDJIBEcVArVris/AEiyWrN6DCqzH3sHdz8Ayojx6Q==", "data/measured_ui.json": "eNrNWd9v4jgQfu9fMeLlQEuvENq9VSudRClcuwcFAd3qnpBJDLGa2JztlONW/d9v7AQIhDbQpbp9Ajv2+JuZb34k/n4ChREZi0gXLqHQp8SDiRQhfCVK8F8UMH46JSHF34k4dYn0QLmSUq58oVUZ7LPnqlMrg1NxPp9WK6dODYrnFWAhmVJ1CQMREMkUEO5BnZ12BGjq+qBRiCrBJ6hWof53hEsEh4gzDeYQBcW1uFKhjCDNM4UYv58AFAaakvCRTAVPZqDgCmVUuCjbkXpiQWCWF27qg1toDhr1XhP63Xa7EC/wZ/iwmqwmWhP3CSdq8dijE9SQ4oQTT4TimYaU6/USSfjULKgmO0RImAFTaKOiyRlasrFZU0hsUMDZl/IKf0tItIJS2ypUMzoY3DBodB+GS00Gw/rwrpHWxals6XK+rUsto4uzqYvzLl16CylyvWA1qD8Mu6Prh/49wr9rtTY8sY2+so2+mof+fZ6oK78uXZ/KbRVqO4lk0be6fcT/1x5aVPO1qB7DB33m7atBQqBEhzfROx/iAx4FwQp4g2gyiwK9jf3LNvZ8W2f4Xsmzde1QtIcw5UgGfgc9NiD/ydnUzzevhdxr9gd3g+Fe0Gu/XuSDr/0gO26M7H2YvQvxRV4s1vKMfTDewVxg7QrJflXpsEp0fLSNQJCnXGbc3t00odFvNntwd9+6aw/79WETLF/iimQjc0dBusgje2VJoSPmkw7j3vVOzmRccNust6HRvf/W7A+PU5B+1CHXIhxngWfbgUa938fq02tbJ7xi/mzE5vc2Tk5+fCQa8b0VAG7cSO7RkiVKJBTKRe8cnG2cg9H3SRjuQZw0/HeYvXZwmswF3osmkyzwbJLvN+8bt3CUUvoeqhSWjf66cRlK7Fy4ZtlG/s0S9TZV9ilNea1vPZz5bMxEpPKVGPhEPu2Hf/DQ7/XvBs1Da6tzFNLswP6VBsFiv+4Lhs37Yb3Rbg72ef1wDk4371UhcVW26r7VAH9043sIfzpU4jbvJ+wuD4oCJin/CavuQTokDRzzfooO7iDobjQmx+jmTGRnPfMh3dw7guS1V4Iv/9MrwTtU2IbuHC/CnY/Cf2JVKBDXlGo1Gkcs8Bifrj7INaTN1buDhmJ34urUhwso3lKiMdaeSpfwqQozMbsCLYlHQWIDSRVoH/9MfWiTZwJagNCYGiE+RcGcaZ9xuADNApr6DDKM5Jhxuj+MP6gVHDJXGSSOQVKGMFIaxhQEBwLfROASLuxR65NuiXymSrdkxPQr/lwf15VTwtm/xNhuqW/ZSI+3r4VG2Anxaa68ZN2GqEd0CNQ5C0mwFthiyt9HYLJuExvOpT7yCZnR87ccMa+4tOtSws9sbT97pDowH4ZXHjbnbPv3yn4XVoCsRumAyyh6xxUh/giiU/5vR+YF6jb7ClLLQBUS3ScXS6ypIMVejupcCg2Nana30kTCDLHrSHKYCAmUuD5wSuR4gX7hXoiBXQY8ckrLMCBzM75CG1O7zWV6kfqSJ4iXi94sUpfgCs5xRhkRzFgYWepTcMmMaRJAMXlOPXtGomnpCpbpAc3HIzQv8tvQ3QS1BnSWwp4cFiJClaiUTAuZAniNHfs0P8QSiDsZYH1v0GJcWa/HMpdur1q3X+HTxfJGgIMrhToU6pCGsyADddmppsggKYXBjOH+dQhUKxVQLtKkjFPL/0CfqVyAY32dIt7Sy7lmaWD21DJy06nAsiChS4vI8MxkXvvvNZJc4+kxg3OpsnngxG6C099hwigmDGNxY9oU/825uWqYRXGkO6uE0dgQ0wzHRKlFrqQbNguwGLkYiYzbmDYRNEZegPEt5gIfyQBIFCAa5kSWwRPROKAJeILmIS4190kkbSIbbfmFIGSGgIulImlXdLCSvOaAzj5VBhdt26hDdTo/dwRmcizE+xK1Qz2M7M0yss3T1xhqYm7fcwwBNVPhJShzv0cwdOOdK3F/SDHfzcCsjW1gRSSwAmPeGQLGu9MOw/FuhNUcoTtQJg2LuWtcXRrGVX9krh5tGU16ECiaFK6vQNJZgFxSkBSyksHZVZqKmU+mCzPqkCl2Tgojc7UbZ+11HjT5FGlhxs1/kNSKPZva1QhYODZ1C+dT7QYUV6fFPLGHLfm4uW3tdjPqYUAIhZBwVdIJnPWZ6cbO0o0GjLHnwxdKZQoU94iMPz2urGB1TpuhjPXWTCK0T06pnCS8i1ol3mcsOTL+GFVHNhiWPuJiFDAkF8bwcg5XMyqrKzbEYwfHn83wJfamid/V1qK9Gy5lJZxvSUiPa+YEK3FFozF2xnNzgzxCOmEgW/cX4otnMecqZT4MyABtmLJu0QgFpwTKF3NlGDWOFgSTjck954l14oS08g+2kRFHydRDkwlYnQ/L880KEgRiTj1k5svJy384P4yw", "data/measured_mapgen.json": "eNrtXW1vI7mR/j6/wvCnO8Azp36X7tvdIkCC5JDkS7BJcBhoZ7QTZWxpTrazmU3y38+21O0mWexudpXqIe0xguxYJik2u1ivT1X9483F5fv1D/v7u8v/vLj8bn+/u9t8vPjxsL+5yJuLw2Z9fXF3f9i9XVzcrv+2ub34t0/rm83F37K8+Pd3F9+tv2zvHkYctrtPtxebv3+4vv/Yzr7bHA7r7e4/Dpvb/f3hw+bisL7b3L67vHr4wtPf3n94/Lrbhy/+x5uLi8v/2v7P/vTvi8sf9w8TH/eULfOr4yfbzfXHhw/yIjt+cPM4+2GZx0HZ4uGjf109LfN/9+vDdr/rluqPW2TmYkWxvLK+r1h1S/33+vDx/kDsqV5ae6qWzp6a53W++3qz3t1tnxdqp5WVM23pbGiZdwv94vqr8WjdoDxbOSuVtbXJ3kK/3O8/HrYfPjtbylbukxS5taU8fz7uX9182Ry297fuw1W1PS1z1+7t6dcP+9mTFOBOaw/u+SSfT/s393/fftzvqfdW2UeycGmpKbqVfnt/s/7rliKkyqajvLQetmq6ZX5/v7n7eX29JV5b3lgbakr3iOpupT9sdp+25INZ+8mWC2ed+vmlfb/dfU8+18o+17yyybF3Pn9cf1x//kyt474fh6yz53X+tPlhfUscT9aszGXKrHEfa/m4zpunpS5bbkOwln/+uD78s13o+EW7/W7zdHrPzKMb2O7BGFm6A3s7MYbm7tD9h83aGZe1JHB7tz78uL39y+Mbd+f+9MA9D/bc2pmZmTO3u93GfugfD/fbu6dzu+qvVZ2O+sNh/8V90tNC5qGsd9ub9fXj3NxcakFNdg7qZnN3nL0yZmfkbPPs2qctjjPXDyz/tO/KXIs8D/MsT2u13LK3VvtmTosVtbnYg8z0Hm5WGlPrZuhwTwuRFNfO7M56Rc31EmHLOU9HbZ/HcTZJl8uOMx2POndorXIOjF6eJt3KWa++Gnm3jSPjB+504RlMnnJOD/Ydq2dt+n636kr/5Oj55EFlS3fw0LVuaoPUCvOWN413Nc/dLgtjgdK7gPd+l+YWqDcz4Y6fZq8Gic5czzzP3pTCXDMvDMorSndB6q53/KoVbe3dNzlp412NJMWiti489Xxht37lW8AjkHLjOFaukHGvZlb5voNktkt3gdpkmIRoyyy9fODy1+RQ8rx75Ngb6zvakhpMH2PZDNz73mzy1rtzc3suxQSs02spsio9k2kKXF7ZLICe7r3x9v2ip9PXPXPZvkkZ9GLkKeZLg5KrhSvinUcbuOkr66LnpWcyLckL/8U2pnuvdWMJc8/eSXJcUdL6ijr5vkSmv8A86/4VXZivKidViN4WHBPZf6fzjB5LX+qSHDyuq/dHe8T5ckBd70+npXntjB24x2VmSJJ8ZdJf6V3MPJSO1k11trLosVh6lvNLdlPL9c33HKR5N11N3rMaKVAqk/KK0qV2l9YGrvqysGS6qVV51yKpsbIZqqkgVJ7FpvKBxjOfPPam8PHXHkFbqn/t+QKSwlu7oX/L3RtjfkFpO7j8bKChh9Iclxw7rtT3R3tUpGpAPven00ygcsYOafQWF8gWprWd577VfNb6ylK1PPN9B1WZ5JcXnvm0WWm8ecLm9ixG2+wVIdBNRrB01qNufTs5M40lW7HPfIuR5Ne68p4FfumZ7zvpViR3/NYznzzpejUmfV0du/B8AW3CF2MGe+l8wcrxQPuveukZSx82PXhcjzeG07c9qwZEvjHfI/MH3HTd9EEOUJjuI9Ogzpfe1TwcoK4s159vAa/kzyqPhm8tQOr4Jv8hjEnfaiQTKFyLIbM4ZO0uOOi8M3lIK74d5529GE2YlZ8LWAv42YBvBi3i3ZtuydvFwMW3voG++dmI864Z0ObagNGA+b7yDB7xyRuDx2++Mdxz84d888Z8+uYXAwfdTR+6+d0OWs9V5Z1PG0KLha3l574Vpqr5mfcZaIvemu3K7Nq3Hnnfa0JpMLX/0o1ODt93SyFaGEdeexejlc6Vfd8L3wLe+25pWN4d0ES7yAnfmXE9c4Jles8sUMu33jrpTThGev0MgB45ouP3ho7b+b3B9CES8YmGmk37nRYDEv80ecjsLy33XW4KoMazFm182gGkyrMVv6xfXQ2eOHX3+852k/QWvmCfsRpt5K/G3MaZ84YHbPxWP7dkZMsUlp6laDJc+KW8Md17ytMehPbtLcz4BXH/XZXAt0FaxzI1osbVaN1XUzqAjAGJn3sGj+j6xuDxW28M90j8csBrb8wnb76LQhm66IUpaQrL6VSU3tVoTSizHfcL3wJeMsy9AXRrBe99twjZMhJz33ok3TkR+CG3XrdggG7fLDx+PXuxaY69bOFbYKozb+lbgNatzPNpqgn3knoFQ7p+7bMmiIhg+2tlQagG3PrkSFqXpYZ6dShqsMcR3QyI+t5sWsVfDXgCT5OHOIAVqbN0z2XjWYsO3Nn3P6Nn+85sZTn2FvR078237pZFFDW9Gi1vTJWjJPTVpb3cYIh+YUXuzGPuMRFzMRosUtKjw9V5Y7r3WFek7t4q4u6V92yPJN9qOYbyaQZgOC2w0X+9l56xI7q8MXjckjeGezz2Q85OYz59zZsBjb6bPmjJ16Ybv/BOp3Ue+27XpW8BP9BucTVy3NPNeFcqeB+Idt0vxiR7Tiw4JNmLQchd5l1sotfOu4D3uE3Z3OMx1gLei9+QAE0zhEbqqdYXDMp1Nyrn7sBF4ByByP57vyJH0mI9p8Z6g0/UYPrKV82AC6I3m77wzrMOSPHK8s/berxnKZoJ2vgvz2y/Bl/SE6aK7cK0JnN6NRq15JqZuekYdh5mQGpbulDnw2oZqmepiV45erZXTc99kDpj+jAClDhzf0yc/gLaGZd5sT3eoH5u5gL4b3JBDRzRz59Hjqvnz2M9h5cPMLrnyVP97/YXT0fftHi+56AcvRRtmud+L1x/tt8HX5PjaX/Q1QiibkWuRR5haXIDShO3Vxu60pZQtqyfJb2SJ4ppQ0YLcrbfrbm02B45nT7geux2ZQNBjv7ytFJejYXrhtBSx6ScKaK5P3LkRveGjl/p3mA6QLkcsJl7kz2CeeBOnyZTl7q9xSUN/W4psPCs5aHBwrTGCs90v7C2Ilf09DHT0ATmuNqasRgdsTDnUsAaZ7kJ97yT1ZUHEm8uNY111vRsL11aaPicnk5jaJbhGBrPw5FWkAWYXS2IuJxL7gsrcW4SaLY/lDznJTV0XA3vDfZE0sohKH9vergIP00euO6r3BLa5o1pCs9aNCU6obSanu6/7tbX09M9YV1SKJAiwFht0MHWv+AWLMfZ3JCDrbQkeW6GLHvBJnMx2q1e+UPoxnS/bDfZclbT82npZMHimwl3fkmvTxtJy7HQuSf0fEpx/XBMun//mHRPps7venLl5uEB39/tT/6fd6fL9GW9PXQJ2+05X3Xh5qv+x51l0o3riNn8uJXk5qftU7Z/ztxJ3Rfbf3v8uPu2x1v/eAa+LP+dYaUZT93Sk+ex+1vodm9tu/3csNWOv3Zzm/6H3bapBd0nNv7qeWSrGsGup2faD9xMec8N+f4K++2XVxOOq/c358lNqikCXrJdN2HgkVsb2Hpkw4/VO+eWL5mbefqTvf/jezG2mRt/GiBhIym5P55cyXNyzxtwT8guCOG9+/Xw3Te+g77YtU0YObVN8pYbcGCCNQzcf4JiBq6IU9ZiZyiABsW0fJg+EA+TMzAtzk5L+4iKCfQ1dP2u+prgEOc0j8GpyTHAHNvyKtYxdCu7HHGIVTrvkmQXxjqF/551iwwxCatsiJdH1O+y8Rtgcvv+Z4VHMpSmZKgmyLchFkpIAZpOzENwap54CX/5bjFZGj4FWXvfYtVDGVA0ysrLjh09o4t9U8dTDVJZMYVAfre/flAVbqdse1VNEZz1IJN3b24+IlIn8f3pt98pNzMgFDzvyTzPwnpRzaAIp95iOYVfDj6UVflm14NeGI9UvCtphmbkHxEAxV728pWhO9hPn9lv09qpWVtnQGWpaS3NBARa6mRpX4dhVlnQhFYOiKXceX++e2VV//FynAfGm41zg3Lo9g9RHqlG00Rk1Rna9RyAtgadD6mTrlAc099IOecomZYFZSmruVeLzMjt9NwlxzMgDMj3d5ubL9ePldksU/LPj7Nailx/eX/39cvj9bn8ePh6vd59vL08feXd9vpp7tP4i/Y/F29PW+r/61TF6fnXpzM4/va/V+b8BTX96Czofj9dQHoB5vc/z1p4pu/ur69Hvnoxb+vP350Ff/dibOrgU2dhs990//90vShyuV5/3gjQyoQHFiWUea/r+X13jikQtYQsQJCMs/2TUJpONeYGek8/gWY+7B9s/t3msXzbDMI5+tzmEY4591lHmE05/q9/y33vowxqIn9jnBz3xolxmaed/7Q/dGswKOZoyMGkUhjFzLyogqwmbAeL+ZItmM2dndMw1ZnpZMdhc8KKTPjXykklLpMZ4e5n1WZmPboklwHRC1MksUnG/Popwnwya5tAL+vDh79sv2yu15/2c6hmJm+NRTglai/N4MuTJk+gF5ZxPVXzjNK0nrp5ztwXp74w2UtsQmnM3opIlZlhMggxmCNkhaPE8Iwljl+Gb15bTz9wdOewr6e/uLcZyw8Xn4Ut5McL9/mGK3Bnk0pcepnM3uT0X0FWE+bDY4omjhMPairxHTJMg0mSyehqM1/Wu0+bNUSRiUww6XlkGFElJKm8WjtJ2t+LIhem+hImi4CMZeb7mr3CWwaDOIcbxqtChKouQfyF77iTQzkE8pdQc1aUUuYgBcQ0F74nhstiYoomKYsladphxAiUY0oM9x1DJEkFkjjcZdr9liUSLsBBNiwQbB6xkFRDmKC4tRiOVGIFkGLCw/CcvFNNygn6jzYeZmTr8tCGqUpjfC5eBoKK5ambrLq8OS3XT5+MEOnLox959KY6qkpSTgUpNbz4NU8DHgltJeDHm0G3ktGCmQcoYnPHFZfk5haE6TggbFUAyDx29SaU0cQRjgzUhDnQcEnOklTcmu/Lw7llJNXgWeAYsbASlGYUFRhIuhI/SIBLbGOSSbA3SzBHiUcpogCH0JyTYD3tbcbTGs6BBQ9UblnqCs9zx3HFcKxoPv5b1JAGEQonRh0qeXheGBDEbrrNeu40pDlKCthmRkYEOHwFKn+YgSROTIBFLmC/3JyscMn0NXRYABVHAhrPnJxqTu4ah7XMFkSiidQc8mBaQJPO+nzlGhhUomgqywqgOdwESC9omLegMaRIMvMhDDhzmROMZlOJqHeFXwWGA/LWLAEDjT4zMwEw+Yxs8CU7HsRJNprPV76FnMV0W1WPHIu5IEvYpWo4MzxyyBT7aMoxsHPSeH45lhQKTgRg5RiJ2UJMnpIU/p/hWGFSChNvORdT/qod/vFgnZiMJczPgos1Q0uoigH/NcErPDJhlk+NR1lRl0FoF78AlkUQJjfVGpOUR5JW0TkglV0KQNdO6M/nV39V89TY+H/ZvCNN/IJYOmOwlGIqv7IpagCSEYRxK6vAkpU8kAFp7brNWMMJV+sbmtkIrMCLc90Jx5GSKfqCTEaLCe0SHJ4W9PUqh6bRNTxw/EU6C0A3bxpsN0nWbJ4BrEu52qFchW9mnECg94SiUy/hyBIKhvmK0+qZxvXLqaY6J0MA3EyLYSoJ56gBi8CkZVzHU/KQXYRBtf/Ei+jWl0xvm/mGtXR/Pv3WE+BOoKhuSLKyiGUkMXBUyRSui6wAA7MaGae8i14tMgyQN7pWWerNJiRRd9w+a7oAGVix5nT7UUcWc9TuZiNalCG13tRQskGCwZkFeGOB9zJDj2jBBFV6E7Gm5cBTiiXq8CEkfAsBZBvziOIC/HxqYJNhJMRhRuyRYV5H5LRDFGnGUQ3efwcvBZNyw2Fwoxs96umw4t99vVk/aL8xgsXjws2wKsewkOJhQSawphNRQDswW18U9YsrnxkelpQryqtqdk9/2/Lth1lRpihagvLSadXaaUlqMyxuwjKaNAtnBtwLDZADu3/ADMMJnbEUQc8+fPK+bNwpjIQGJkecvQ/OsBXtEcqr1aunv/CbULwYhVcABaFa9SGa8s6aEHF5NEQg0fA0m8jahGrVY4WD8lhhJ3xcW5LN8CvJh0kn0dLggZwmqrCTdnRbOozwelQasVRJ9dBBXPVBNIHjcilMatYSVCihcOKioclQrsKyqXkFQSLqaoJqAXqpXbBKHDWj2QhUqvQzpzYesLQDK/uE5/NV7hfLZCw8Zy/fnob3eBRPvmYpunqNcPh6LgZgJdq5TxUkHot2qxoNQGYU4ApTRdU2KTx4JIuB0Q0/xpO3pEkysnYRl2B4GoxebSpO9Og1thpGF4Fm8RVUky0eVEpYYWHGpXk6CxSKmVCNGBbBCKBfsG6XyPuZd9DvX1x/PWz3u3NBv8NvTjyJkVjtNyL4t6rqK+e4021qAe5cHRHSTjcvMqpKZroptbDe1cLtIFGZ10nl0GL5SzBXlowNIB0xUdWoArbjmoOYSlgmfWtEMCeIBGzhJpqoH2Zlv5zyvWoOPCjqG9q2AloaXDxMHYr8TtikFvT5AtkLv9OJfvMt0eoxWB2Gl2IyIwuSZ1uLJ5okVK8qHuC3dvUGqdRHHvBbkVReVucK9aa0s+llBuQwHggVuxdtXGgHxWSBiFDgwJIfuhWgoW3bUBhwbhsCHuQBmyObsNaLxPTGkmIS6uqNQIVh0gy/7B0M4SuI1mRpvclX2dRuTOA9e2WAr6qpBMtlk2YxidVkBYaUwL3QXwp2k1PuQ6tTDirvkQ/Y5KQ+YrVdKNoOiZ5CZ7EximjOSl6INAQ5EBfocL6/3O8/HrYfPuvUeGbLKM1Km9HVSVRFPvCtbagmLAoRT6j/H1doCUM4NfNSwIXMks4q4DAaYLVEdH0HZBJ2XP259DqN8sEP6YI4wemSUDYjFpdUhVnFkrYUxly+2dz6wQJh3sIKSyYUwRa1uJMq3AvOcBNvRKDJZkSL9ibdxi21es9ioimMzUiWwXtF+m5k5IJNj0ynh7FsSXludqQy+EHUSGLB8bgNljjV5NPpSYuE46FqVaVtJ0nh8KBJkWkVAoE2guTHkmKyrF9VvaE5VUrPBt7EIfES74qTLARCD1olVGZT3Q0DtpB4KDxgNTxmMiRD140gWz+iBFpVkoHKo6TrmcXTPZRHMNF19euQeL+6+bI5bO9vz4TEA5ebF+Q4wJ5KsL4EoeIJ7cDDmkxy3XKU3XcskuG5fHFBgjl1XM5mXmt6e6NB3c3w9YqCewHN0CUBvnoMBlkPZOiaxh2zjkgW8dy+WGmkzmNeSLHNdElGudlf4lKJSTTieKqEKskAoQ6oqmYpJyixCAXdZokTJJA1q1XVGGauPs4FI4yJYTGXMGoRDFprQmLAPZaY5JJ0FzcpYaSsuoBrVAm0JJC0qxN29GqymblvLcqwkrapxEdUJeuPERdP6lmQyKK+cbQ/UayFJ9qNVjU8wIPeJSyURKEOwGZcOCsJFkrSKmcWWemGpKKOLPEjnfz4jWAAui4v2QRUtgFXuVfbnI6JVJh1NdW5C09xiSh+pAuCQQEaUg4GJNkPB0gnOBmE7YMD6zogXNIu0DuHboTDsIjQYohZB5EphrCpArhAI18Wyem6PorpsgV+vf3web8/V9XeWLoQMMPUqbXHgfZTgqcM4MgGB7gT7D2raFrDy64KksqcuCNDRonHqhWxMAPVjGJPfxQzrTl1NPk6cCCpgHNlcW2UookgKbZRmiMCGeUu0m8zi2vPJmlXp5IdkGj95pno0xdUaBWm3gKTAqIxntPqQ8ETQdBexDgxxIHSSVaIB/tadAONGpU9UpFI0PKqHOOZpeKqh6Z5EolnPc+oQ3wuLjPD34KsUAaMOzKUXlQxO2hSIwv2wmMt2NDRS+pDoe3LxVnTkrXhwxNI0kU1yKGkeAwGCGZgFcrUK6kKtKLRCbDo5nxsgQR0vgBLq2KxLyxZxA8s4ip48AKLoqGiOeY0rwEFjGKEu9t86wB6fltaMBbNsqKxwKhzAOtiRC1AvbuCXSf0qmJKA3dBDZD0g0bCTQRmAF1iMaXDRBGUu7wor+6rIZlpxlEH9P7N/d+3H/f7w5mg3liHL7ZV31w7/zz9S5T9d6KpjtDih6rlVeOKE+jDHOJB8TI7PeoW5RUNQqonlvCrTaEzBphmN7ilbDxNtpSF1Gw/Db7ch3AHv+AQAhKLB6ygyc1i4+XO8thM2oVX04ZvirbW4ia0abcmfkFQTtWeSVhmE1NBcOVMayivQddejaGBaLJJBqLdiZUz9CNq0ZYUxbAD3ThkBLg6uKiA0vXXJGw/SWdev96SVdpEE1dEQdXwjgMPrO+nYXevkK4nD+zxpwb0jMDijssnnAzUE1qgFQnHYkqm1NkM178XR98K9U6iQBAfX5fhhi2RjCZRdeZbCCFVaETaZUPw4gloOAUi+n57f7P+6/ZMeD7pwPdrSbCMpf6ZZpU8VgE0UbM7LJOFz2x42SywyjPfUlnCiYVRbuaFVCjCsZRQLWZmSoAQpYDTb5FtIVFxSdFOBcBOf+qVfXnEAkzux9aaETauNVuJgkNL7IIz/KRtYEM3cfyDYsV5qLL7Ior+ppu2zQoqwTpZ6GovuEbofK4iZxillJMS3GVZplsxSs3lW0VWsvOMZG22ywWJ2mTrLrz6RKg+kREV41SPPMK8ubEoLYrtLJAaCzodO0n0lCRfSZVUElVuwziKFIngeuN8a1B8XpkjGmrWjSCCAQuznIHnQdNpBhElAZhAesGVTmRaylPrk0ZTEy+aEg66ckgQysLy2IbJoVTrbMbm3ueRS5DWIh0X0ivji8o3iqdsIkcKMVTbME0FbCRHBHri9GYLkz5TK2ufObmepdICASy0DdQBcH+3v14ftrdnQuBCQS3gpDR2jhG40B3HlI4Gi8trQIBNZMQZ0cmVgREvcZdWfQbhfMZE/HWi6Fx+rxN4wWeOJ2YORpcrn1jVNIe3HzNQl+OISV2lkWIzmt4Ybt2g4FjIGZ0xeukiwtovRyQp1+kFdvqLyVziuHlVbe2Eu81KV2Tg9oYMZC/wAlPTNaizJqWpp9djyWau6hmFlR2R0qsFvYygwBS4lEfKIEy0iApnseeBT6kmBCANJbgrTwyMqVxjFasAx1SSjGNe4xIZwWWCgDoMtxwDsn5dEiheMfAUqF4Hx6BWlT4xeV+Y0WouQ0H56tJCxLwYb116VXehGgsYyZtusTEOj5Htc4OPOYZnv3J0l0hKAyVYDxNXzhBbqlnOkkb2EUg26MhlL6o9J2STBnj0wqoGPyce8HJ69p0njtTBfX9/v7n7eX29jRTvG1nBb12nDNuUCojHRtWgWLRPUph3BhzCjqlZqG4uimjzkjngTbBSjEyjjSU9XzmB6QX0QU87I0XaZ6PLc1hATrhmI8lvWA22UsI+SLVKSqkYSNoqcMpoTsGiZWB1RrOzQDS9bvSKDUG5i5xvLzneglR9Zas4zzCYEu/4CDW0mVEEZE1nrEePWSORUQBPtOW5QKGq+Cs7y9UHUS4TL1cmMVlHTGCsSRgTrl4DPC5HDDDbjSmKMNhNbSEEVHmhBc2g+i6OWASTaJWDSSy2EhP0IZRYUi1RhQ5Yy2fQ6kojVnPHl4MUT8vH+y2KxMu9RtMOzlxK24MnFUxik8xr6WiScF2QVE2luZ1/X4xCw2Iw6CqKbEYj3p0tkNsk3ZlNo39ShwP+w2b3aXs4FwpYLj6JM7/1UijBeXAcGyoyFKe29S2HemBBqkJBD3KaTRi18KTT1Drw0vXn5doUTBaIkgpNWL/TeDrM8lgLuiC0XDNiRtQ6TAyxY9YMQvHebb20WsW+fgJu3yRb+wl77lQ7ykbSBDIRlZYRdWQTyQgfe8EVfFGUAm2oxFBm+f4WIOJbAAXDoBdhTwtUU5lRBoSjr0h24eIDeV9DLy6kysKXRlEkISmylniwdUlVpGKKIr7BLJogwCsAomkPzS8uBHKszI8t8mlEvLksjk40q1DNrxDDFz/zySVd7RbaglguKUDPs58skUSTzcgSO6Gh5pSDhtKe/RR9LJpOWxRLAbYr4VjIKdZwTrItn2QwWduNwhI7DNBBTOlErK58oMSzQBJ5OdaOohuFYRUjC6jK6bF66BTpqgraxbw1ENodYvL77e77bYyASQEug02HTtdkhjUvR3trw9SnCKHaLAClGNBJt5QLq75YPCn02gBK+VouvBhiWghtOd1mDpeJAH8bU0nDNPqV8BUaOfQtL2NRzQ0DZTEvK/KMLk2nmQkCBeTiMszw3aj5RMOt5Q3OpedDL4HxAXGLOyl9OKqSL+pFmgXCj3j2I1gMXg+KqdEnKdK8eiAYk1mTDOvbA0MyY0DuKgYmpZWaZJpWR5ZHwmyg9aoEknB29KSdSxIOI7SduAmVbsYaCFAl6QnWrO6NLpIp59ILREJgUxvnc5bpNZwire7NkkavvQQZMr0Elz6dckFVLJ+RS7ZPq4IHMnkAWuubTzLoCryRVKgDOGOkg9nh7t9XWYNX3FriNqoOZDjync11HXlSAHJNs4lTqgFa3SO2+vBAGHmo7SSNnAmtv8synHiATjaLOTuos0OS/3H9cf3587mg5KL1v1mRJma6CldG6fZ4TDmlVrYfG6t5Hw6rB2Y4gZEDDl4PhyJHyyecIziaLtWhKQei+PHQwKRYov6lahcK2Y7m4TFJZpsbLlgG0kEWVtQhqpbmjPpTLAE0p7AQvKM5MqeJ0zlAsM0jByajWewhHosomEpEu61xe4KG48Qlg5GKyfxYQJWc3pJQd75ISoNog3slSeUy2fJlTB1X0x4Cpp5EJYp4JlFY1kk0rIVnCwE7ToA7OYY6WpCtzCV5S5go4rlaYml5D0HzQgFULJQmrAIejF7gdc2SJRaWywVHLoKNBFLrKAxz+rNIhVdmSDSaqOyUY5rPUaVWq3r9UcA6sOrCCj8znC2CWSV6PrlI2k7oli4T5Si8bjbJlfkAtguOKulem2zAAWixzjZDccUOT/enzQ/rW6XSrOFklHIR6ADO+y12FEEHUFF0FKdB33Q7LeVOWpjO09FEjNStI2ZiAAxwGVcNVn4zrcv0qrC+6qQSUY+MJsPBl0mUTZVNpLo8lF6S75/Ec9sFYl8w9clEMS9IGAOuBxtAAsWDZeDZRFAnr3LmNAP5AoZ3w+p6fBM/jCJT3IIM6cJ0k0N2YysyiEYbVa0h0fJkikUZ2FZ0LPmLSai4kjaQauYilK1ItjBRM5ZTpJHkO/bhqjXHpLHMyVoUDBDNQdPBmphLqywsMaQHvuTpuJK1OSBtHuMC1XFoJlUPrmaBQ5RzLibeothFKxIdF5e3mKyjRQt7KQn91y7jjXP1SyFZJhv5Uilo6fbkSzUqJGYHKQLkeLqJpP9N07PCKNkSQRNqNmMB+eGQ3tqE6UW4xDtLBs0IHmIKLuO8ttJNhHlJra+1dgu/hligiiue3KqN2UbBE6ZXXH7zRDuXP3x93yObR0J61nWPv19cflrfPFFO+bTu5e3m7u56c/Oo5rz/sjm8v73e3z38dfGurpfHEY/T3x/Wd9v90+erVTvzbn143NLTtONzPv79eAiXh/1Pz/R5JOmHKdufHzdWtAfV//aHz5vnzx928fhJtug+etzGaQOF8dPNOe3ncVD7Wbut+un34/2aups6Q+wmq+nd9D4/7SaviM3k2bKppuwhn3EiU99PZfxM2U153M2bE1E/bcsIQgRT76IgqXeEdsusGCdf7yuqJryiYlk3nnfU2KeSVc2Mt1StprylYrXs/6yI/eSZvaGqzOZcpIXUhgqhDeVTNlRmpfFDvbI8d3aUNxQtHy0Dm4qbMSoua5KKm3I5QsfLapyOc8/x5Ev7eIraOZ2mJGm4cmh4MfSGsqVnC9mULRRlVTerRZYf/0Fsp7S3s6zPRi9NMUrAmXOj8qIaOp7Sw2ncDRHHQxKsw3lX5xJGTfPwggpyF85rmcXo6mraaxkV0MRrKRdzNlRM2lA1Z0N5TrGVno8qVEJWOS0hq9NTDMjI1XwVr5okC8q66f8sqSNypGXZrOZIy3LKhkyNpqqnbWixmHO3ygl3qy7yZZZTV6u2d7Eq55zKJI5XrYzX1FCn4uynqElK7pm4oZSclyQhZ9UIIRerej4h55PuelZlxg/FkVf2CTX1nFc2dUMm88mJHRWZu6N8BiU3Ewg5e2BDHiHhnEuezZESeTXtXMZveLFwz2VB0fLJrxdKx+XSw5EXY9reuflxlY2LLJf9Fcs57C+fQDWTGF+WL2eQS7madiC58TPxQOYIqLKetKFF3f8hWfHS2VBp291vHv73r/8HhDVblA==", "saves/botwar_t5.state": "...", "saves/botwar_t11.state": "...", "saves/botwar_t15.state": "...", "saves/giants_of_shaza_t6.state": "eNoVmXdgVFXaxp/3PefcMiX33kkltBlSgISSgFioCU0IbQICIiKJSAk1AVGkmaAIKkqwglLFhvIJEV0FUWlrYa0ooi4qWHZl110nsq6riHwvf2RmMpl77jlve37PpL7USq2pb+hTtgSAhmL42kFZGhZkOmBGs0sgwFjelTXVc29cEJ83LT5mRvXiar8NglgCzT5DyYUGFmw4cBFCGBFEkQYPPoI40BU+G0QY9xu5iY8YwhEkA3ltF6CdjxsABqowRm6EC4QSUNeH4bEsVdISVa1MbyDzXtTrdYhOBrfNQkrzPsjdvMGwqAd5eXLP77JRFVmDDHoihGbPtwzIotAsuAWglkQLEejXoQLKv0BT2iEVdgy8AkQDNw/eSwj5oTi8z2Wdl6gClXHKoPc6IEUR+dQQhINoAmlDgIwOtC49zUFkFlS2rMKe3KYGgR1n3Ya8Eg4+Qu1aitxKyOmCONpOplrZHXy65zpgP5p1voRmLCMf3lEJqyw5jjOQyvgPyVuRRlA3NhXAXYRCROcioJOKJDgSRT2KUox5jBxErofdAZjDyGtNj1n5wCyWu0RslJE7hFtjawEi59maCZrVETvLSFLTAoa5BIkIEqhuZWaxVbOBtnoczaUE8WZOgz9gA+E4HkNveLcQzGe0DqfZI55maCJTDiO3J07FINnlO2AVAq+YSyS1OyQ78OfuQLMJDWWMh2yda2FdS7RUm4c+we428OQtLee4eh+ac/lBWMOg41BXM+chVE1ujHENSzrIO8kBFrDXG2SOVsNf0B1VXpIdLDiKD6axHOSPC+9IOHPVGAQLLwdGwl8YR6U3S8ohhPpIBuEo4cot1E473yPVI8LeAnLQC8gD22wdhVPMfCskEV6xlGU72RjwjtWOOQIMgXejRh2Z/gvnzZ/VBFpGPk/dx82EHnA0YjdkSxyRrrs/g2Z6g9EJvJjtR4G9TBMyEUQDqn8U3nfSFEfacMpEXfL2KR9/ZQlr5E3yKRNvo6yl9NkBLg3hi1NU+29yWeGNr1Bm0QE1Ain3TZaAeTjKEsrIQ+CXSXbGb3MV1/+bHcl3PxNB0xCJwzGYB4B3i6UUvwKmwWbwJ3J+dndSLeLZmMJ1z5J9ljDqJa5swWGs2Y0jc2D1Y3wO9Ae2M39HVXQc7hTOx9YKuC/ByC2m/J19PQZ+xSJqNlcQ9DuWO0GjgrkAoWaLZtchvohOoP8silZ8jOD0tQgqRuD0/6yA6myMUSgk/NAC/pAHkOB5CxGfh54qnm6K4e9r4LhT0RNYyAMrSM/tLI0xF24+HKmRJ8AvkZUHs2YY4i0wkCv/RC6cM7h+KNicpCUoiVQgl4OBHRF/QabJqeEyN0gSynGkVWA8BRWT8cCXbH0BmS3qnxgTqWDV9v9Ac1Qw+CXVbGrIm2NnSEb6atgvR8wqwqRZqJvElJC664Zg8v9waDXyiXwgnfHnIRqz6OBukDNi4YI51XMWLniKmiP2DrKGQw9EpMrK5kVAE2MRIpuhx49A0HQCye9loOzJ4xJC/x+pGp+TN5BvUfF3Uaim4u9AK0yXIGi/eLaqZqsj8FaVTtBbJYi9ea+quh0BNLVDUk+SsfeTLmXv7pMo837hiKK69y33ORu/fqaa1TPau15G9ypbpoA2LzO+XI9kGbtYHcI6dZLNZEPSBYaI6qex18vYyE/yIOtFhFogeGCGHneblsF8UDXwNTq02TZPHYHf5xNqKlVwf47odVFaNVgFa3qr5GFN10Q8BPf21aXzEAoB6yjhzniWg5s7yFKrdJl0YF4PNYhag0+aqizp3bGmBZLXMZfjGRO7doUp/87GdW2p0lmscd0ytW0t6Cib+wlVMjc3s39dliojGuP8gPg/9H/MgJ5s50sh6Sd5a3uK1JB7jjDonotjbBCjxZ8RDNqE6k/IPGxL09CrFLS6hWMDVqLsecaAbJQy7uiODycoFKAnwT3o4ud5KojPU3X57Mp4TfxI8wugB0ENfZZjV/5IwWedpG7r0BAdws6LwJXLMLARbkUOymU4r2wPbon6p7TXzwow+YBuYNtY12p97aNo2AuvJ4cwuBCJntyX44vwjql0u4iS3sYJtW0/xW56A/1fY9w0kyt3SILviZnyexjLRsG/e4Sa35PhHvWRZ8rqCP6bwIsU3N3byosFF1+cNyWtWILuF4RQmsBjSZRXg299nfyt3VDXVm1RNrbmI/F3PQ7Bqj/pZq2gvrFk1tqBwS+IyghtNwHBnRNQ+Zxsc34PXN/NMP3bPGLhYzVmGHDGjFPBjVHm7jpYcI4bLmOY2cDEFshOv6D86zpywtKS9vGoeoBCmEMfUkJ/lgDvocaHiDYon87hFSnK+YTpjCJ438jw6ABa7VY5QXIStu0HZkDVgkZNtbZ+xLasnmTO72PXpuEVeF/Tj6jWX8I7Tb/iwEgWkddmH0lfr1Af7mPcF+EqxcMIbVYiyNpnJzEFZvgIYL36MMm2DIMrYUSdRjPVjWbIBzuWIi/Jf1ED/jBmkAjYAG4cTt4LaR79w2l0DpShn5BPMmTj/6Rot02FdZxI1sufYTWzFLUWvX5a07dGz2IqWg2/aG9INHUG7dfxL2R0NisR44eo/YULo4EC0qNBCx9Ds7MIVhFhMJyRwC3sdYCWHqVN92LlNrDH7iiQ/ZMd2Si1QN4EaSrp775CW7uBz/INllhN9hDGEhi57Mtc+/5nka9EStC+lS3ztxQyFavMoEU1s2rC4Bnw1Z1fhPwrXtQpHengZ1sLgW8Y8niTcr/hHa7/qBy9N5W/znrZHPjfPENT8BVZM+kKCr7J4crtUg9vVaqkYlrCR9H0BeEd9jrC0qNbk1+UwWcRmF8seylTpcSU8JdbHIyiqmWIax0JA8cpK37BNGuhGGtQzQ2182Zngq9EQneJtIZ/6UmnMrKKdPeOSF2MSFwJ6HErsh8H3U1KHuMqSmVbYU2fSZUqH/RTe9R3AP8EtwOeRtB+JA7eAl7IVns4s4n6q+MoYKkxk+PhEehN9roKEQiR57f5ViepvLByQygbDrvwdoV6DpY3UPwhjiGYWs9+95gM0i0mOku6My+C/dXYtg6oJWMR5lUj2N+Pdt9MZo4IJ6GGYtjWEZG3iJ+2nKMCtkWXcdzOGWdX6b8xbTXSTn1FW9sNpBRtgYgCHoJzI3DORocl9lURKZBpk/m0+RUYA/6ArF6k8n9FkN3FLsEq8xfVbYl0hB0sudTsHsvuZBgR8s5UqBO3YagbcBQlQ2Rwhfz8y+l6b7bQxYIrI9u6GlHs8EhG/8dsf8HlRrC9FP3V7p3k9SSPNhHumqyaOINDx4AYYeHJcHw178GpGaBirVczScEu0kN0crRQac/DiAVL1O5IUoQS7mKgMZ2SEUnMG71Mgi45hsZiRKPs7SzICJwExrY1scaRVmzt0nDAdRRfSpa79QnoBNAXnHci+lPUwRrmwQhvC3sPZHlUTHiavTKh/XpQJ254hr32IHr/OVUm0vfctzyA0UMkN/awDMBjIkj++j9TJf+m3QM+BeuWuFsNyfL8NaNxJFLygWpGe0TWshkBHplSsFWq5UK46+FNFXL9FGeN8HlaRfWCeXNrq+fPmllz87y5+aLGfmcHYfM+UpdSyEILjtmrIFLtbzmgsF7KFyFBTZFLhj5hSV+sFXIcXjN3ds1SpFOFEZzzda+wtxReaUZBjxuHItUttBSva+UMP/zC3JqaBQt+k4CHNP5JMXc5sAI6O6xIQA9XfNLrWR+h5djlanvk4R21NVPmLUM6bnkVqcyYWgbvZor2a7jt7DI4W9QFa2T1jQvnT5X7arE817dyxxW7zS3UFceOHWsB3Q+yOV1MZV2HF82qLU6swHw1OUtwiGuLL1xBl1U78SJe4aVCnXIDnkvqdhK0oxVkaou9fKmiZ0xrxDfkrg7HN+e85wQ7pCmE1H/aE6oMyQR9MrCDsf0iQbtOHGQctsu2GkFoVebgEXO7V58HJ9OiXTqCjIICZK6XnwdEmdYIhUffj3mNxqMm/Tjx4+T5EsjsxV61WUYC1y1V+37mJi/R0slAMH6A2tYIS9yebWG1sG4m6mfDzhe2TYuGU8yvWD7dmNVA1qj5s2bXzM16wq+nP7rYfxTq4YfMKfNjGu4Efd/JEvx6NqFTobSNcunbZI+DLkJ0HCBhD5Q1Dh9FD44Dvwf3CYozBwvgf57w8I1TQuWWj8RJaTJ/b4ZJ4NQW3jZcYQqpraCvLnFKT2ZHrlKJLLIRfDVRJXKzPoscOEHR3+FFQh71cobq2/aQO0kAhLdH632y5xeKk25tgj7DDO/y8qZil9euL+3y6HPyx8637xPY1XkcOqr/Zfe3ZDMdfX/GRlTqB4HLljmNP1umAFLbfIVbX4wsJdbrht56K2Gdq8aCkjTNqhPQWSDdwnaKBojWOTgDPVKZweSsPqeT4nei6xFLaethYBcLJHt3MyxZtQ5PmSa/JX3T2vwmWmrejgTr8zjYo2ngXDGZUdwfeRlwSbeiv3qYkza2idxG/ty+nwexWGVzJZ3yKmvEr8RPeVujdijRCWXA4KdjjVxolPx2XfArNWfmXGK/r2X4W450ENG3RFZ3/A4Kg+TjnaBPMYbbyZA8XbHBacebTnoZ75706v+nbPGj9l6RwTK7bDr03p4Bro5Vchfg+eto9wiFWkTOEbWNuNw4SskAscKj1A7T0J7D9xpY6pAWBoo7qojrj0DE27VVEdV1Vq4c3o1gdwVoUobZHUZEjXOS3wl7EKX1n0LeP39wg7UPmMp/kDpGW9JpJudc3exNXs7YqZs9TlD/TrH0x37xyrzH2D4cqbRKxOzW2fUTFVpIakK+W/aottrXqA+4PbxpnWBeIVSbB1FdS54gxLvwx07kRl5mu98Vhiaanmn+gjxVqnFNFLHXovhwDEuCrBC5ixijq9L8eb+ny0Ar20J88JR3KR2kAc7YAyL6sVPeAJlPVzVhD77NUZWaxmkM/CTEEx0M8dst5r95OE+jyyWM+lI3jqUWTum9mqIQK2+2AqvATZyrLI8cmfIrbwrXrxTOILej04SE26C42dlC3kuZjukD/KDRB5GR+dzKwvM3X1TQq53kQ5yGgt+9lAo9Juck/DssmI8NtvVYp6dU8PO+oOwWwtCzFx3uPjckUMxWscL+Qjvh1r2NZmFFHsXWLqLDyhwn5KK7at+evru4b6xB2a/w+nzoH6BJ4r4Ha5bKLSpy91gzWAugbuqM/N/ogHS09TWH9gKOSgKL0awayS6fP2vh/OsXVslAjRmKPOj5wxLmZ18bpb9XzsYobibvZtnNO6DtvpUPazHhGxzwsDTS5BJ16yOH9fqrn5zgyK5YcoJN/Z23KL6McihZbOw78KYd76CG+FVaavm9Pico5b4sE9hzxFqoQkUnoigid52YcNUDyS/Y229aAcsi9X8CnacQq2k8+iEIwJTQKV6n8UhIUk4nYUaIKK3n8qE5NH9kKNh0X3jMec1j9IVI8Eo5182H/Qpjqo4GM2IoKW1jLw9dQqbwdHqwfropO8i61rOU/PVrNy2IN1kxR54rL+XySqa5fd3YqJVeMOJoRvwWk2tVuQ8TWurjSnCHXmNL4Pyq/1CK1fcmPCwuuI4JwFPQB3qgOcixxgN3CIkgKiLWE5Y83mVEcr1qOfVqkx9CSq2jyOMUEt+S+A5bd+aGdjK+JRR2Ck2xUiSTeENa7I5DfnkW4/ZDfiL0X/EOdlrZnTC1cy9+AVH3c2s+HEZxd8Q7CmYNvoHd8pl+AuV7TezpI0Lgq9jKkz4+onEZWBjt3OUIuo4gPzQJpQarokjcJ/SSN47Ph+s2SPlQaFx7e/WXWsATH9BfOW71vT4L00K7y401Feo5Yxrpc77/VeAIRWIqp116YxvyljDzB14U8QeQRONSyRxC1LV3OuK7cT7aSLUsF6Mr1VEJyvVZShykJltcqjX8SiX1W+01ydAccDXiH/O3Ku9j3hwEH260ynvl0bQ7rbzdtCmr2fVu6Bw2fUM4xmJRuIotCe1HCt67CD54yPEntYlVqV1ZeAaRjLPWs6FAbyyIz66Wd8pE0g7k2mPM/c62csZYQTnQxi3RAyOJaxiCIVuh/uTH89V/vYaPE7D+xlicxiWLTdoOU9e3Da4TX5fODR1Y9yDqZYalr6Pon9gpEnSsMKkwi0Y+ZOkOrfEjVCfgUsJkpXMJA79Dc8S6jfFflrrg3w3uCnMP0O8mk0Va//up8evni2V4XpsON+hTeij0ctDD4L4ha/xkieR4RJYPyS3FAlwAlajy79lZ8mNOPKI8JM7wdiceVUN1+TqifmeMvC7D6G+BnTTaVLvdFE6Qk379HuiTRLujzYJj1r+YkloXgmQoHD4djSe5BZd/DbS7iWJv9w4F39ym/Vcro9UZ+nWDV2daKed2ZX0NJcN/Nniz4QJEX2V6HnvSvpaDvgbO5tAImKhZapXyd7THkjGJ3Oz5h8jqyqj6QPvqK6d+DPm01AqvLshJMcoJfUjoWcvsSMF0zSYcUSukbR6GSbDObIjexZRwsOnFTN/qRLvFs3E58XZGn5a+Hy0iv/pHP2FWDKNg+5ygaiN5i3ynfbRsP+G1IK0pMi+PflKmDeGP5bnBtM/trJHzg93Wug70lKVHgor+lY65QcqSNHQvy7b6Eu4jKgelQx7vDW0rB34jXG+hssguGwDu2ipbZurxtbD6k3AViTnpvCRg128W48w5HBoFPkVYK8wYi1v5Y2U8h2oMhkFkzNrP7mqRmpJwJQ+Hu6uSmzMeRmhmF2kOXMjF3Qj4l9ZqZpeAHs+JU0avdJyQRhrFvNZKRdor6gwu8o1oqABRJ3izu2S7ZyTaanzEr7g8lIR4lCm/ccLM2ojgzBS7coe4qPe/1Kd/orQGaQ72imSS5/blPYlCaybTKnIf4AnB7uUIze2CqYRMr1Gn+Jhjya+/gO9jPEk8zPyRlYqIGfw3bDFhN5Eeqry+XYbxZkLC4j50olWzLR2zBfgfId/BiyApqUehC2vDqeifoR5kktN9CEiwthnnJtaCLBUwN9mxb52vS1AFXI3IFYrvaUoPDlzO/nIrWmnGM5o2WY3VxtsogFSKQXbJDhM9ESs/wLh3uw4OHHJ/4gli8DVagS4jvABzjArcpJ7N4nXED7ei4Sr4QSZMEdTs4Wr083AvOtZaLTq9itC3Ja2I+fyrLRCUUTsoSOrN0qwuEnbFXh1/Hl1iyY2wAhoW5C8yXTkYdRR1Y6DfI4y5Ev6ovREMdprdDETzRbJs2krO2EXiVWoPv9ALdqxU+dc8m13pzATmfxq5UZ4GkDZRNz8Dk4Jg+VPuTcuZhP77XyuAdI2MpP+OjqtiX2QkklKdgdtgrlAJg1+zTZ6FWGY4lRYqFht0FtZpg4Y8riY+x942MRGlhFfMG+lbP2HrH+S2C+PcoVAK9nHGC6yPsznI3IM7OKnY7wyHYEHw4j7BUzLlN06tEedHK2FH0jAO+2K+M8UTudiMD6oln0xpoMmMinvSk/oT1pePjyXQNcgsF4Acsjx99zbYP0jMWL+J74P4GtrPweh/uWUVcEbLGE0F1HW0SgVcqY8wnuC08udG470aye1lBp1HhPBuTmzQL22Cj/5QAwtUqHAeBV2Om9iOmrT8v/JSimeapXQgh7xLErZZCepGg9KuKhe/+6QT8/+Z6+22hjOdtdxnCNNWqipGL3KLDmfLyIkfFw0LHluvyt8wbpdWVvAoxQKXYv1/YDwiLy/p5bTdQDER6p0dPRpHXu+ul0jA/f6j+bOg/qPQYQseMn7hWolRYdjiRG7KNIj3xcYLFy4YIV9oLb0a+0t2XKevcUoDFfs0O/ByERPk3/4y6WzweFB2IRrVHRCmlk2/GWrOeq1rMcYo/80lQjEulbQt8RD7o2Pab1Bvd/V+QwfsVHiri0r/7Wo0t5A3Wni9SzojT/sL5KZVURrZB2PVSEkizkMNKfHOo4P+ek289jxKvFdKOt2YllJPv65J2tDQoc7oKmzRBUNbJlunMVsnse39EleTkAeOzUXTOXay4UkJoc9CoRPiFl7/0q5aPMYvFJfcpwkV83R1GQxOQj8zF40n0c2r0FrdVaL4tPhrn3aW+hnlJkHetjbYJ9UNvK1i4b+5WEG+ek3jpOkdjr9qXufGlcr32lu0LP121IrFuiDlV2CXsftwtCd89x59KidcirGewCRe68tt+6ikb45E8V4GcrBAWZjcB3lnsqrKDXSXA2OtZnu09nDB+jQPjwv3WHhHhA4DnD+7OOQFazN5d0twBD7XpvGy1ljHsY2Q2Yc57WzfWakT4T3zVMM0eHvE3XebjnigLQSLl8S2olwNFpfV9RBWpZeNs3Dtk+Eynz4ObXfrX4Zv3rGxuCSS5BNA6LxOKV0C8Q3hLYR68W2Ob7bsy8ZEKzhzTAWTbsjZdrXIH5RUjvdoq/pzhEkQzzbcnK5zXIgDgQq9TPh1qkmZDuIdEG4lWgPJnqiitqfvTQtoMDWDA0kCZ6n2hMuBX/Jy/PHb/VI/7Ux33PY5sokIx9Y5ZTH7bLsQks1QmTzF8vUG0ySXWJnMcuGgbVoOguUtwwf8tL9bkvZwS8K+iYirqheB1pb/5V1Srnaxg9GiKZ/J4QqUk4uscJ7yzVVUYkM/OTbRiFuhphBmyE5q+8lIJzP8+xbY4pW1Jry+hqqzrALCPelo2ADfuaOTdWmUgm53YsVg+NYuxUXbRfw/4rwbaBclV0KNpcva+OnF0aQRDoveHOtGHfSLGRf/WfTPdinKCs8Gyi3VHrgEfKdcc+os0ABINp+Gmk5TYoFXE0vGSL3JoyPCwWgezlW+PU54fyr6u68zPWInO5O5h/7ilH/MemIl+8c/o5ISC6/G/ekxKQmrpJAGH3Qu/n8jOevid43+K0u5RKVUWwT7clD/JazVInWC0/66f1mJ8Gqpq7mjEOeBD2bEMf19bqA5HbAdGPo76h6HBIpn7kHd1YKwhqJukJwoMECZbrD0dN7NEurroCacQu2nCBuFti5K1Iu0KVI2Arj9fTehuiQQPPoatm0mdy1ZCwl7nmT/zD7Iqbppvnk2YTWXdPetPZGUKdf6QcnQi1Si4VxmUW8LHW5JfMBdVKMU9sWvsVbg/wE6+KxC", "saves/ankiian_t1.state": "eNodeAd8FVX2/zlnzp3+3p15816SlwTeC0noQkJRQMAAgiFECMWClSigSEloiggIFlCKhhJERSm2xUIRy6KriyKu+tNd7B2xrG0t74F9wfzP/j98ZhjezNy599zv93u+X5a+w7nm/XMChJprAIDBIFgfyBXEzjmlIYB8TC5R/oAygyGzpk+b1jgre2bT/FnzGqfNat8ewkQHyCcJDHlVgQkW2OCACx74EIM4aAgglNd1DwgNEzwZXcmIASTA8+HRUK6tWdDQHibDmw7k0oa8R/JjG8I4wJ6AJaAJTM5CtQK/OxinwpA2QDoHctbfgGzwCSIKocQYCElMA7WmYGmsEahQ3n8Tjva3APdDkRzDZB2qFFhemQF2BURmd9AV4GBfxOGoZ4AV2h1grCwULEaVRGsGRE6FjFwhI984BnLkWqArwct4laDsEZjxbThqNEEUq4SpKEtva/urPPrISZDLxuW7TeRUAu/A3rKelRSvA2ovMwc0DXwGi/V8qpSCYxh2gAv/hFw3PRJ5H2QSFRBkIjlFyQrIKJlrlKoA3SLPylILHOhvgL0XnLcoKqyEElUJp+/Amp8wU9QXjmXTFeDPBLsWMsUVMvgUVLMgKukIAxDs/2KRUQ9RaUc4h2Sb/iwjoxvmu7ezIF4Oup4D6kUD5TM3MZUAZ0CXgx/SXRSv5iJcwfgu4HmG9w8jxFVK1hHhyBW4lYtAG+ykVcUjmBttRoh1oNZzZFYA3QByxm6kOkLG6giJtN0RzKcN+xPOSGGV/kkW6Q6E/gzWTkM9ALySaSG6v7GzECO/Dvy3WC+HMBMbKEt5lUzMD4hXgv8Q6HVmGOmB4H/MfAHgCIv+DZoFgSsgCs8AqufeJiT+RHRQvk3vWlwHMgF8nvgIR1E9UHMg+3kacUdIJzsKXoml2Kn+GDRPExSmCirB3c+COuNZpq8wVg6qgaMiGboCeruQJrRHcyZ9NeZQ6kyb2KyFEJ8zuLuaj3mDZoH7J8iQUancrQOn4iDm/HblEPsbcxkQQUhtVqhqmD+08APk8djwPegy7gI5P0OyGxlDThGXAw1l6xbAocTH7Oh1yHU9R1mXM97HMnXezlQPFUoFoDcZtBatkTPnzpvTNO+QQJ3LlDDMHgU8DBIbC0HtwVx5b4jgRjdFNyvrLnAOQGQfAD1O+L0Y+BMOaazt3sN9Ba/P2Sm3Ftzt3M84GnkCqm2MnQxVBxm/Iyi8zeBqJxOTPY3iHUFvfIA+FbTo29mnO0mf5Fi3YG5YBVhnMn5nchfmXWzM5aQpE/XImG2e13hJ0yVNDwJ+JEj5xrTqgIdCOLkATMGV0Rv4Gzb3g7kZCqFCdkgu8yScLTLK6NmLKLsfe2K+SjaJ9edAIipRoYBhFiOzQ4r+JZqhPDpBWAEpt6PSvoWNUuRpc6etBpwCBe5lyP2ynOtwGUYkXxun8AJDj1NxUwA6HO39Kiq1UfdUdkY2TdMR71QAY5yi4jeNpVRpCV9wVh/KlVAtiiKA2QmxmcrA+LvyfnV4BOBdjLJ3tjL/AJwJaiYLAAI9QWnaAPpjdiKrHqSSdjOkZF3xfczPqxs55+uPRYCe5JRbAbNFQkRLvWpFM4Af5xaskoupRs4ilM2P5HUt0IpiFRAbG8dnj1FzRjaDceNMClKbMG9vclyagbqNPeGhvpSNKDEdqLYWc+1OYzSQH2EaSV5HcEaqKCms6AhWPTkydkpYMwekGnQF4GzM9ykHVi3Amxw6DGl3s20/ZqSKRsrOsxcaRXSGLao9z8EqmxucqLgO9BFZY4ngZp7jPutkRHOc0GgW8s5k9UEN5qtDfgrS6n5f/9cL6ey4rhHoTLKF3voUM6TxGMILkT01UhOYlzA1Qmwbq84Qqs9Af20Y7M/nI7oWnJRVAe4ExrPIUPtFXuhrw7mbVAUs4GabvzQit+IeI0f6bFbcwkZHSDqCwUasMfLG+PkzJzfNbByI6FKB84PiUzNUlrLW0qTXmN7VI35SS+8Cu8iNYs99SjnlOsj/higujP47yGf16xauJriX84kwUlORl8QpS/pL9qStxDKJmdJrRjqRSLn/7CQjTH6r1jspkfLYJuZLsQrnqzXcIhKuF8Q9Ov43c5iVSddeybMPs/UyWnUTZbs7q5cwX7oH8R6HD7NeB6QjCqJ2l4CuUWQT0K12bIhTgG+k5JsGTipVOW+XGz/MrpTiV1MbmpXs6zaObyh064sp78WTyvoIQnOXE5fte0uFkSXUieK4fQ/nTFum85hRFjnyWztFns8FHJ0JufgANi9ifFQaI/DgCC+xcEmhKKpzkhP59TeYedOCPx9kGm/xHEiJLpTqHSnVPATH+RMcrBvFs2Xrjxj4j8+M6qASblG7r47Ti+gy4wzBZo4TtVAqrQx7Onx1URR1BH/9V1Y2k3xF5bo8CaI7KYGk2wx2nfTAChjA4I63rVohvhS0VsS/JMgUDQTFKaTdTAMx9hbPxf3FI8ATAl8nLbXkWjPbHQtgbWklpJ0LXKfyKYKFUJXEM2qcXERvGXqXgNnMO2OVEC92NaJdiBeG/DZHND1BdYOMfGiv4Ej8QKwO8OO4NJW0ENF6RTlCY2k0sc2R0WIK79eDORJ+hJZ8gj4CfaUDS2Cs9GS3shCO/JjALbJONqcDHjGm4utDS2lFlLEcNiNv5ud2jjK+dP+U0FrPZ49awH3Dzhl92vHgUvoQ9A8xl0Zg7GljLgjwbgXpTfpqK4gSA0GvZG3eCL3jkPwE8W1WUznG4U0HoGwqd7Lyha5MuEAU7hhKGTNSOXZ2GJkicTNMSb6KzcPgzII6rJGpXrUG4QpoKamE/qJCO+hFOFLh0KN+6Hxn8NmF2O4Brorajbjazik7gmvRX2zQGou/M9IoXzj5PsolI9ETox8nY8K4m3zjOe9fS5teWv/SniUH75lyE6Al1msElPFpjxrZ1f73Zg743gTts90TbL/LEW/ZaTSa1egehDorOPgWdNAHB5j7TUHmJWB1gsgqfwDyoN5HtZj5FcanlX6Aj9FuR4zDdeyssjPuOtozHIxlpaE1Nxn74XEjh/4I0CkpmXcv259lM0LmvZeDMdChW0D63ZeQDUZsdI64w9CeDvxwuV8JNOwcBf+i/TSZzW8RbzbUSn6dBDDvsNTb4BdYGnxFPC6L7EM0F826aU2zrmgtODs+3x2yD3jAz5BP/0SFdIfI6h1AUx3zDsAdntovSBaWbgb9JahMoSi3fv4oZDNFH1JY+yFNSosH+cq+DXdPKaQXDFdazlbjTfjJKKkV67TNDKJSKYVWWI37VL5qoXLOMfg+I1bJltgvfNmgGOs+yqG/FLj7GK8xMaGkx2v5JB61+H6VEqoWc7OLwvKOP9Mht951LsOMKV2VbmWERdjbf4W7YotdDgMJ1EjnHsihI3BbwmZK5qNHgJnZZuQzKV/+MVHwK4WluXGrm2FuNegFDu2eSqrqZCmSTtWfwP0dnMpazkVhFIogvMSyV5mEfND+wk6LUFo9GD/rwIdjmeQn9nMNWtiVEd6z3XwqX0f9lHkm2hWnck5tZnNtki5mOk5eIfsLHNoQJGhpncEHS/RI6AJVqZJ1P+NYutXg2xZ7VX4F9zBy8XZCLL4uQycMtwJ5lxHhk76QDS9Sqoy/oOf8tw2JLylhuJtpb8Jdbotc2hlpJ5aHeRZqe5PZfgwjS2Y/hJ0P6RbVcMKwM85AN1cuJVH6PzpIeQPBzRv8HRhuIZ5heItiziuSEmolS7zt6EutMIoLWT+O2U87+YLYXLYr45GoYmw0s+B+LlNhO3eSo3eI9U00yup/MZ4wkw9eprJBtNcc+xo4D/NUWtbRcScbfIg7Yg5+MexdFAmT/cE29cVPMJdOi3tfzni+hcNYLH3o/BLOdUTYCErmBtIe0iX1IGBQVVhGTz3gzU7bvJujdtfbr7vio0Qbfg/0Hwb09XLWDjiNgefEVSTWNCYM2uIEGeNNdWQp0BYO3S8S7hbGWJ0ItnqNqm+A4Dtj7OfglLtD8GhMGtdFhWFkC1p/Zn4GjAeR5lXqnG1G7jpWM+lTd5ETe5Lp5a7uR4wPXeDk/XgteD/Y9tmcEnulz1BmKpDa14K91JlOQ38xaNUnRlC+1V7qZiKBDw4dKMRJfuFdkhIkuvZXsXwoULzHsBYb+lYKMmIPpRGK93aoR9p7zE5g9lpnXVpsp73XThdf4HX4LXWBly+mQ1BBFEA1oNQggrArLgHx1WPmT5k548BjRYBPQpJGgpnwc0r/EEfuIR7/LrFJKIYaf1BBrKWdtISZaCyGDU6jdx+UcsUGP8CHAG7382ZXJvw/xftiVFKh90hiEbPFT2DofeAeje13EOXNHyLa1Ud6HW1BMZF4oI/eAOjqlkg0SZejHT1MDdezyNtFuEy6m3s94/KzEN4zqrQwRDW7I8A5Lv7drVkmnLwjvvQCdkIlCGvAbXZKCuO+zPR6KeYxIbbOsZOR1JB+Iuef/eI5J1kOAwicK+L2IhWlBFFC4PdEqi61a2znv+Y4vUzoY64D/OAXDoq+0PlgjBOJVvlPsL+W6UHyujrOYTsqkbx7GHQx30tDQu6C9fF8ur1dw1TKsQWFjvj2V7qnpUezxPnzWdJDPKJ+vr7UAFVk2DmbijjEx07Bf1VLPOFDRiRiQD8ZIyH7a3g1B6pKw/V+3jqCvoy5tpdbo5xZRmSLqXnCsH3VoRZ9VXXcMAdii3fcsPvakV9Zbfeyo9hMwt76kjLGn5G2xHJGieHH9eBYGAUC0YuF/3wfrdUXiiHdjrEXe2KOoxFQajQW6422pHKh7sVgyyOnEcQqDBUVLFPb7KhQACfVvKUPPFckLDmD9Qc6iNK9oQbU98YDbrO+X6x9Bdzn1ugbFPWgqHSnn0eRJEm6dkrxI6RPMezJlM3QH24+jExnGeNsPhsY2trOZ1uLNtSBcR8nUXpNnq1P66fNmjpjcWfcxnyNMgfHc8rXbqGSRqOuBLqRpdG4+2EkNjtDSH0B0V1eQwPaGXs2wXCsfpDNJbQ1EgT5f2HxtkNxdokxrFTPxkJ7q0Tu2OcFdk87uo6qPeXe4dYUxXGZa8GNlLP8h9mphJRouHfAtp9XKdFwd2yjWRZFebsl9oHnnIZRsvJX+IndlGDI3QD8Phj7mR7q465L2+MLR4Df0iORp5TUSj8k/Ba5ct8BrOkAa4vlp11SkhIbqsxMqez/7fFxnBusG4Zis9KPqyBDH1mHltjqbSMtll5nWE2hrf5wpmcYbSjfgC2SASVUYj71mwGuM2kaW/TEyb9z3kzZZ0BxeIql0xikxOa6FwOmV3mBW2LsB055MtlX2ZlhVONR269jvgKMLkR50Hc6NmJGN5TAlzwn0tPBH8HvAbRTNXq3YT5hDNVomHuMmt8hE91pZZOVJU6LuZLtt1hY9JQ11vwBnaZ14ST9pHTNwqMQFCkI0v2hpoYyxWNU1Y1Ij9hQBEGpgbu/Es1qJ3aWlsBmXXO8v3bbI6x1cvEhgOtJyGI601jPALSHcYang6VqIaNujGrETAfD42O9oYxvPAaXSOt2M3g/lqecpbZkyAYtipe6ymi4LTQzsRTA+Kh6TmheqavL2TwGiRnH4FL3MYP64Q6u0VFUfidWuVFyHn16OuuvnTBKCfobwb2bx74KzlED4zVJNKNfoWqbbf8Bn1ll6J2MdAHTKCN2RaORqP3dqyqtgN1RoKbb+H/2+gIWLJzWR3e/xAhSL6qxYwwzMr52G8Jqk66/TcHMxFK9Gmx6tE7nITD/6SasK6hDZC8k2KsO+bvAOSzmQ5rCdFjE8NdEaO5VHf80DISUtAAdg0QQgyA6hlVVhu1Yl4jhcwahZ/XqQXSvmhSz2c45v2GNngnuu2Gg9sWgDaqawB1ptwxWEHugIxPcoXLxUkPiyOpi3mJExaI8daLRozIGjFLfGJkShcGoL+IwM4ARTk6lhcTuvz1eBzykTKcNO8Lyaq8M1y60EnSmCUl8IzReGm3dCvpAFIiQqZx/g5BURHOHmF4wmospO8zdbzgSaKzaq1IVOHSXu7TUaGP9qkQBPYlejJk2DoRlsnL/FqZrz6SsVz5Nkq1LP4InWNjnSmS4GLyB10UBjrFhQXwPu3SdZA4LPzan8NJaoELDTOA2LwcZyIQX2hVRokbMp+FO5rP8sxiXjLWzmVSxEfx4xISzg0C692x/LOvRTnjMywXqoxT/t5rGofsu20M5Kh4J9F82dzNavlrNzyVhpp8jzxNxeghoNqrxlzdeNv+Gr6CxwBub4v4z/Jw7NhVhb+C5p5oSoe8EnFGcEMbhg6Ob1JDzCBcvNHLmxn7p8IjSmfIwQjhDd2NbbMcEPAoh9qrmo+0ioXysnbK+cst13XB70nqR1eX0ZljlW2qGwOW/LkzGFn86OPuTkVP5TmEOYz8y1dRm+PzASnkSgubFtt4KxpoikuSYHGovGwHue+y0wFB7zwdStengfsB402vG2kDulIuBxYec7An+SK8VofPvZudx2EJDaQ3gRKyxRG1ujjU8Odr1E5u8nFdQC/0VmH0MSjPXnBWJsy+VKWPK2RAFRUXOUjWA0yfTt+1DyBlpgdYUNiXv7mL6ucZL8n+CbJLH2zjYKSvteGfBskmMfwOuh5/cpWXK3Ahk4/epn+iPutg+g29N8Q0V9OvAdDi+wdkvy/ErGcd0Vh8bY/FR8SmhvTxj/D5BnSsmr+6riK614I94eGSmPWlenJ/m6A/16acGqXpISZjXzxhm1lJ1Pn7Jk9I8kp3R3yvow2Pjn8WdSiPdoDuL63iM4AcemvZaYLotaQResUL1tap+JWbmOMjkGEYhtE/AGwU5OIPpvXFzTWlfFhQMgH/rSQX1QCP5bmie0Y5PGJlCz/ktXiR5cY1h7+NUWiLvL7ZHRcPJ7OuuZX7bqBGmNoNzSexTjou+rxyeDv/ZHTuprv6eKUjvGebToFc571uf0UCgfizpEN9A9VI/XLoCrCTjZLURsxJvgqoK224Kch7QMjQnNC6cNvfqFsBpUGZX+MBTMvTr4uIOickXFR6Nm4LQ48VUeZYn2D5ezJshMjeDv5LpdmIrHfGwVGDfb267KBaI8QeF5yWBb7YbY5Ip/x1Mx/kGeC+zXr3KXloqnDU7QzQHxz4jwqv/f0cLxsD+egwyobxr3gepJjylVJlz8Fg2MrGTwb8zLSf3d7ZXMMcUFTsVliWk2oC0HM1zpsy7/Kopw6BAnM+4YH/WsK+EqHA/+DPVT9RSVAGSM02TW40g/VL69T+DIFUsTW6Zwhfs3kFDpUSK0g2F4el/BbjZyvtcgGviREB/HeOuc5yHIMJeXuxkZX/kXKOCuorgmMG8+CSzGazzFb4Z59s5zbE4PErZ22BYMntlfBflMVbDOP0CiZORoMivVm5JzbxJGfvvVuBIj3PBDCpVSTbjKWvI7S5v6FTY4B8GmrUqDh28sb707DfyXhRvTLbckaYNZ+vP2L3M75UKp/3HChJHCOqw+juW6DeJpiJf6RQVLvNfZmyH6se2+O5zGUyg5x0M7f1GUPhIGv+Pwo+mF8OLtHUx4xxDzYFM+u7CcOWTdlC8BZMjO0TV+nlnnLesVLj8nIN3nQga21VC2uh6D1zXK+Ke6Ft8fREddaptKzhEOfdOxxyM/mSmaRP0WwzYhjzfkeyEtVZWzdrF2fhPrzhjVZ3gpKcReqfj0gcNuoLEtN5X0tLbByc4i/Y5eylwBltlO3ksHhJ3n/Y6gtuAPAsak7BOqZyVQ+8eh08HOt2h18/TNdr8OjFJnyacCz+B8Pfh8ShxGW7f5dOIScU0wcXnOExephtAvWKkfi4dOz3h0fQfCvLDo0LR+vdt90G0pRG/Re7ehP++rSXn9kSKFYSp4uXgmowJwyBAH6KSLqJQXcBaYmMXyJR2ATbOBKpy6VqmJ2WSgB3WtMup0RBBlaGzEg8fvCxCMa67jSOUpYr28SpcGe+AWaPjNn8SXq8k1VR6Ze4InYBTvXzHV23zMKq3gVsmhYVlxZ549A1uEFmNIDbTmmtHdgVcxhAXAy1b9pXJmnn9wajK94PukHdoDzpnzZl+4G8zDvxt5guAn9gFPhMPutIo+9Cr55zjDaJ7Gk6JA24B+xeOvANAd4FZi4uMwP+Mgidu8GGFytLMPgXVzlrMxybNAudSL7o1GCski4LvfbwJQ73IGbuHnd95s39EAb1p8m9g9PXpckx52Sh1Iva8BN+VFEQFYoFeXl8aFE6ws4vxRidnUqtjbUceeB7XOvQquAnl1MGX6VzMP0U5IaVKxCcvY30MNM6U+taIB5sCt2BVuy7wgF7qTHPIvND9sk9YJUXNZkVGJaYHQAnEy6EDBr0JCx4zJ3n/A1xsoLXdqIeYXB92eSU2ePcwNYMBe71DbLuI066gVjEzsMnM/88B0x428010v+I9rIqvHwXWu1LhorepRuER8ISdz5c2y8h6E3hZ3k2bwHoW7A39M7HnuWyWWqigMuq9AmhTUU7FNrlcorm5U1jUjfSzhWYqrN2cDOxT7Um6H5dlomv87fcAP8V089n6qTG8NyqYBn4fpsmAY7EMJy4rhAmYy2BkHkH9NIap9EDwZjFfCDS4KCruBPpWJxEWLT8/NHIG5wrVGKBtZwVDEM80VqeC2c+AlFSijv6XTSrtRHgrlxhb2DggWJ7Ne/mf2WNOIEo5HKkVrRFT5sz57LaVp4NY4pebY4lPZ9jIL3Fk3Al6gPK+o9PvY+O5S/CIbTw6Gb/nMP1xYbwWWpI5y6oFq4J4C6fsSij1DltmJUTOg8U5YDI7QcbtJFs4ysl4D1C+8H6F3zn8aDZML6a0t5ic8ygTGyXSfpJLbzA1zXT3lNLlLm/V0p4fYsd0Up+E2dnOLidnyaaZmwroHqJ9GJvNCXzxSvorK67C7vyGnU2uOyPbmyqwffuGUnWlY21zKFEKa5L79UHDzxTJLPBj7MYNByFKd1mdzLfnWQHtNGinLGa/H3ub+W7EBsiUNEAyKu0MIhaxqJ38fYKBF/zc7lh4UqO+phScR5mqp3rdHKy0eRBHNGB8yHMd6yoRLbhKd6iHepz9hoE3otr0U0Fe6yFspSRqjPUEqTzvdJzdLCbFvkwa14Vux4mp2XYdJCKn46nZPM41ce1AfjNGDej+VVB7drrCG+kXz72E6Vl0L+lViD+440ai/iV4P5U48H6qQY8TxbPYZG7YxHYXfjWVk+3mUL9+Pv4HcRlJE4qS4pvWON+nw7pk2ZEgXVAB5jeIB1F9ym+bMMNNHp2tlhZ1Af8dQ9xwCfzbyHVOdwGH3zewK6SLu0JobOhhdF3gdJbydJZCXgNOVyFnV+nnE3xjXD9cBFG7ruD/Nqx0dwRLxboTqK+MIh4ae4GsdWLh6SvDvwXtpxJU3ejGDdwLqhkyYtUSmeFFWZ540IVswVLrWkaznDumbajIHvMqWOitf3SM2eawy5suvbxpEOIELuDj+hO77Jm509TufufSF9Pdn5l+w8edZmcz6O2Iz5Ul3EHe0CWo9/dNB97EMFy+HOFnXR1T5oyoaifbrU6e/UqHp4LxY0lYciymyx0zFXxo0ol4uLNeoVPQ4R0b2uf4IcYSVJdysRQxqWU2i3G6ewzGH9g1/cCuFYDD5Ndo1seJfHE33y9XEuVCmjWB+4Bc8UyVLtgMZprc5yEl1jZt7Pew4xo8dC5kpN6s/wOSSYYmdmfSonNisJxveTPmwByo8IuklLi4pCskjY1mzgFqRB4yr3HeHYDDoczoIUbMVDzwNshnlH2JikoXwGUo2tbWth2wa8boCtipJ31huOejvklhWHqf6xUre2M4I4RrgKd7RxxluWSPLoskvOnlHxaVVcKn5VWRGiheO0ZYr3hOkRlD63EV6+OGpX+Pu6cpv4+r69b4ZRthcBrfd8L3VqpGCSX6YTUOYVEw/lJ0M7GNJXmoV2FpY71+v9TNlNYrvUStC1dDrEnpfpmvSnPOTQrwOPOOON2D7stsfy8oFcZXpPwq/p4/cvKujpSmU8gdpPFlVBvFt9WD/1cD51yMiV7H6WifL2PO1CCVngfaZC8q7gx0JVgTGH8nQ85XSafuCnQVWFKIGZ6UoxxKGfS9rGkUpY257XF6drufS9IZ89ztrL9anqJbmtwuMacSI+qp6JBhbhc7B9hefR0bUg4Y3RxluXYbBOWVid14GJ0VrI8CWXcq2cyaUzlj73Wys2AkZ/2PjAy8Rp/6fwU9Oh5GnjTBSfChFfhJDmM/tg/LTuaKOL9WXhabuasiD20Opa/RbY5LtyxtT68vQRjUoXkCO02kzFqGj2O9nSg5WyVStqT2gp04pMJQ6zbTpe64QuwIR8287TeJE8MuFGC9EaWHgr4MSEX8f+HSXewOQ+N6vt/JlnROpXb7ysmIuCVF2fbb+UCv4vCYCgHjdY1zm2Y1N86ZfsW0q5pmVVQhZE5S78FW/K43vXAlvgnjl3Y+OmzLqY5hgUPQP1nZNwDzj6I2QVxbG5Ndf+CxWdOmzZ27BBJJLJqYcJYA7lxsbC/QcnHaO6c+tATsXfL26AM7mqdd2rQYCukFFxefC73vuBLOhd3Lrj8WgPtf2CK84SPzpkwXTeFanfB+h/HHOlfcO78bHI3VXdBJHloE13pKTbh8QeM8TIbmqU4vAP+ac6B857zukBtAh8gkcK+Ba32ThzYtnP8zRHSi38kAeiHoGSQydntW+QuhSVtq3NWTv6JkEr/vkTCvAd2MsZNh6u1HwRl4yyffCxFM2zzz6hlNC6f9KtO5uVciXAR6Nsaq4cTz83pjrtadjVtCxzq9qXnajPk/YJK/XlhgLQY9Dx13DvqLoclyefT86RcbSb7quoy9ECL/GnB/hS22Z0vqv7pp7gxKcWJwIrYEMtZ77PcBeLW5KZbX7hLYFfOtIQvnz5neuPAoRry0MhNfDFF1afni+Ythz2L4JB5zDz7QdPCB9zK5yqUyw7HNCSX6+7aKldy+CN2lsEXFzYlTZjROnyL1jJKzMCHly5gfsI8v2xadjfm+f3e1U39g19QDuyZPuRaKwpLPWL2t9GF2Tnnws5MXg3+tJPjAHCcGYW7jf+UjXiYR/Bf0N0bstBu3PbEI6MdEKeUoaDNHN86bP+c3SHI/TOCi27tOivV/4403yqdmcwXkig8wa9ZW7NQPbAj7nRbhVbEM3bOh7awKUsUZvrHQD1q9rsGYvm7OcPn97tpbDmYT8I4KPWUB/NL+dVH3v2ygJLYXZ1FwMKjybqZv3eDEiXgVbyjzIdd+WMCSjyfM0A9vALUJuGCa/uAcl5eE/igwdgLvjdGJqnZ6apbXYuJyM3fBIxva2q5UYu3fvUF7UpF5Ea4scHesjC1kWrXKrYRypCeipJLcfz9HuL0yQyNKociazFjVDQcY8YlgrWK8up/qBLTfTLQ97ov7wX8N0nLW77faqjtSuyt01XV8p1561qXxfgpX2DLVfo9Piv+D9ZQiRw0C2nFRXM6zI2alv91Qpi4F/Z8NDbhTMtKX2uiguGBkIW0r5hvtCCRvd+hIhF7jqgTNiApyrjkAaLnCOcAbCvVpQDjcpJOA/kg0JnOZcSRRS5/cOhfHLC9wRyFPauVrYjytnRmi+eAquqUTv7rcPLRKUoW+vPXpIGeiW46V7P6wWH6i19PxR5djHawPWt5Lxl9ZRe1Bbeyy+wrw5s21PgFcBZuCI7cuM0PK+Po4k7KN5ZkwaIhyqW1sdiZ8tZ25iPW0VlesIN8W0mjQV7SW4Ps3mBbxqEY7y9ZJKldWxAuAVi9SCwAHd8OFDnZCtdylxALdF8l8QQxlAv3zVtjVq+j3pnhn9bgwc2brQpUkevqc+Ckun4PGoPH0MYQSO2iV+ZPOdTiV6eMGrgBrVAt2MPhG5u6R8S07s1rpW6QPuo5Aka5Zrca2WWqGUGMtGiflyicoPbj1vXh/pMGtfAryHj8qPgX9eKv9FtOUQIZz/4MsmfH69vptdulY2i3KEX87RPutdepLou7rhBeEOpd8ajX3YfM44AIyNYdqjqN1K0sdaMwNPBq87283R5Xn5cooU2oHx+IuSVgpX2HWnVHerCtMUJ/D4c573Ntb/AfYnAnm5l14zApbLZwOCWc98Xs9I+MEF7Lt8MVRnjLTsKXcm341HijGicBHeh4Ocmor6NNb29oeUt8QTeF4dQsObn8L5ZIdgIew+QQ7Q5gWtOhSNnhlRK036ZECoDkSXo3C/Z1qbmYs/0LnsqH9Um8dcwwJ4PqM1i72twa7xbQO3OGttJypfwWNs5Vch5eHOHQR/Yw5qyXDj7AtT39aQebEZJR4F3U6+ZGxlKobeLPj3rqDGvjl0WT175irxTri0+ppOulfVpu6Tibc9kMRbmK68Da3K5cjVoK1Y1WYuCKk67jQutVN4HzGAUP4XIH+ufp+mb0k9nsDPMtzSzT2CNV2cWqtOznXyVAlrJfIiD3lifbZyNgao5FFYexvGaO0TAnq6+XeDXjXPOO6Hsa2MwotG/yv0wXw64A91SBvXMO3VNBSL7T7dSFpKisHpVXfOyLqDjSgyPyGaZ5na0WnrJNcGJmBsvbeERnjgC/p6s8lewnona1tCFvvxFWdzLfb8y0DL/OO4se+pIYYHzmL0j30hrXOSfC7/QGOD616jorblfv1Em5OY6ee9ZFasdgw+1+qY4v+tPUCyAXOXtYnNsT0Z631Zs81UWlpPf22wmpYi4+MirzakheyVW7ZoH3ZnDugvlR1a03g0cF8zkmaAPmOeuqw2pk0x8zq3WMgX9SjgqwHDL2n9ST8B1vt7oiK+/Zj7wL+V7l9fYalXZR0B+1svLhgUmw9o1N3WzI3+fPWOzF+mXXaWo5t5Ccu8k9G7g30+9l0fFBoTFS6z2qlUxvb2va5DzP9UuO2ttjP2VHxrzfpHqsJH2He2kTeCNral97py3+r5QmsCza2qQRWHMMZY92CjfQe6y9kS97Dbq+bSzrzj1PHcM48aFmSKMaviQqHDqRt15vT1tpfrW5N5ZKxxdf7SUUffeG2v0l/dJ7Ln7L+stV1TlJ0KbhXKfRHHUxkS3dOKWzWN0vYaDdoYyp7Ea7psSz12gi3501407t+3sIv1oqL5WUun3G9rgNTvSDRd9SdRTmc5qCveW+j/qa1RwesbK0+CSL9U/wiUKtAkyx0Pj7LSevYRXSQuXRdYXJnHV/V4yc1ZwXNNdy/MD5jCBkj+Pw63THq0Ev32fgV7t+BY180X3P54Wa2837eLzSvWR25Z66ngTeZx9f6B1eHhSVebPz1zoHVGPS5zuLBkEtUh0FhxA1JY8RatNbJQFdjiq0W5kkbfTn31UHnPed/ZL670WxhrOUboabnm7rjlYUNxb7+pJXTWT21HqrGnl36eK+jfeLM469iG/TdNtrGujD9xHnemHUR3rjGenZjZHym9bmgEqo3YA/xkQL2c0GPUxYtgvgvNv9zdcqsfbwb/L24wazfqDddCM2vuo879LJ9D+Z408ax2PEm7/e1UXHVxEzJ3vkGdsNclJ5ASze6T62mF4EtpV/Z2LYDd63w3l/L+0r0BS34v/8PzXX/ezG1I/f9tXawRh+X8l6Jt5bztdPxRcIvNvFjK9w5E/E9xzEUTbhJGyoWllRc4fa7pRDqoAq23o/PvWY+5/P+jdNx/3O+3cZRahrRgbXtYUiHTOL4BswHW0IcvCDsYZoRZD1eeqfd4NnHC4y2S/Gc63SP257AnJqD65cY33k0/GrzL5LTweh12zLV52dzs+iSApshpASgXBVKMnYhxXIoOUw5LDlsORw5XDk8OXw5YnLE5dAuJANXzApNx1ghGJvQLMzofZuqccO1yiHeZdnywac3aYvaVuuhd3RPG99tOPfePrtPkq4id0bcoSWiH1Wv3dntLAYAaebcdaYrd16/s6/ow7l3dZ1oyO9fmFRcZBC90ERjPtaldttK/fndXSYu6p0zbldEu9bR1b2EU+Htf27WTn+ombO161nt4Sh84BI98yF3Wi03acHWZtjatq1bO2hvU2P/Yr+DsX0G5PyV93TH6+/j11JlQAW3D0ZQpSYWmzpG575cfaiv1kOCtkJdfF+XszCAvFliRvRoj7hDxv336AKWde24/6QSyBm/xYgGfpguLDVFW3HX/dqbBNUn/6X7Hjj0tksZ7uWCMfAv2k3DoQM7euLkcSpGEa154E/o8I8dvaB69EPd/oTtVRbpGUVAB5wHIafef7hH/OIkHXvHfONsGdX49GEdL4IcX76zGx5+1hGr3f44yuKMGTurYfdyfWxXX/Oi7rz5q4sg98rq7T1DuDLbdpH+NM3G3eiuyX6BAP0gKX4O0AQasoDHLqBT7y0jWl9F9/55OplgfH7AnP533PRU2KMT4IYew8GQJ1+KrFmdadkdlaz2v3qWlrwOi+/Hi0fj+FOxx629CMynF0wBDcqE1nujsHo09YRyU8VzElS8CphvyD3ThH1fRoXJPqF1MVRa9HDDYzYIxDRYJtg3nMsbzqVB6TK7cuTGMLUC+tomvLKZZ2yjHx4a4mjwbjrdiEHozq6eAFXV0NhwSi7lUCcIxXWc7pjgDtrBZ+6gLxd3cq2Ce+VR+gxUNaxfBLkS1wSvZP1FUXGP80Kv5JQbvUL5qmfCZx/yNzWhLz3Zp7uPrJN22QK5i3xzfAecUoHDgJLDhyDE2rdiaO+F8TET4sltp4SJUUDGw0MUxI2HXg2tTnB63AR9Q1WUPIGh7gllWuZFFdRBm0sG0cRDZEMYdIJE0Hw/ff3noUBDm6D6f+mhH/w/z8TjOg==", "saves/ankiian_t2.state": "eNodeAd4VNX2/d777nPb3Jkzd+5MkkkCk5BJ6JDQq4RqiIgJTbESpEhN6IqCwQIqiKElUURAbOiTYi9PH4r61Kc+7A0Bfdan4gzYy8t///5fvnvnzp1bztlnrbXXSmOWMw2HlmuEyisBgMEg2BKVIwif278mCtmwHKL8gTKjwxfOmzOnbmHR2fXLFi6tm7OwfXvwYx0gGycw5FYFJlhggwMuhMCDMERAQxR8uV33AN8wISRPV/LEKMQg5MFDvhxbC6GmPcyAtx3IJA25j+RkG8IEwJ6ABaAJTC6CCgVeNzAGw/A2QDoXMtbfgWzwCALyocAYAnFMAm1LQGO4DihX7n8bTg20AA9BnmwjZR6qEFhumQ92GgKzG+g0ONgXcTTq+WD5dgeolYmCxajiaM2HwEnLk9Py5OvPgQy5FuhSCKVCpaDsMZjybDhl1EMQLoVZKFNva3tCLn2wO2SKIvLeenJKgfdib5nPTRSpBmovIwc0DXwG8/UyKpWCo+93gIv+B5mueizyk5CKpSGaCmQXxNOQUjLWIJEG3STXylRzHBhogP0wOO9QkFsKBaoURu3Fyh8xldcXThcl0+AtALsKUvlpefhMVAshKCiDQQj2H5hnjIOgsAzOJVmm/xWT0RWz3dpZECkBPY6j1IuGyGtuYCoAToEuAc+nHRSp4Dxcx/g+4PlG6J+Gj+uVzCPAsetwF+eBNthJqvSDmBlvBojVoLZwYKaBrgPZY1dSZZCyyiCWtMvAfNqwj3NKCqv0jzJJdwgMZLD2Gepe4JuYVqL7KzsrMfCqwXuH9VrwU+EhMpXXyMTsoEgpeA+A3mz6gR4C3ifMFwKOsegL0CwIXAeBfybQOO5tQux/iA7Ku+l9i6tBBoDPE5/gIBgH1BCV9RxGXAbJeJnglViKnRiI0YY5gsJETim4h1hQZzzL9BWGS0DVcJAnj05DbxeShPZ4TiWvwAxKnamVzSrw8TmDu6llmDVoIbj/A3lkUCi/VoOTfhEzXrsSCP+duRiIwKc2y1eVzB9b+BHyRKz5HnQxd4aMlyJZjZQhu4BLgEawtRFwBPFpO3gTMl3OVdZljHezDJ3vZBoHaaWioFsN2oTW2AVLli6uX3pEoM7FShhmnwU8EmLNuaAOYqakNwRwvZugG5W1A5zDENiHQU8Qfq8CPs4+1druHu4reH3OTrhV4N7JA4xTQUhAtZuxo6GqIeWVgcIWgyucVFjWNIiUgW6+lz4VtOhb2aPtpLs71kbMjEyDdTbjdyZ3Zt7PxhKOmzLQEBmLzPPrptdPr78f8Kgg5RvTqgYeAf6MHDAFV0Zv4G/YPATm7ZALaVkhOcyScDbPKKZnL6aiQ9gTs+WySKz/AySiEuQKGBYyMjuk6N+iGSpEfxGmIeGWKe1ZWCdFnrNkzgbAmZDjzkYeUMSZDrMxIHnbBIUXGnqCipgC0NFoH1JBoY26p7JTsmiaToQGAxgTFOW/bTRSqSV8wYV9KFNAVSiKAGZHxAYqBuMfKvSLw2MAdzDK2tnK/B1wAagFLACI6klK01bQn7ATWONAKmk3QELmFXmS+Xl1PWc8/YkI0OOccNOwSCREtDRUoWg+8KPchOVyMMvIWISy+IHcrgVaQTgN4doIPnuaGlKyGIzNCyiaaMWs3eq4NB91G4eEh/pSNoLYPKCqKsy0G8ZoID/INJZCZeCMVUFcWFEG1jhy5NkJYc1ikGrQXMBFmO1TAqyagFsdOgZJ93bbfsRI5I2VleeQb+TRmbao9lIHy22ucYL8atAnZI4FgpuljvuskxLNcXyjQci7gNVHlZit8PkpSKp7PP1HyKcpEV0p0JlmC711f9OniejDC4E9K1CTmFcz1UF4N6tO4KvPQH9tGOwt4xO6CpyElQZ3EuNkMtQhkRf62nDuIJWGy7nB5i+NwE3vMTKkp7DiJjbKIO4IBuuw0sgaE5ctmFG/oG4Ioks5zknFg1NUnLA20bTXmd7XY35UjTvAznOD8HOfUka5DvIXEESE0f8Aea1+08INBHdxNuYHahby6ggVkf6SQ9JWwqnYAuk1Y51ApNx7dprhx/+rtjgJkfJwK/OlWI7L1M3cJBKuL4+E6M+/myOtVLJqOS86xtYraFVPleXupF7GbOFBxD0OH2O9GUgHFA3aTQddqcgmoFvs8HAnB99KyDsNnFaoMqH9buQYu1KKX0xtaFayrrs5sjXXHZdP2VAkrqyj4Jv7nYgs3zvKDyyhThDBOw9yxrRlOI8YxYEj59opCnmcw8HZkIkMYvNixoekMQKfEeB0C1fniqI63Z3AG3edmTUt+N/9TBMtXgwJ0YVCvTehGobjBG+Sg9Vn8SJZ+hMG/vMzoyJaChvVgSsi9BK6zDhfsJnhWBUUSivDng5fkRcEZeBt+coqSsVfVZnOj4PoTkIg6TaAXS09MA2DGNyJtlUlxJeCVon4F0RTeUNAcQLpANMQDL/DS/BQ/hgICYGvkZZacLVZ1A1zYFNhKSSdC12n9CmClVAexzMrnUxA7xh6v4DZzDq1SogXvgLRzsWLfH6XA5oXo+qhRta313EgfiBcDfhJRJpKUohovaocobE0mvDtgdFkCu+3gDkWfoCmbIyOgl7uwGqolZ7slubCiR9iuFPmyeY8wBPGLHxzRCGtC1KWw2YQWvAfO0MpT7p/Qmitl3GImsB9y84YfdrxGYX0MeiTYZfGYPhpYwkI8G4B6U36CisaxIaAvom1eT30jkD8OOK7rGZxmP0bDkPxLO5oZXNdGXCOKNxplDKmpHLs7DVSeeJmmOK8gs1j4CyEaqyUoa64GWEuNBWUwkBRob30EpxIO/SQ5zvfGTwlF9vdy+VBuzFX2BllB3A1eqsMutni74wkyhv63U2ZeCB6YgzgeFgYd4NnPBf6d2P9y1tePrj6xT0zbwC0xHqNgWIe9pBRtMH73swA3xWjJ233L7bf54B37jPqzAp0X4RqK/riO9BBvzjIPGQKMqeD1RECq+ReyIL6ENUq5lcZn1b6Xj5NBxwxDtews95OuZvp4Ggw1hT61pJ4+OSjRga9MaATUrLQXWx/VpQSMj98GRhDHNoI0u++hKLomGbnhDsS7XnAfyvxSoFGnqvg33SIZrD5X8QbDXUTv0kCmPdY6m3wCywNPh2JyCT7EC1Bs3pO/cK523KmRJa5w58EHvQTZJM/Ui7dJrJ6G9Asx7wNcG9IHRIkC0tvB/0lqFSuKLd+/hQUpfI+Jr/qY5qWFA/yld2CB2bm0guGKy1nl/E2/GgUVIl12m1Gg0IphVZYgU+qbPlK5Zxr8N1GuJQtsV/4ikFh1n2UQ/fluE8yXmliTEmP1/JKPGXxPSohVM3nBheF5WU/0RF3nOvMxpQpXZVuYYSrsLf3KnfBJrsEhhCosc4eyKAjcFvNZkLGo8eAmdptZFMJT75MFfxKYWlJxOpqmLsMeoF9u6eSqjpFFEinGkjg/gZOaRVnAj/wRRBeZlmrVExeaH9uJ0UorR6Mn3XgY+FU/Lj9XI0WdqWE92w3DOZraIAyz0Y7PZgz6nY2N8XpEqY/KZTL3uUObY3GqLHa4BcL9FjoDOWJgs0/YS3dYnDLqlC5l+YeRibSTojF16ToL8NNI+83AnzcE7LhxUoV8+f0nPeuIfElIQx3U+1N2OE2yaGdknZihTDLQu3QDLYfwcCS0Q9n52PaqGr+MuyUM8TNlEhJlP5WRxOhIeBmDf4ODDcXzzRCV4WdVyUlVEmWeNfRl1p+EBGyfhK2n3ayOeElbJdGAlHF8Hhmwf0Sptx27jRH7xXrG6uT2f9sPGbG75+tiqLBw2bt6+D8jWfRmjLHnWHwES7DDPxs2PspECZ7Z9jUF49jJpkU976W8QILR7JYet/52V/iiLARFCyJSntIFowDAYMqx2J66t7QoqTNBzhod639pis+SrTht6j+3YC+oYy1F4Yx8OKICsSahoVBO51oynhbnWgE2sm++3nM3ckYrhbBVq9TxXUQ/c6o/Q84Je5wPBWWxnVxrh/YgtafmJ8B436kpaU6Y5uBu5nVAvrUvcoJP870Shf3KOMDFzpZL1IFoZO2PYUTYq/0mcpMRKX2VWA3OvNoxM8GrT9uREt22Y1uKhD44IghQpz456HpCUGia38VzvoCxT2GtcrQt1A0JfZQGqF4b4d6JEOP2DEsutrZnBTbaT9sJ/MvDHX4NXFhKJtPRyBNFIUKQKlBAH4XXA3iq89ZNnPB/MOP5AE+DnEaC2bMyyh9MoLcQzz+DrFJKIYaT6pouKmdtIQFaKyCrU5d6G4o5PRWL4oPANzqZc0uTPgvxU+GqSCtD0piEbPFj6Ef+sg9FT7kIMqdJwPa30d6He1EMZF4uI/eCujqpkA0SZegHfyNaq5lkbeLcY10N/daxrWTET4wyrUwRDW4Y8D5U/y7W7lGOHlbpPFCdnwlCKvB3XZCCuO+wvRmIWYxJrbOseOB1JB+JOeNAZGMEy+BQQTO3Ih9lQoSgigh8AciVZfalbbzhzlBrxH6mJsBP/qZo3mf62z0HCcQrfIeY28T0/0U6uI4x+ygQPLuMdD5fBcN97kzjotkk+3tSqZCDl+e64hvf7VbUno0S5y/gCU9RAIa4OlLDVB5hp2xKY99fKQ//rtC4gkfMQIRA/rRGAtFv/hXcFSVa7jWy1on0JNnburlVipnoRHYYmoeM2xPdahCT5X/aZhDsCn0p2H3tQOvtMLuZQfhBYS99fRixp+QdoYzRoHhRfQZYT+ICkQvEf7z3bRJXySG9E4Mv9QTMxyMgUKjLl8325LKhbqXgC2XDCMIpw0V5KxRu+0gVwAn1dzYB57LE5acyfojHQ2SvaES1PfGvW6DvkesfRrudiv1dYp6UFC4z8uiSJIkXTuh+EHS/Q17BhWl6Hc36wems4ZxEU8Bhra2C9jWog3VYNzNcZRek2Xr03FzFs6av6oT7ma+UplnRDLK026ukkajlgNdz9Jo3EMwFhuc4aQ+h2BHqKYG7ZS9iGA0VtzP5mraFQiCvPtYvO0IXFRgjCzUizDX3iWRO/yfHLunHVxDFSHl3uZW5kVwjWvB9ZSxvL+xUwoJ0fDQYdt+XiVEw93aOrM4CLJ2U/ijkDMMg3jpL/AjuwnBkLsV+EMwDjE90MfdnLQn5o4Br6lHLEsJqZV+QPgtcuW+B1jZATbly6n9UpICG8rNVKGs/62RCZw5Q9eMwAalH1XRFB21jqy21btGUiy9TrGaSbu80UzPMNpQshWbJANKqMRs4lcDXGfaHLbosX6/cdZM2GdCvt/f0kmMJsTmupcAJteHom6BcQg4EZLBvsbOfKMCT9leNfNcMDoTZUFvd2zElK4pgC95caDngTeGPwBopyr1AcN8zBih0TAPGpW/QSrYbhXFSwucJvMmtt9hYdFTVq15Ep36zf40/bh0zdxTEM1TEE0OhMpKSuWfo8qvR3rQhjyIFhp44CvRrHZiZ2k13K4r/xyo3fYIm5xMZDjgFhKymM4c1vMB7ZGc4nlgqSpIqeuDSjHT0dGR2tAIxrcegenSut0U3oMlCafRlgxZo0XxEiuMmhbfTIUTABODisW+uVxXlLB5GmLzT8Ol7iMGDcC9XKmDoGQ7lrtBfCl9Oor1144fJAT9deDewbWvgXPKwEhlHM3gFyjfbdu/w2dWMYb6IV3IdJYRnltnxKp+C5UXpuFAEFXzbPyXvSWHBQvD+uhu041o4iVVe45hBsbXbo1fYdK1LQoWxBr1BrDpoWqdhaj5hhuz5lKHwF5J8LA64u0H55iYD2kK8+AqhidivvmwKvufYSAkpAXoMMSiYYgGp7G83LAda7oYPmcohqxePYjuUtPCNtsZ51es1AvAfd+PqifD0Abl9eCOtZvOUBC+t4wJblOZSKEhcWRDPu80gnxRnmrR6LNSBpylvjFSBQqjZ30egQVRGONkVFJI7H4R4s3Aw4t10rADLKkIFeOmlVaMzjYhjm/5xsvjrVtAHw6iImQq410nJBXR3CumF4yGfCoa6R4yHAk0VtWKRBpH7HcbC4021q9JFNDT6KWwaeMQWCMz9zYyXX02FYVK5kiydekHCAkWnnQlMlwCoSHXBFE8x4bLIwfZpWskc1j4iTmTG6uAcg0zhrtDGUhByr/ITgexSjGfhjuDJ3uTGVfX2kWpRL4R/eGECVOiUenei7xa1uMd/3QoE1VHE/xHBU1A9322R3CQPxboDzYPMFqe2sDPxWGBl6FQSMTpAaBFqCZeVjd72XVfQV1OqDbBA+d7Gbc2EWBv4CWDTYnQ2wHn58eEcXj/+Ho1/HzCVSuNjNk8IOmfUDpV4gcIZ+qubIvtmISnwMdeFXyqXSCUD7dT1lduia4ebU/bIrK6lt72yz1LzRe4/OHCDGzy5oFzKB44pe/lZjD8A1NlVYoviFqJkISgpeFdt4Bxcx5JcoyPsNeMAfcDdppghH3wI6naPHA/YrzhdWNTVH4pEQOLDzhFf/FRvUmEzruDnUdhJ42gmwGnYqUlanNjuObx8a4Xaw1lQjlVMFCB2cegJHPl5ECcfaEMGRPO1iCal+c0qkGc7Ef/be9DxkgKtGayKXl3P9NPlaE4fxstivNEG89wigvLtuesmcb4d+Bx8KPbWKzMZiAbv0/8SL9Xh580+JYEX5emX4Yk/Yk1ziGZjlfKeE4n9YlRiw+JT/HttSnjt0nqPDF51V8FdLUFv0f8EwvsaUsj/DQHv6tPPzVIjYOEhHn9jGEWWarawy95WpLHsjP+ewV9uDbyWcQpNZI1upO4jkcITvKIZKgJ5tmSRuBVy1dfq4pXw2aGo6kMw1kI7WPwVk4GzmT6YMISU9qXBTmD4As9LWcc0Fi+Axrmt+O/jFRuyPk1kid58WbDfpITSYm8P9shyhtNZl93E/O7RqUwtQGc6eFPOSL6ftPopP9GN+youngHZyJ9YJhPg17vfGh9RkOABrCkQ3wL1csDsHEdWHHGGaoZiyTeRMvTtl0fzYSA1qA5qW7lnCVXNAHOgWI77QHPTNEvq/I7xGZcnHsqYgpC/8yn0skhwfaf+Xw7BObt4N3EdCuxlQx4ZCJq32PuvjgcFeMPCs+PA99o14UlU34RnYfLDAi9wnrDeruxUDhrdoJgMdY+I8Kr/39Hi54Dh8ZhNOXLvebdkKjH/oXKXIyniwITOxr8G9Nacn9jex1zWFG+k7YsIdVWpLVonjtz6WUrZo6EHHE+E6KHigx7OQS5h8BboH6kprw0SM40Td5mRJMvJ9/8XzSayJcmt0bhC3bvaE2pRIrCrbn+qCcAbrSyHufgzREioCfOcTc7zgMQYK9QuJ+yjzpXqmh1OnraYF7V3WwA6wKFb0f4Vk5yOAIPUVELjIwXLY/spyyGKxnnXShxMhAUeRXKLahcOi1l/8OKOtLjXDCjpaqgKBVS1vBbXd7aMbfGOwa0cH0EOoRqPenZb2VDQaQu3nRbkrZO0Z+xO9vrlfDnfGtFYycIqrHiO5boN41mIS938nLXeK8wtkP1Q1vkwHkMJtDzDvr2ISOa+2AS/0X+0Xn58BLtWsW42FCLIZW8I9e/6XE7mr8T42M7BBX6eWdCaE2hcPk5B3f8Fa1rVwpJo8seuKZXwD3Rs/jaPDrlVNhW9Ahl3O2OeQZ6M5jmTNLvMGAb8jJHshNWWUVq4X4uivz4qlOrqgUnPQ0/NAob7zdoLolpvbugqbcHTnQyPek8TFHnDKt4H9fiEXH3yVAZuDXIC6EuDpuVylgZDO1xeBTQKIfePF9XavPr2DQ9TDjnHwf/t9GRIDYb79zv0Zhp+TTJxefYj8/WNaBeNRI/FdbOi4Vo3smc7OggV7T+Q9u9H21pxO+Q+3DM+9DWknN7IoVz/ET+WnBNxphhEKAHQUFnUajOYK22sTOkCjsDG2cDlbt0NdPjMkjADje3y6jxEEC5oYskHt4/O0AxrgeME1RE6faRcrwp0gGLjLLd3jS8VkmqKQ0Vu2N0DAaHsmWv2eYxVO8CN03zc4vzQ+LRt7rRwKoDsZnWEjuw0zCbISIGWpbsK5M185YXg3LPi3aDrEMH0Zm8eN7hv88//PcFLwAet3M8Jh663Cj+ODSOM05oKO2p6R8B3An2zxyEDgPtALMKrzKi3mcUfew6D9apIlrQJ6fC2YTZ8LSF4FwaCm6J1grJguj3Ht6Avr7KqT3Izm98u3dCAb1t8q9g9PXoMkyEioLEX+HnJfjeRNEgRyzQK1sKo7mT7KJVeL2TMWmbY92JPOR8rnLoNXBjyqmGL5OZsNdfOT4lCsQnr2F9GjQukPpWigebCRuxvF1nuFc3OnMcMi9yv+zjl0tRi4pERiWmR4FiiJdBB4z2Jsx5xJwW+j/AhYdYdxrjICzHx1y+CWtCe5gawICHQ0fYdhHnzKVtYmag1cz+nwOmg2xm6+kexQdZ5V97FljvS4Xz3qVKhScgJOx8vrBBnqxbIVTEB6gVrGfB3jowFX6eixeqlQpKg97rgFrzMirc6nKB5oaOfl5X0s/mmgm/6vZ41B5sT9MDuDgVXOnduQf4KaYbp+inzuGHg5w54PVhmgFYi8U4dU0uTMJMCgPzBOqn0U8kh0BoIfNFQGfkBfkdQd/ixPy8tRf4RsbgTK46B2j35OhwxLONDYnoomdASipRR//bJpV0AryFC4ydbBwWLC/ih/mNotNOVJRyNNI2tMbMXLz4s5abRoFY4lcawrFP59vIL3NgbAc9SIW+o1F3s/HcdDxhGw/NwO/ZT36SG6mCpnjGsqrAShPv5IRdCoWhY5ZZCoFzf34GmMyOkHI7yhKe5aRC91I29x6F3zn8UJGfXEXJ0CpyzqdU+CyR9u4uvcVUv8A9WEiXubxLS3t+gB3TSRz3ixY5+52MJYtmtubQHqInMbyIY/jScnqCFZdjN37LLopvPrOoN6WxffuaQrXcsXY7FCuEm+OH9IuGl8qTUeAn2JVrXoQg2XlDPNueF0Zpn0H7ZDKHvPC7zHcg1kCqoAbiQWEnELEIB+3k8y8Gvvyndqf97nX6ykJwHmKqmBXq6mCpzUM5oEETfV7iWCtEtGCF7jAOxuGitwy8HlXrjzlZrYezlZCoURsSpPLSUbioQUyKPVsa10Vu2dTEIrsaYoFTNrgoi0tM3DSE3w5TDbpPCGqnJNOhsV7+kulMz6I7vVcunnQnjEX9c/TDROzwh4kaPUEUz2KTuaaV7c78WiIjy82+fvMC/BZxDUkTCuLim252vk/61fHiE9FkThrMbxBfRPUpv2vCfDd+apFqzOsM3nuGuOEC+MLIdEp2Boc/NLALJPO7gG9s7WF0udzpJOXpJIW8EpwuQs4u0s8necaEAXgVBO26gPfryMIDATSKdSdQXxl5PCL8AlmbxcLTV4a3Ee2nYlRR50YMfBhUA6TEqsVSo/OKeOqLLhTlNFpXM5olXJa0IV10OpRmobf+wTEWmSMvq7/0svqhiJM4h//Ux+3iZ5bMUQcGnEefz3N/YvoVH3UanNtB34n4XHHMHRoasRr1ob7JaGiq769di/CTrggrc35Qvo/tbU6WvVKHZ4HxQ4FfcDqsSxwzEf3YpL8i/r5xCp2cDu/Z0D7DDzAWoLqU86WIcS2jWYXz3NMw8fD+eYf3rwMcKWeDhZ/EsvldPa9ESZTzaeEk7gNyxAtUMud2MJPkPg8JsbZJ41AIy27GI+dBSurN+luQTDIidiCVFJ0Tg+X8l2/HDJhDFH4elxLnF3SBuNFsZhygOuThS+uW3gY4GoqNHmLETMVDWiCbUvZ0FRReDrNRtK2t7U7ALimjC2DHnvS54V6A+gaFfuHdbihf2c3+fB+uBJ4XOuEoyyV7fHEg4U2v/TivuBQ+LSkP1BDx2mHCcYoX55lhtB5V4T6uX/iPiDtMeX1cXX2zV9wMZyTxQ8f/4CZVJ6FE/01NQLgqOvFSdFPh5oIsjFN+Yd04/WGhmyocp/RqtdnfAOF6pQekvirMODcowD+Z90ZoD7qvsP29oFQYn0545fw9H3Wyrg6Upv7kDtX4Cqpm8W3jwHvCwMWXYKzXn3Sqz5dhZ1Y0kVwK2uRQkN8JaDlYkxh/I0P2K6RTdwFaAZYUYn5IylEChQz6LtZ0FiWNJe1xXtGdXiZOZy5172T91doEbax3O4edUgyop6Ijhnmn2DnA9urr8PASwODGoIirdkO0pDR2AI+hs471KSBru5LFrBzMKfthp2ghjOUi76iRgtfpU+8J0OMjfhCSJjgNPraiXpz98A/t/eJ+nI7w6yXF4QX701locyh5pW5zXNrY2J7eXI0wtEPDJHbqSZlVDJ+EeztBfJGKJWxJ7Tn7cHjaUJtvp0vdCblYBqfMrO3VixPDzhTFcUaQHAF6NpAK+F9+4352R6JxLd/jFBV0SiQOeMpJibjFRdkO2dmoXs/+aeUDRqrrltQvbKhbPG/unBX1C9PlCKnu6gPYhd/1pheW49swsbHTqZE7BzuGBQ7BwHhp3yiYv+e1CeLa2pjscYcfWThnzpIlqyEWx7ypMWc14L5Vxp05Wg6GvTf4gdVg75e7xx/e2zDn0vpVkEsvuLjqPOh923I4Dw6sufZ0FNw/YKfwhk8snTlPNIWrdCz0G0w83Sl917KucCpcfWFHuegquDqk1KTLLq9binHfHOz0AvCuPBdK9i3tBplBdIRMAvdKuNozeUT9ymU/QUB/DegHoFeCnk8iY7cWKW8l1GtLTbhixlcUj+P3PWLmlaAbMNwPZt16CpwhG49/L0QwbfPsK+bXr5zziwznxl4x/yrQizBcAX89v7Q3ZqrcRbjTd6xR9Q1z5i87iXH+emWOtQr0UnTcxeitgnrL5fHL5l1ixHnFNSl7JQTeleD+AjvtkC2p/4r6JfMpwbEzYuHVkLI+YK8PwGsN9eGsdlfD/rBnDV+5bPG8upWnMODG0lRkFQQVhSWrlq2Cg6vgeCTsvnhv/Yv3fpDKlDbKCGsbYkr0910VLrj1KnQbYaeKmFNnzq+bN1PqGcQXYkzKlzI/Yg9fsS2agtm+/3C1M+7w/lmH98+YeTXk+QWfsXpX6WPs9L//s36rwLtaEnzUnCAGYUndH/KSUCoW/QP0N0Z42PW7H7sK6IdYIWUo2maOr1u6bPGvEOcBGMOrbu0yLTzwrbfeKplVdIBcsQHmpk3pfbq6a8ZtGN2uXigJQHu2tk1Ok8pP8fW5XnRbpEv0nNZQJhdHhEpV6CRyF2DL9l7NzbPOBQ5110PXgrkb+OO0brkcwqkjb4D+aCvFsT1wn5yj0WJ+HUKNPfOltfBLxR5k+iyLsqTmtfP1sa2gWoGr5mjvPJcP+J4YjyuRvghTzwpfTzTMFYYP/TRfNJY+6ZvUj+WI1cbXS3BgdHo4U62Pb12mfmzUJ7a23aXYIPd67UkR7w3wmRz36E3hB6VHr3dLAVXS7M1Y0RW/IN0RXFw5QHUEZ5JhjgM8MlTLFfqjbbb6Haj9XF1xDT+qG6dcGrEU3mDLcM3HpkWaWc/Kc9RQoPsvjgwFHMJRJzPg263Fckp/t7UG90l6+kobN+Vy7thcujOf19qeJGHC0PT1MSrJkZRqDgI6aeJi4G25ehgQxk3qDvRHbEX80ASS/KX7b+vu5KinVMbtscoogRy3E/q5uckE9NvGZ7bjk5Jvrc4ZmtLFnALmvvX2z+upqSO/sTZZmgnruduUXSWBqAR/NLznwczbQDfkRh5Zi9VwKNr0UTzy2npqD+pw59PGXAgtX8IG4pGe+L+Qvk/q1LnSUo7xcMqPrgpq7mSzM73sZ3rO2+aadzFvdmi80HRbvjUG6MKr1BhRp66Y5+BPoKa49NkKXYYkxQycE+B1WWfjenqlPvLLEj8zSA/dtlJ9hrT+3Mi7DpeREZ5In4C+zLHowRpOg1XQhBniScx/xown2TljG/3HoPu7jEFRMfmyaGEXbF+heNz/7Mxgdhj/IPzCNC8X5hxG7w6wJytduW2u4yiqQvdMpkqL871E3hHQDzSFyDAk6DnDt0XmIw3fZszF/A5LFL8QO8vJROaiHrnNty1Ft0VFdkjZSE+017ZyqVN+bV6GDJv08G1nqbMMqtusxzNV6ezQHzaIgbO6K/yV1a3sq0dBj9rG5nam667j8eB2uM3Hb26k/yL75Xx6na6zMc+YCoHVCMZapSqVM1PRrCsjNzpO6a6SJvJCEnXsfDWwcwZc2CQAp+KFvABMY8Nw3HT5NgsbMccJrzMuTeaot8l4OsjS2jn4S0norSuwSwFOBT6zfJCf5V0y421tbQ+ob4haONLahJduimXiHYCHs/OLYf6D6XiTLmSDswH9eYMeKwhcLJl4bm5hp0z7YAPe50nw9e2effRdjiHBXu/b1rkCAKdfZc5i/mCbJ/u9oVyswB6U8V4A99g2mgD0VpoKHCXH91/WBYuPM5/9KGYsdZwNkfYT2z5Nk3koHsQc0o3xftxIr9Zwb9d9jo0Gnn4OHXH1pU4m//Ntdhy3gP5CZtD2ivrM8P2HfK3Ezi0L2OYgVgbeOeuoLZyNZydkztMrmMStCnjxvJD7qK5FkoGemGe8WGp8fkagbMi1roozycd5oJPN8tCXVH/W+c1FKJn9kyYeCcwlvMoN8By2P2nSZ8s15amSs5vjxiqKQKYL43PMyzpQQ8p3N3e0b+3AVzn6nOZF5kbQ7Zrb7rY3bApoxGb64kE/HCvF6wlLknxbacp4QkeHDU1X6ILmN81MFx/Pb+TWzrSIfWdpD1EB988u7VTa8Z2tUZ/PS1EvwNaS/PBKh1c08/Ke3s+hXOOfzfrqYZDNbfBxXwlNyecTZXzTmFwX4jFMAX6czz1AH2zervId/VDzDq54uNk+K/fIls3exO10XSGvA/3oKMjQaJrThXcz/Zu571m27F9In3J7Ayer6Oo0XrU5ThuG6q+afVxTZq6A8V3aQkwODnToYEe8Xy47l2p76tc3O3JNG4LKBP9U3Tfpr5u/wFkX87K073sF7fwLuvLgc/R/m4tVHehvh0C2dLDzGGurOay/ax5nDtwYFJaeTXSjdf5mpO6x0GzGLbP1wxDmWUu86uVYeTalQJ1TkcnjPWfrl9nRTzc7eICt+27mbsCpC7hvkX1lqV2k/NjbAyM7r9ydzATfy3L9J03W/wx9svkbPJ95npMKOjLQTZ24M+sfmic7WccdyzRruFu/yWjnWKt2cn4Lz6vznrNmdM/k3dqJzhtDX/Xl8RW4p0riqe7Q0taR1ne2O47lDi1G2dgJRdIMH/VrGi+ku8/ohaeDp5H/cnS6pa3txRhO+IW+ICfdYtZtDHLDY2j/deaKzTtyMkF4/XVelaL/fuF2u1F/c4HLbzj6x2bXEQd7KbgPKPphe6I8UbhvQV6DvlHCVruhd+QUjeObEhWJFc6SrhkHT212D4fxhIgZ6F+aV3bA0i7EvahNZ/RoVy12dbSlRwfyhHvnS6PfFj6wSf15beCMxmvbZSZqv6Vtsz2etW7p6aM33Nxs8OAW/mm+OmYZQgiu3RLkTSDvXqZRzDBLD2+x1SClR7S0PeU2M5297cyISO8PN2PhxeoZ1qNabFF2PbrFNTuCHiPf5qQzIPfsUEE4vI3+DuYy7yjXjpVTVaP8jK+XK+RnlT6rpa0Mh3Hc2jinN0DefNR/Zz9IdruxvFKvanGrrEz/rRF65Qv32ZX2Zhs90Ktb2gapyUbCvHVL6Mpr9ebZDi80dKO8soQDezrQwU3W04xNlzpPs5/7rItOi76mpWBK1ya3HmjqjWY9HOOaF/5p7rJmSMDtuNjbZeWpl0BfJ4/4FvT1La5agnpti/uMfyjyMog5N/spPB3SdvENLfOMT6PSg99e6dGgXwrp3OcXexlzDIR/Xmm+NJEujNFtzT49GuJpsR/CRf/0Pgvv0je2zFEtHORl4mUSEddtA1s65RvgXhLBg0e7nSrWG1raRhZGp5P6s0WOXafxan1zi9nHYPvJBt3U4vYxPA4rvVmKm2a9pQH9rRYcCUIz59JFylxk4JuJXNjULOMWTLe0FHGs9TVo0gfA81wshm1dM+auRTR9Vip0C6uEPxaSNNl1buGSeF146NX6E7NbEjrHYXp400++n3d7lT7JaxKwpCzTNZFTAu5INi8DfVtL21a8YSPGlgbG9i36spaEfpVDfnI6hdOQJl66JSGVDR126PmbrWEbl2Km8JfVfMVZ1nfFZtcWfXuL5eNXb5ljHTGGnuzPhcmixgZ/4+h/t9g/FpyCIy2u1Qn0m7IMbazfOgWHvbGoq9aS2sH6nQdwuATJb9bZje8JtR6ZhYUf9c2kw8zUymtXeVPCubzXMe1W9zh3QLMU7G9uF2HQ54FvDmp1z4Nwf0X3trif3vxd+0ysNMVbZ9NAFeR/PVsPVDE//1HlTgVcDTzFDeyfx10dftMSIDe0LlXvs17UOheLFremKDNcL2k9hqvW5ehKg3e08uItXqWRa27ZmMgZTe4al/rl8aKJ+u7W3jw1hB9FEskrIPRIgKGuxrUh7AL6ntY2193B2BPVCxHznta5OZny6ch/OHKsy0CcgUtvbHM3My0B/qqFc1xjC+t7W/PMURDYj281RoFu2sH8ba03x7GrgSvP7hib9lFEl6ecvpzhZa1tpWhttCpaeXert56tD0I9Ug2eppfWz6Ci/pn+ug/o/UjOva3mXg4SQ4m6a72s9Yawp+h3iHgKO4NltvKdrey28ieKJ6D38EZ9X2t/d1mJeV+ru0/p5a3tzUwsge9wAMej9G2r+xt7k1mvaJ3aBWOLFO9p9WS/lLO5RyBoVw7er1vwdk/9DPx530ir7ffclKcJHP7xUe9XG/flhoob64AedKzf7Fvp4AydaDc9Ykp2P/IBRPcuhkzkpFkbdQnUy6zvb+2TdLq1BvS8L2eMfa1r/up/6Hbp3QpsBp9injHZzgXfcCHBsinZTNks2WzZHNlc2UKyebKFZYvIpqVxRl3I5NE8DOeCcQeaO4p139v6BrnLyXx9vS0v63+btqhtg352e7fJHfpnnGKTFhmi/92h51M9afxaia/Gy9t1JA0ZOGdH16kov50Mk2+9fD1g5eQd2imAA0fv6JH43KaAMh1yofLEHdqUj9m7Ok/9bsAp3KOIem2ln9IWgX/rfyN9oeLn3V2Wwq5fXaLhn/B5G62n0+W7td0OKq/d03XSjZDJ2dG9Pc7sfokNCF5cmtuGPdqjX/rKy98L6+GRZTq4u9vUzpAJ5YTpYlOuUmHClKnDdF4uzEffXI4KD12m9fhoG+g993Se3AMyheOLTB6wKOKQ0fNuncO/2FtIPpbpXvd1b2d0t/247ukR4ID7dGgjZOP/2NutnRofrWBwf3rO8ch4aa8O0TB7qaUX5LT10Gc/0PMb2F1eaiYLLbJqH0hB0Rw1DGp76A/+1vW/8NZDFlFbqx4XAXrJUVg5c1+PyZPxSNUID2s8LH0pwPbdl+Pw+n06Wobl+7tOHQINtyTZbx+mS7Doz/1DIDNvrb72YF+e3ZX628ak76WCxrqDPX24qajtYu13YGM7uruL3pW6NEBc0hygCR1XBfk/9KSH7yomsv7lu+s6jCITjF87mtd1wk5/93t0BPrir76GCd+WB1ZXoMu3l7I6VD5ZA5sQug+vq8DGjnXDB2UuUvReIf3Zoa8yYfI9VD+de0KJqe7I9gMIpWGZARpME8Z+zX3H+dYlUGrRNWc+YoPgUgvjwO45iUdMpbHJYru0vsVP/Al9bROe28Hz7qRVD5Y6+tU7MQxmyR2qAupWDs7YDnUEX8JEX8eEZ+/2895I0QycjvsHHck/f7UZBvoM5NLrroBMgStD3TWlQz50HOqHajdcH8qVd4ZMyIzmM4f7HmKpRz0+22yDtxEyH3lm/SYuviuwm/sNRwi334a+/TBMDJsQiT/Un99p9SMToThiOKYvItk3YsID5UH8L/QlNhdrGRWlqYM2/zOAvnmTbPCjcjpKT+b4yfEwPqqhbScFzoBpfUygfRTk/+1a/Rh5QeHmLnT31rjxIvGYct8YgR2Nni/Kz5fdTbe90xHgKAXmcfCDftCRzXcowH7gw9vQFeBb4sEnZ2hQCTDfo+EKTHqL6DcaJeW2niE+o9oXgx8+SvTog6WWtffLMVLw0Ps0aV8qs9A+Srxzmm9vh2KbuvzDjy/EUXYCnKNk/UD8Jvm5vSD8MvEh4n8+6LsdodSlIw/KwPke4oun+qElYO8lHlbl530Bce8J4vAw36uAWs+EMC5C+hfw+2f0ki/PIl2MgfMi9VIQriG6g0ZNxGz0GfL92UAvkiUjmbXdz58JHSPuO8TfIx2mjseFijMH+AX9oKeO/4eAF5AULnqK+NrBfrQflETjp6hJdOb//kPTAP8Pim03vA==", "saves/ankiian_t3.state": "eNodegd8FNX2/znnnjttZ/fu7Oxms0lgE1KoQiJFVIRQhYiYUFSsRAFFSqgWBASxokhoIUEUuw/1Cdi7EdRnefoQy7OLXd+zTMD6FPM/v//Hz0xmd2fm3nvu+bYPrszoaH77hARC7aUAwKAINiblCuKnHFWfhI64XKL8B9pKDp83e9asxnmlJzYtmbe4cda8rl0hSHWDjjSBkkc1WGCDAy54EAMf4pAAA0kI5HHTFwJlQUzeruWNSUhBzIcHArm250F9V5gOb7oQ5ZQ8R/JlJ8JEwH6AxWAILC6FGg1+H1DHwvBOQDoFIvspIAd8gpACKFZDII05oM0ZWBlvBMrK82/CwaNtwHYolGOkrEOXAMsjc8CpgNDqA6YCXByIOBrNHLADpxs0yELBZtRptOdA6FbImyvkzVeeBBF5NphKiOVjlaCdMZj3HTiomiCMV8JMlKV3dj4mt/79CIhKEzJuE7mVwDuwv6xnDSXqgLrKzAEthU9jkVlClVJwDIJucOZfEPU245Afh3yqApL5UE5hugLyWuYaZirANMu9stQCF45W4DwI7lsUZiuhWFfCqB1Y+xPmCwfCodJcBfhzwRkL+aIKefkM1PMgLK6CYxCcP7BQjYewpApOIdmmv8pI9caOPl1sSJSDGc9JOpKGyDDXMBUD58GUgx/QTZSo4UK8mvHfgKep2D9UgNdpWUeI467GW7gQjGI3pyv+jtEEK0SsA72RQ6sC6AqQM/YmXQV5uwpSOacKrCeV8wnnpbDa/CSL9IbA0Qz2/UrfDbyGaSl6v7G7FEO/Dvy32FwFQT4+RJbyT7Kw45hEJfj3gtlgBaEZAv5HzGcAjrHpSzAsHXg1hMHxQOO5vwWpvxBdlLHp3zbXgUwA9xAf4DAcDzQ/Kfs5jLgKcukq6VdiKXbmaEzOnyVdmCmoBK+dpevUM0xfY7wcdD2HhfLqCujvQY7QmcD53CUYodSZWtkaCwE+p7iPXoIdiuaB9xfIK8MS+bUO3IoXMPK7lEP8KeYyIIKAOu1A1zJ/YOP7yJOw/nswZdwTIj9Psht5JaeQy4FGsH0D4AjiQ074BkS9TtH2+Yx3skydb2MaDxVaJ8G0KlqP9ri5ixYvbFq8T1qdy7QgzDkBeCSkWrKgd2NU3h9CuNLL0LXavgncvRA6e8FMFHwvB/6EA2pwvNt5oPTrc07GGwvebTxYHQxj0lS3MnZXug7yfhVo3KK4xs3HZU/DRBWYlrvpU+kW08Y+3UjmCNe+AaORFWCfyPidxT2Zd7JaxGlLJhojtcA6rfGcpnOa7gH8UDrlW8uuAx4BwfQCsKSvVH/gb9lqB2sbZKFCdkguO0gwW6jK6JmzqLQd+2FHtWwSm8+BhFTCrDTDPEZmlzT9SzhDx+gwYQVkvCptfBsbpcizFs26HnAGFHjnIQ8u5ajbeRiSjDZR4xnKTNQJSxp0NDrtOixx0PTTTl42zdCB2LEAaqKmojfVSqq0BS84bwBFxTQWhRHA6o44n8pAPatjv7o8BvAmRtk7R1v/A5wLei5LAyTNZG1oE5iP2A3t8SCVdOZDRtaVeJx5j76SI998JAT0KGe8ClggFCJcGqvRNAf4YW7GarmYqSKbUDY/lMeNtFYYr4B4QwKfOUTz87IZjC1zKZlpxQ6n1fVoDppOjgkOzbmswtRsoLFjMeoyjFEh/51pHMWqwB2nw7Sgogrs8eTKuzOCmoUg1aALABdgx4ByYN0M3OrSx5DztjnOQypTOE52nmOBKqTjHWHtxS5WO1zvhkV1YA7IGoulbxa73jNuXjjHDdR8Ae9c1u/XYkdNwE9ATt/lmz9iAZ2cMLXSOtMcgbc5ygpoEgbwfOjMDPVk5hVMjRC/lXUPCPRnYL5Riv0lfMCMBTdjV4A3mXEKKd0u9ELfKPdm0hVwMc93+CsVehW3q4jMyay5mVUVpF3pwUasVR1q0pK505vmNg5B9KjA/UHzsXkqy9jradprTP82Y37SK28Cp9AL4899SpH2XOQvIUwIop8FGda8YeP1BHdwRyoI9UzkFQkqJfMVx0RW4vnUXNGacW4oVO4/M00F6f/ojW5GqDzeynwuVuMSvZabhcLNxYkY/fmUNdLO58ZeyAs+ZvtltOumynb30C9hR8luxNtd/pjNBiATUjLscg6YWk0OAa1z4sPdAtyfkTEVTivRUWynl/iYPSnFr5ZRhrXs662c2JT1xhdRRyyR1vaHEFg73YRs31s6CG2BTpjA23ZzZDkynYdUWejKd100xXwu4PBEiBLHsHUW4wMijMBDQzzHxhVZYVT3CDf0x19hdVg2/HUP0ySbF0JGeKHE7Mjo+cNxoj/ZxboTeIFs/QGF//hM1SQr4Qa965IEvYgeM86R3ow4NRZKRMqwn8uXFIZhFfgbv7ZL8+lXdNTzURDeyUhLevPBqRMNrIBjGLxJjj1WgC8FHSvkX5zMFw4BzRmkXUxDMP4WL8L2ojEQEwBfLpJafJlV2gcLYH1JJeTcMzy38gmCpVCdxuNr3Sikt5TZKc1sdbgNWoAXvwTRyeKZAb/NIc1OUd1xqiNwruZQ/EC8DvCjhIhKToBov6JdgbEITXxbqJotwf1GsMbBj9DckaIPwVzowgpoEE32KrNw4McUbpd1sjUb8ICaiW+MKKGrw7ztshXG5n7uRJT3Rf0zAmuzhGPUDN5+J1IDuvDQEvoAzA9xj8Zg/Em1CKTx1oFok7nEToapIWDWsLGuhP4JSH+C+DbrmRzn4Jq9UDaTu9sdWU8mXCAMdwiljHmpHLs7VL5Q3AxTmi9i62Nw50Ed1spUL1qLcAE0F1fC0cJCO+hFOFDh0gN+4H6n+OQsdrmbq8MuYy5xIu2EcBn6yxWttfk7lUMZYdCdFKVD4RM1mNNxQdw1vnou9q+VTS9tfGn3ihdun3ENoC3WawyU8bAHVOn1/vdWBHxHih53vMPs/JtD3n6/arRq0HsB6uzkC29BN/PCMVa7JZ15DtjdIbTL74YO0O+hXs78CuOT2tzNh2iXK8bhcnavc/LeBto9GtSqksBelI7/8LCK0B8DJiMli93BzmeleQHzg+eDGuLSDSB69xWUJse0uAe8kejMBr6v3K8EGnmKhn9RO01n6z+I1yq9ht8gaZh3WOqt+HkWga9IJGSRA4gWoVU3q2neBZsLTk4s8YY/DnzMz9CR+4mytFVodSvQTNfaCrgjptulkwWl28B8BTqfFeY2ew5Cab7wAwrGfkDTcuJBvna24K4ZWXpeeSI5t6g34SdVPFas061WMiyRUhiNNfi47qheqt1TFN+p4pVsi/3ClxXF2QzQLv2twHuc8VILU1o03siQeNDmu3RGoFrE8z0UlFf9TPu88Z57HuYtUVVaxwjLsL//CvfCZqcchhDoce7tEKEr7baCrYzMx4wBK3+r6shnfPkwVfpXCkuLEnZvZd2i6HkOnH5aquqWUihKdTSB9zu4lWM5CoMwEEJ4iWWv8ikZ0PnCyQlR2n0ZP+vGH8fz6U+c5+qNoCsvuGdn/rF8OQ3W1onoVBzLkd7G1vo0nc30J8Wy7F/s0qZkilbWKX6h2IyDnlCdKd7wMzbQOsVblseq/Qruq6JEFwEWX56nw8qrQN6pQnzUF7DhWVqX8Rf0nP+2kviSEYR7+a4W3OQ1y6WTFzmxY9jBAu3YdHYewtCW2Q9n9wO6QdcfVk7eHeJF5VISbf5rkpnYEPA6FH8Hysvi8Sq2LO6+IilhrGSJt11zrh2ECQHrR3HnSbejIL6IncpEKKwYn8Asfb+IKdvFm+aaHWJ9U42y+l/UI1b6nvN0aTJ80Gp4Ddz7eCatqnK96Yr3cRVG8ItydlIoSPaHOjQQP8EolxP3fhXj6TaOZLH0gftLsMgVYiMoXpQUecgVjwdpBl2NZfTE3bEFOYd3cdhltfOGJz5KuOH3pPmfgoGxyN4Bwxh4YUKHYk3jgqDtbjKv3tQHVgJt58D7IuVtZ4zXCWHr16jmCkh+pxo+B7fcG44H4yJcZ2WD0JFu/Zn5aVD3IC2uNJFjhd4G1nPpU2+ZG3+U6eVe3oeM957hdviJsRD7wXFO5ozYK3O8tjJJqf1YcFa6s2nEL4qu+0Qly29xVnr5UNoHRwwR4KS/iJ2TkU70nK/jHYG04u3KXq7MOkrmxR6KEIr3dqlvLvaQk8LSy9wNObGdzoNOruiMWLffMmfEOopoH1QQJaEGUGoQQtALV4D46pOWzJg7Z+9DhYCPQprGgZXyI21+SCD3FY9/k9gkFEONP+hkvLmLSMJcVMthk9sYuxNKuGKTn8R7Adr8DqsXE76q+fE4FVeY3ZJYxGzxIxjE3vcOxttdRHnyh5B2DhCto+0oJhL3DjCbAD3THAonmXJ0wvuofjULvZ2Fq0TdvNWMV01BeFdVG0GInu+NAfdP8e9e7SrB5NbEyjPYDbR0WD3e6mSkMN7LTG+UYAemxNa5TjqUGtJP5L4+OBG56XI4hsC9IOEs02FGOkoA/K5Q1blOreP+YU00qwQ+1gbA93/hZOEXpiN5khsKV/mPsL+e6R6K9XLdj52wWPLux2CK+A4aHnBPHJ/oyHV1aplKOH5x1hXf/kqfnGg0S5w/nSU9JEIa7JtzFehC5UQOFXKADx2F/6qReML7VChkQD+pcVD6a3AJJ3W1gdV+h30AfXnn+iO9Wu3OU6EjpuYR5fi621j0dfWfyhqCzbE/lTPQCf3KGudIJ4zPJexvzilj/BlpezxSxcpPmKHxIExKi54t+Oc7ab05UwzpbRh/sR9GHI6BEtVYZFocSeUC3bPBkVuGEcQrlA4LVulbnTArDSfVvGEAPFcoKDmezfsmGeb6Qy3o79Xd3nxzl1j7CrjTqzVXaOpLYcn9fgcKJUnSdTKa/07mKOVMp9I8/c/rCELLXcW4gE8Ghs7O09kxwg11oO7kNIrWdLD96fhZ82bOWd4Db2W+VFtDE5H2jZfVIjT6QqArWYTGa4dxON8dTvoLCG+K1dejk3cWEIzGmnvYWkG3hNJB/t9YvO0IXFCsRpaYBZh1bpHIHf+8wOnnhJdTTUx7W73awgSu8my4kiLbv4/dSsgIh8f2Os4enREO9xoarbIw7HCa4+/H3GEYpit/hZ/Yy0gPeZuA3wPVznTvAG9DzpmUHQN+c99UB2WkVuZewbfQlfcOYG03WF8kX+2UkhQ7UG3lS2T/2xITORpq6kfgfG0e1sk8fWjvW+Hot1VOLL3Js55Bt/ijmZ5mdKB8EzZLBpRQiR2Z3xR47rRZbNMjg37nDivjHA9FwVG2yWEyIzbXOxswd10s6RWrduBMTCb7T3bnqBo86Ph1zBeA6knUAeZG10HMm/pi+IoXhmY2+GP4XYAuutbsUtYjaoRBZe1Wtb9DPrzRLk1XFrvN1hp23mJB0RN2g/UDuk0bgmnmUVHN7EFIFmpI5o6G2lrKF52kq69E+rsDhZAsUbjra+GsLmJnaQVsM7V/Hm28rgjr3SgxHHAjCVgsdxabOYDOSM7zbLD1WMjrK8NaMdPJ0YmG2AjG/Q/BOSLdXh7vwvKMu9KRDFlvhPEyF6n6LYGVj2cAJoU1CwPrQlNTztYhSM05BOd6DykajDu41oRh+Y1Y7YXpxfTpKDbfuEGYke5vBO9mbvgnuAcVJmrTaIW/QvWtjvM/+Mwuw9ggpDOYTlDxCxpVauzvseqSCtgVJvVsB191Nhaw9MKwAabPOSqZeVE3nKSsUH3j1Qc1Fq3eomFuaqW5Hhx6oM50QNJ63UvZF1C30FlK8KDe5+8E92MxHyIKs2EZw2OpwHpQV/2lFEJGJMDEIZWMQzI8hNXVynHtc8TwucdhzD6yL9EdelrcYSdyf8NaMxe8fwdJ/XgcOqG6CbxxTvNQDfG7q5hgq44SJUriyPVFvF2FRcI8dcLRJ+QVnKC/VflijckTvkjA3CSMcSOdExB7X8Z4A/DwMpNTTojlNbEyXL/UTtGJFqRxf6BemmCvA7M3TAqR6ci/QkAqpLlDTC+o+UVUOtJrV64EGnvsRZkKHLHTW1miOtn8U6KAmUYvxi0Hh8AqWbl/A9NlJ1JprHyWJFuPfoSY9MLjnkSGsyE25PIwiSc5cHFiN3t0uWQOGz+yZvDKsUBZZaXw1lgEecgHZzoVYapWzKfypvMUfwrjiganNJ8pUskfD1hwcjIp6r3Ab2AzwQ0OxaKk/jDDf9TQRPT+zc4IDovGAf3B1i5G29fX83NpmOtHFIsJOd0LtAD1pPMbz1tyxdfQWBBryPDRc/zIa8iE2B940bGWROgbAecUpQRxeM+EJj38NMLlS1VktQzOBQe0yZcHIcLxpjc7Yjsm40EI8MgaPtglFMjHu2j7a6/c1I12pm0UWr2K3gyqfVvPkXb5w4Pp2OzPBrc9HbqV72QjjP/IVDs2z6cn7UxMQtDi+C3rQK0tJEmO6RHOqjHgvctuM4xwdr8vVZsN3vuM17ym1ifll3IxsHivW3qYPzTrhej8m9l9GLbTCFoLOBVrbWGba+P1j07w/FRrLIoVjIWjNVgDFOWYa6eE4uxLZMqYcTeFycJCd6U+hnOD6D9dA4hUTlprBluSd3cy/VwbS/N/k6VpnuTgULespOrGglXTGJ8CHg8/eSvLtNUC5OD3mZ/of3XxxxWvy/AVFfTrkFwwqd5tl+X4lYwn9dAfqQZ8QHxK4FyVV79P1qeKyav7OqTLbPhfIjgw15m2OMFPcvg//emnivR4yEiYN08rq9TWdT5+xdNyPI7dCd9rGMANic8SbqXK1Zse4joeIviBR+RizTDbkTQCr9iB/kbXvBK3Ik7mI4YTELqmYH9BBMczvTtxkSXyZUPBMfClmVYwHmgc3wzz53Thwyqfjbm/JQolL65VzuOcyUnk/cWJUeFosgZ665nfVrWC1PngnhP/lBPC72tG54LX+2B33cvfPQPpXWU9CeY69z37MxoCNJglHeJ+1C8NxpVXg51mnK5bsFTiTbK6wnGaklEMaBVakxuXzlp0STPgLChzKnzgGXn6dXlRt9T0s7IHE5Z06J9FVDklJr39ZxFvg9DaBv4apjZiOxfyyEzSucu69ax4Uow/aDwtDXyt0xiXTPllcjYuURB7mc311zkrSwSzVg8IF2LD00K85v8rWvIkaB+PyXwgz1p3QqYJjyrR1kI8VBpa2F3x70xXkfc7O1czxzUVuRW2LaDahHQVWqfMWHz+RTNGQoE4n4nJ9lLlXAhhth38ufonai6sAMmZlsWbVTL3Uu6Nv5LJTJGI3CqNzzv9k/WVEilKNmWDUY8BXGt3+FyAaxNEQI+d5G1w3XshxCNj8UHa+dC9VCfrKpKHFPPyI6z5YJ+u8c0Et3GO4wl4gEq3wMh06YWJndSB8VrG2WdInAyli/wa7RXXLp6Wd561k65onAdWslIXl+Zj2h7e5vGm7tl6/2OgedcloFuswRfN3t8RCxON6eatOdp0svmMvfP8IzPBrP/aydQBgjqs+Y4l+k2jmcgXuoXZVf7LjF1Q/9iZ2HUqgwW0x8XAaVfJ7N9z+CoFH84ughfpluWMC5VeCPnczdlgzaNOsmg7psd1C2vMHndibFWJYPk5F286nGzsUgk51et2uPzIkPuhb/PqQjro1jh2ch9F3o2uNRT96UyzJpu3GLATeYkr2QnH2qV63k4uTfz0itug66RP+qkgNgpX3qPoAhLTemdxc38f3OQUetx9kJLuULvsfm7AfeLuc7Eq8OqR50FjGjZoHdkRxm53eRTQKJfeOM3UGuub1DQzTDAXfALB76MTYeo8vG2nT2OmFdFkD5/jIH2eqQf9isr8XNIwOxWj2T8UdIwOs8L17znePeiIEL9F3oMp/z3HSM7thxQvCDJFV4FnMaaUIkAfwuKewlA9wV7hYE/Il/QEVicCVXt0GdOjMknAbmu7RHoChFCtTKnEw3vOC1GM6y51gEqpomuiGtckumGpqrrVn4artaSayliZN8ak4NhYR9U/Hetj1G8DN08LsmVFMfHom7xkaDeC2Ex7kRM6FXAeQ0IMtGzZ1xYb5o0vhNW+n+wDHS7tRnfKwtl7n5qz96m5zwN+4hT4THzcharsg9h4jtzYcXR7/VEJwO3g/MJhbC/QTWCNxWUq6X9GyUeu8OFqXUpzBxTUuOuxIz5tHrjnxsJ1yQYBWZj83sdrMDDL3Ibd7P7O2/wDGuhNi38DNdCn8zETKw0zh+N7JPiuoWRYIBbo5Y0lyexkp3Q5XulGFm127duQh5zGY136J3gp7dbBV7ko7h+l3YAyxeKTV7E5BAbnSn1rxYPNgBuwuktPuNusdGe5ZJ3pfTUgqJailpYKjUpMTwKlEM+HbpjsT1jwkDUt9n8NFx9i36bGQ1yuP/Z4DdbHbmeaDwoejO1jx0OcdQFtFjMDrVbH/zlg2s1WRxPdpXk366LVJ4D9b6lw4dtUq/EAxASde0rmy5tNK8RKeRe1gv0MOJuOzsf3cNk8vVRDZdj/aqDWwkjHWz0uNjy/e1DYm8wzWSsTjN2WTjrHOtPMYC7Lh5f6t90O/ATTtSebJ07iB8OCWeAPYJoO2IBlOHVVFiZjlMfQOoDmSQwyuSEQm8d8JtDQwrCoO5h1bioovOr0QEWKo6w+CejWKcnhiCeq6zPJBU+DlFSijvmXQzrnhriOi9V2Vnullxfwg/x66SE3KUw5Gmkz2mNmLFz42ZY1o0As8cvz46lP5zjIL3GobgRzjI59R6PuZPXcOXjAUQ9Mx+85yH2UTYyF5nRk22PBriDezhmnEkpiH9tWJYTuPUURMFndIe91ly08wc3H7qaO7F0av3P5gdIgt5xyseXknkb5+AlC7Ud4tJ+paa63u4TO9/gWI/J8L7uWm/kkKF3g7nQjWzbNai2g24kex/gCTuGLF9JjrLka+/B+pzS94fjS/lSBXbvWl+gLXftWl1IlsDbdbl5Qfr5QZoEfYW+ufwHCXM/r0x1deV6S7ld0vyym3Y+/zXwzYj3ki+shHZb0ACGLeNhF/h5m4It/7nIoOKLRXFoC7gNMNTNjvV2sdPg4DumYSQEvcu2LhLTgItNtPIzHBfsVXom69aeCDmOGs52RqNEQk07lxaNwwXwxKc55IlxnelVTMwucOkiFbtWxpR24yML1Q/jNONWj95h07cm5itg4v2jROUzPoHfOkVn8wZs4Ds0vyfcyqb3vZerNRGE8my3m+lZ2evI/M5FsNwfmjdPxv4irSEQoTItvWut+nwvq0mUHkrmCCrC+RXwB9af8tgVzvPTBBXplYU/w31HihovhSxX1yPUEl99T2AtyRb0gUJv6ql4Xuz2kPD2kkJeC20vA2Uv0fLKvJg7GZRB26QX+byNLdoWwUqw7gf5aFfKI+PNkbxALT18r/wZ0nkhRTaOXUPgg6PmQF6uWyo8uLOWpL3hQWrDSvozRKueqnAMVpYdiFSzwNj+6aoE18vymc89vOg5xMhfwn+YTp+zpRbP0rsGn0hezvZ+ZfsOH3fnuNjC3IT5XlvKOi41YgaZ9YC4ZmxoEV12F8LOpiWtrTlh9Pzub3Q72K12eCerH4qD4UNyUu1Ym+YFFhxPB/eM1ugXd3nGga8T3MhajPpeLpIhpI7NZjrO9QzBp787Ze3deDThSvg3nfZTqKOrt++VaolxA8ybzAJArnqtzBdvAypG3BzJibXOqPYZVa3HfqZCXerP5L0gmGZHalc8Jz4nBcv/D2zACa4jGL9JS4qLiXpBWLVbkAjUiD1/cuHgr4GgoU33FiFmah2yBjrx2ztFhycVwHgq3dXbeBtgrr3oBdu9HXyjvdDTXaAxK7vRiRdppCeYEcCnw7NgBV9seORPKQglv5qoPCssq4dPy6lAPEa8dJxyveWGhFUf7YR0f4AUlzya8Ydof4Jm6tX5ZCwzN4Xtu8O4a3SihxNynJyIsS046F718vKW4A8broKRxvHmvxMuXjNdmhd4QXA/xJm0G578uidxrNOCfzDsSdDt6L7PzvXSpIL4i41fz9/yh2+GZUBs6irzjDL6MukV823jwH1O48GxMHfknHRzwVdydmczkFoOxOBYW9QC6EOzJjL+TkvNFotS9gC4CWwoxJyblKIcSBnMHGzqBcmpRV5xdepsfpen4xd5tbL6+KkM3NHk9424lhtRP0z5l3SZ2DrCr/iY+vBwwvDYs5bG3QrK8MrULP0b3ajYHgewbtWxm7bGcdx50S+fBOC71P1R5eI0+9R8DMyERhDERwWnwgZ300xzEf+walA3iigS/Vl4Wn7uzogM6Xcpdajpdj25Y2ZXeWIFwXLf5k9ltIm2NZfgo3t8N0wt0KuNIai+4H4dXKL1hG53rTcxiFRy0Ohy/SZwY9qQkjldhbgSY84B0yK8GK3eyNxLVar7LLS3ukcns8rWbF3JLC7O1Ox1Jcx0Hh3QAmKhrXNQ0b37jwtkXzLqoaV5FNUL+CP0u3ILf9afnL8Q3YdLKHgdHbj/WVTa4BEenKwcmwfpfYad0XGcnkzN+70PzZs1atGgFpNJYODXlrgC8f7m6rcDIxbB3jr13BTg75ekJe3fMn3Vu03LI0vMeLj8V+m+9EE6FXatWH0qC9wdsF9zwgcUzZgun8FiTiv0Okw71qLhjSW84GK87o7vctAwui2k9+fyLGxdjOrCOdY8E8C89BcrvX9wHomNoH1kE3qVwmW/xiKalS36GkA4PHgRgloKZQ0JjbaXaXwpNxtYTL5n+NaXT+H3flHUpmPkYHwQz2w6CO+SGT74XIFiOdeIlc5qWzvpVpnPtkalgGZgFGK+Bw3sW98dorLcAtweuPapp/qw5S37ANH+ztMBeDmYxut5C9JdDk+3xhCWzz1ZpvujyvLMUQv9S8H6F7U7MkdR/SdOiOZTh1NBUfAXk7XfZHwDwz/lN8Q7jrYCdcd8evnTJwtmNSw9iyCsr84nlENaUlC9fshx2L4dPEnHvhbubXrj73XxUuVJm2DA/pYV/39bx4rZl6K2E7TphTZ0xp3H2DKlnmJ6HKSlf3nqffXzZselk7Bj4rGfc8Xt3zty7c/qMy6AwKP6M9dvafMzuUfd8Nmg5+JdJgk9aE8UgLGr8QwaJ5VPJP8B8q+LDrrz1kWVAP6ZKKKJkpzWhcfGShb9BmgdjCpe19ZoWP3r//v3lM0ujLHniA6za9RX3m7s3pR75bdrZPQGAbt/UOaWCdFGer8z6yc1+r+RJg7xIefxeHxO7Cqwm4B0VZsbF8GvXN4Te/7aJ0thVrEXBi8nq2LX0Xy95+K9ENW8q86Gj68gkS0CePMfctwl0K3DBLPP+KR6vCPwTgLcH/FCc/qruYs4r5Q2YmmVFszet1zev5K/K43suwo3I/0aza1PnHdpS9P4VJiE1uijEtQXefWviy5luuM6rBHy70BrIiL3RU4nugN+YtI6x1Qf4774/v6SQpwIuG6ynAu21Up2P+ZZ0//7jjDxoPtzs6D5IpReY/pfzfWblqecmjtW4xvEhyj42LfEKm/MLXX0c0H1nJeS8OGRbm+83lelzwfywycGZ0JqMkldDYKYgzcfyzs5ySVHPG33SBNqSytJlRXyuMy8tPXvKBD79KnvGdfh2V27QdMEYWnmOWwUpqir4oyxypwGNJHsG0DkjzDAAHGfxDAjSq3+JVTufQreqKDxms+eO11QHXmK5KocCrzfSQwMyULCZu3fhVyX6lveMsm/2pd7onwzWLdrM23yiMwcwWY492DSDHZZMRn/CVqyDN5LNH6cT+66jrqDf6nlIXQCxpYu4O+IPiOSbD5n+1bPW1jH1fD5I3hBGg99hqxfFDrNp2uxZdzFHTBMEmZuL7DFA05bpMYCp3vikjT+DPs2jry4yVUjWeHGdB8Dve7Wjr6PXmhK/XxFEthm+uR9uwQK3WtHt59NpkOXTmceVRaY/WPXTzWAdmJcgTs+dkjjOc4qYbjuZX2YrPa5nlKAv6rkCvO+5F3YdrVl2eHioi1j9RM5o/TeIYjdq7+7NNAQp7D0GhQblw/vzemHFxbCNAieIRaXfezjethI6DLPgP62dn8Ds2NzXPVnkBWOrLPO67TqPe3Qpen9n+pvdHo+CwjfAZNbHSLxFBbj3bk7MRbp3s5qNRVWHNfcN29woPhvN3zcH/npNd1N8rtajMSz5DMzj230c+JcfKZuvHWHu3VyHfyrrAjAPMbWYKD11LXeydZnCZx2Bf0rvIXP/ZiUWnX64gidA7Pqt1oHyCCeAekLrxdpt03TzpYmrf83Ah+VRF1ocs56FGumWly6iqWSdfmmCtMoyejanPTr5QhbYjLvel/P2HbFoSC8sW6bp4XnkgLtMq0vBWaZNumWiPowm09LZ0Au7PcV02uDEU4z1mqazXsz0QDXn15vjWzr76V+JJmNiwHqcanqrqHs34DvZOZ3trJRrvVnJhCPWWSuZewE/4vHDltKchvcMtww0Ym/wDsnfgwtX9aj99Hp8/M5kVBk4D2kWiL0gtmt5S08/1NZG0YgWcpGaKukkV8v18bOsFTr0HwKakHZO02N61V6d54owH7F+Ds0JLWsrJH1mwtRBMCWZu3gVDWzggZ576AGuf+2k+HfuvG5R7qQWJ40bJT60dH5UQfFjc6ahxdEnM63YYIo1pPBThSe6ZBcrOKwi+51rE5VGl8l0j7iSzgHvYza/h+55pVFX92Ay8RPjubbuylS5nEewNZOxDFCDmdTyeCbx2npn6uow6IW06Opd0D5KfBCHKTH7T189LSydnb4/E1lnTTVLmfQUxtDGB2LemgRCamE86j2lRWT7cav3//3fYdQjGauQQJnCq1DEomne+Y0jwS5dz28yOzJYDCXeec0t1lagjHWCH5WFy2O8DfjWci/Qprmlc1EvLC9r4XSc91XquKTxEshi2Vqzp0VfIS3T44SaaMS+Uks26+FifUaXFIWnOxe18FKgj8Igvg3w+G7mPaL4wZYKwA1iST+J06O5ID6iAMeBvRdMe0vn83Yr48jB3MphkCLyVzbH60/V3p6WYxKRxAYZ5t4lqrmfLUA9wPoINS6MzKuk17lhuAy40xVT7shFGtRgbX5qyTX07Fm1YIJCAq57StWLsdbqZox4K8ivD2OnxyfWSDo1T18LAzHq+UtLmXBzWPI61jC4k5DfY3dF3PzaMsH5QVLgj3F1lMYWwM5uuDHnpDWlCj05b/1cRTnzc0vnJ9pn80tLZ98AB+eta3RY1KL4wgH6Gq0IsuJcTY8tM+0o93SApw6xHHH0V2yOOzqFrs7KR3PalpN0eSKkDzbEb7zRzEn0zqzEIYCbE0QPJaMYP5rgz7R6D3B9CZ/qmhu22AGuv8qqMdWq/lRP15iLneioGpNnQYD3lmv2bOns/FO/FDd7tyykO+K89xT6vZ+5aKNr1m85Cb9ktSeZgn3Hetu34TeJDHWHUHUHf6zG1fxmskOzfBFqIY/lGy2taU0dtcERXeGU0kV+C9AIwNuKun2zrNmIx9wsI3V3p3MQO9zFW5cSPbGrXG7f4sv5Z1rUd8GX0g89zUu+ePeh7iNMxx/hP8LWTse8vKUTIaLT8ajTefVxFL0fBM+cxNlisCL96hZHjwXzT3n5efql9ea1LZ5eUTTtxbFm32X0Mu7615Yv9evK7JPbtvVZuaVUimV/fO1IPOhPOYU/yzOB8/G1hls9FK3u6EvX9/FvuOH0ouq8Vf5U9rkVfejToiD4fGw8q68qPiAD6dbOzuH6GjZWa0+qNXar1z0TebkXyVt/rT91I605OjZ1I047OowFiu+cusiO2Gmla4ud+8Yqp1XVHp/v7rWm/1YchV7rjXobm5i88NMKsruw8Vs94WETb3XxQbYr+SYr6ncx83EOv5l3p4Kpbe2Twievw08H4WnAlw5TW4C3zaUeI2NVGl8Yxq4IwEcDlHyYvwhnOXjikS0U4dTWSe5WDsNXHSdbgTMqumLpaa3FPGIBm9MXQe0ZrY4VlEdea+cCul0HdKemF8bxma30/LhzsFK82Ynp+u/OpJePHq0OlYyy8IujaPYw3cmhHrOawoQ9eSOuPS1wqoHOSlDRJG/HWtw9Wo+Hb/pNU6OBG8dU9KjG4qndivGNAR0pc7YsN5HCtb/RMHLPbrVf5zB7O9F/r6ywYckst3JAPMrGX7rSL9TU/Wtv1hrTe5yL37u8ticdB94tCbrR4WrXHNnqvVy8r+R+8MaDuRZS+S7H7cHSCXxBYU1mnDupb2Rhv02xea79hGcGtF7aDSv7EA+gbDbqelCUvLRB8yWt8QatPhQvoLW5rLVvGflXgHKPodtj3oXaWX9l6F6ETaUHy8zK1s4NzgMbzYrWpfrdTXJv5xPedKZ3Go9PiN73WYdTztbXsVkllZTZXD4DU6tnYKSvaHX0fC+M17XQU2C+6qLMVa2vE1jpKGFGaeTrtbmmtbMKp6cK7AlOWDgH+RJXb/NHcLSIF3KY6wP+uW4/pH6+6+5msr72/riUlo3n98fxuVeaOa3oWC4OgkzJBvAmgXoaeMGgfBdBJFfN8FevcSbcYJ5qvbmCnOIrQ+sTiD9xhTniONseKjza6lQqfzqEznSggxvsZxlfm+4+y0F2j4d1rWZva/G4vs1eE1DpzeK9/xQ3ubHV8Q6wtQEsEWzy+UI2m1s93c5homVQ1TR7POCKzDtF1VtancbCDkjWgfnsUj/IvhoroZl7TzUHaQzE9TLn+q6mtbW3brnUtJ2tyra2Pqui8MZWTxCwelhY+BMIrVpf+bywgn4/RoglzAm5rEzs6TMfH+e0HW/8cdDBTG/xa/tbPf6OcUuWllDsWTYblCMMKo1pagHxYXoyPFQQdvka4rNAVYKQVgbO3OQNSPiog8LE0bHL1+HdXfiMdZmWbOmbdfXq4LA3W7sEybIEv4pKCPyuTebNVs/9x0paOs176losmYDfnsWdU6mv5hc/zxefNA+m2wCKnekLjdvmDVA+/6xMrM2zbhAotx1UzW1ugNbr1rf2IdVROOti/tZW/U0PUFMhjD2L9A9tLUIaqvQ/tBne5ukX2IxoK8OykW2/8zlmF/hLveOcyObV8+j28/Kx3lpngnGQox093EfL78puin+z0jyqTD+4J9sBf66hc1QXiV/8CYeZ408ob88UVEAsFcYmaJozvF9HCQ4Sc6XNuLYy6zod5q4Bs5xFAzQ2gRnf5rm9ICyeCP6mVVT7eXmy5PNs+R6qOaLGbIXBqY6qNuDD2pwQkt6/jq9pors5SF5opXLHgfufm+hf0zGry5OU22xK0hQ7sW14iTBHzXt96bojOzL7lNNVh+o88aAasR3sO9vER5tTIcjxqWB1ZxHkXn2iPiMUJ32R4j/OD3InW95W7QZt5vI2aVXezGZ1m2NXibvfAiZmodMDzJVtjihoPnYiuFWxPb5Q559Xk76LzTWnY0NiPPgHl/gRr2nrtHH0FQWmxOaH2viYZh5/feGurOVr81SbkIm+KxY8ncboaPNMW+dkb4iH3WJ6i7aeafOqAGPs3sHUp82buQ6HejjhPWsihdY/TuZ5LnebL/JRwu9yUHSYvangHubQmQg3BHvsCm1+blus/yZa/wuW/dqWVVHf39om8H3K/N62t5zUJE8/zPS5yEYC+ymvN2RyzWR6A3C/GE4Bc7it0/PmM/ZD3cLW4bY3qGPWRUt5lJFrUwUQe4apaos3g3ExeNNd+mwNdgcXtuKzqTJQ73vqt24KtqrGI3DbbGuk4QfWiIcLLn6TqVdMcGJdIthPBuoS0F23O+3G4NbO49z3mUrIX5hA0eZW2aOtfi+XxyRHlUfqX5voBIq1tmE7qYtYF29X8YLkaVvCiJQzYKs1CsLMEKJ2yJjJJ3aNEPeKlaVCqxms+338iF+yIqStnqe04a2lmNP27LZP7KhrUxsv9ex69BaDGbz1THrTydJPigZvpYOKbmmkvyMtNvTOcuNs7TzlZGqwV6P/2Td5+Ap3zRSHmW2NfdJir5kK0YBOcZRs9Nbn8W1mLfq81ZNUgLsvp+nkDfX8SWzsrVPsiwF33ByUXFlVpLjgNEfsYqpdh13qld+uK6jD0T8BnwG0TFkrLXPf5oAvf9Qcm6IrKxsaJVqpzemkGqKSvRq7RoXmqK1inn+gg25CiOcpLQsYgIdPVnPh//7tIvQtAjVk66rFQ9q3yX0aHIaAUsBTnCwEyoMMy6HlsOSw5XDkcOXw5IjJ4csRlyMhh/EgnfQgytFsjGdBbUXr8rzZe2PNVBSktbnk983of9xoXOq83ky4qc+UrcdGfKZDC5T8fAT0e7+ivwP4R823ovg3mVR3qP385t6TukNH1wF3VFFAgW1TVz9B6oebTYJ+rSGAZzS9FaW37dfOF9r0tM4yS27pNR/mPxmnTOFJLSfCwYI7LOPQr04Hyp8lhm7rOYVltKcqrZBaBiYCLoO2SZVbIOL1t/fCoePdgPOpoz1QW243UoParnf2nvTUcavyX1ZTd2ed80q/jqTPWQbqdacxNIzn26YhcVOIPRxTal1idt/V5yqIwstyvMICDxC0Iayw5NbhzkW2bSgL8zCwbnH7Y+T0BTMt2Qlm5N+OyJ3Xz6Htc+mHPjbtK9xnCnmY8wHaLskc7oCGrNm/44gzIErNzPFlacjKi+0kSSTZYZJ0hTPXMpekvzHT7+0zpVJu+peM2dl/pIwf98GWHZ57r4nTIGxmf7kdaM9almjuwp3l5qf7+jUcCc2TtiGvL8yW6J09DrL67T4To0GmLSD529nXXHb/ejwI72U5oBfvy+OpCaTX3KFY6+/qO6U7RtB0ulRxdA8a4JQr7NPzCuxQuV0mpB7Ou7Yp4c7R5pbdvXtB/bpyq0FVDYtgbi9P5nTXbumDHg459XBgmDniwYF8X2/Kkvrrh6cgWlbzYL8A7irtPMsc0Y3Vjeg9XfqFNNEjkAY0IBGKHr8kLLqlGx26s4zonuF3eLKpBsgC9UN36+YeuP/poK+o8rrOgcqCITWh3bs/PbqtkvWH1VMMsAW7d2C2H9qD8KfmgDLHBrQEBmoLWu7Gl6aq/scOR7B064AAd1/XKxq3RMnLLQsOfRtmWybQyWsrbTq/4VYHBBIGbAv2jiZuoPgoKt9Z5lTe2xpkroaBjgWv30xnXEibauR1rjlvBd6/rkYualzqu0LXQOOC2ijnSmAJeAIMdC346y6+5D66ORju2VD0ex6XuRQH+gzk3itWQ1TkWRBrPT0sunVgEEsVXRnLyugxC6aO4YuHB/40rPTp3c83OOBLQ7/tW6M38sY7Q+fJQTJ8vKs0nvMgDIxb8PY6Tm4NEpOgLKFcK7C7w8CEBS9Vh+nDGJjuUGZkTlRB3Yz1xnGUGkpCy8nukEou6BfkJsCEpIHO7RS6kxoHyE7cT2HRuCvMI2K4Yi+Q35JWL5B1WvV5AA6ofi/Ir01Eo97pDvAh8beDg/Bi7M7WWxSKLwzgTRBB+S/xlz9ON6AzYL1DwzVYtJ/odxolNbefJr63LrBPhfiHRGfsrLTtSd+MkarH/k2Tt1dEfR15bWVj4NwIZQ691R6k5+EoJwPuh2T/SKG37ZYgeyTEXyJupyA7uA/1AxnAo2/70KswaiJ2nHwX8QtTg9gicHYQ7x8bFH4Jaf8x4j+GBn4NNPgWxHEB0pnArw09Uj48g3QWhu4LdKSGeD3RzTRKipqQeQ5oDRJNoGVCt20Tkuue8N4i/h5pL3X/EqKh9+0nd20/k/5cgvxcktolDxI3vh0k34CeyfRBqdTF19C1wLsA5YN3fN8w1ovtZzXff03QZRKUk/eo5s+Il/QP1M+flURnKf7nEdT3gUplD3ipIHZQDEyBCq3lEFojRsglJxQfHETHPFDJ5uNBGJfvB/YtiJVbKVUEvL3Gylkh7NhAT+zWoxUttOhiCNJfQbnO9rC4ToXJ6yiwvgJdbOGnNwx/RkUjX6b4XMUzFOfGqbnKv0JR0upmW1+fhpl+qpkDR1Y/weJ7pwSlTWBfoEY64OAki5sU37wzcEdC2l2kUu7NY/CHMnuDwtpep8BBLuuEctebaPHlikY3UHeKzerL0cBuzRC/VXGb4l8b6ZaHy2KZZitIzbD1ImuiAg2xYicItqhRggC/XXHT8eolxXIhjPBfKPP9/wB94iWi804QQMQ3WLyjLSjvB+WiDbKENalcFvgGm/evod8fVu2W9YfiDxX33BIUnUPliUQ70tu3pRJ/s2gQjBJ4GPmx9/CgeDGkTeF+LhwcmLVYbrLy2Lnv8YmjgpJBIn+HFCuxncNwXw/VLJz0f/+I8Aj8P6SQsT4=", "saves/ankiian_t4.state": "eNodeQd4VNX2/d777HPb3Jkzd+5MJpPCJKRQhYQmKCX0EBATQBRrQpWa0ESlBaUoiIQeAkizISqgWLA9BPWnPguKWBAFfPZnmQF7y/+8/5fvzty5mXvPOfustfZaSX1Xmao70j+EUDYfABgEwfqwPoPglRdXhiEd1Keof0Aa4b4zpk6eXDMj77LauTPm1Eye0aIFeJGWkI4SCH2rBANMsMAGBwLgQhBCoCAMnr5ddQBPGBDQT5f6iWGIQMCFxzx9bs6AyhYwHk7YkEoIfR/pi80IIwA7AmaDIjA4D0oluO1BXAp9mwHpSkiZzwFZ4BL45EG26AlRTABtjEF9sAYoru8/Aed7mIBHIFMf/fU6ZA6wvmUaWIXgG+1BFYKNXREHopoGpme1hCq9UDAZZRTNaeDbhfrJhfrJyy6HFDkmqCIIJANFIK1BmHQtOC9qwQ8WwUTUS29uflp/9ZGLIJUX0uPWkl0EvBc76/WspFAFUAs9c0BD4POYpeZSkS44el5LuO4fSLVTQ5APQzJSCOGkr1/8aCEkpZ6rHysE1aC/q5eaYUMPAdbjYL9HfrwIsmURDNiLZT9hMrMrXMhLFII7HaxySGYV6odPQDkD/OxiuATB+hMzxTDwc4rhStLb9E8+iXaYbp9rQqgA1DAOUyfqqYe5gykbOAmqAFyPtlOolDNxBeMHgFeLwP8JD1dJvQ4fh6zAnZwJSrCdkIWPYGq44SNWgFzPvlEItBT0K7YjWQxJsxgiCasYjGeFdYaTurBS/aQX6fSEHgzmo0I+ALyS6VZ0fmP7VvTdCnDfY7UcvGSwp17KG2Rg+pJQEbj7QK0zPF/1BPcT5msBB5n0BSjWCFwBvjcYaBh3NiDyD6KNemz6wOQK0BPAo8Rn2feHAdWF9X72IS6GRLRY45VYFzvWA8N1kzUKYxlF4BxhjTrxAtNXGCwAWcl+pn50IXR2IEFoDedk4hZMoa4zNbJRDh6+KLi9nItpQTPA+Qf0I/0c/dsKsAtfxpSbWwDB55jzgQg8ajY9Wcb8sYmnkEdi5feg8rkNpNwk6d1ICv3icwFQPzbvBuxHfMHy34FU2yuleSPjfaynzruZhkGhlGFQjYLWojlk+uw5s2rnHNdQ53ypGWYNBe4PkU1xkAcxVdAZfFjmxOhOaW4H+xj41jFQIzS/FwKfYY+qLGcPd9V4fdGKOeXg7Obu4rwf0KDaxdhKyApIusUgcbPgUjsZ1Hvqh4pBbXqAzmm0qC3s0lZSF9nm3ZjqXwjmZYzfGdyGeT+L2Rw19EQDJGYaV9eMrR1b+xDgaY2UbwyzArgfeOMzwNC4Ep2Bv2HjCBjbIA6Feof0aZo0ZzNFPr1wPeUdwY6YLtGbxOo/QFpU/LgGwwxGZpskva01Qwbob8JCiDnFUrkm1ugiT549+S7ACZDhTELunseplpPQJz3aCInXCjVChgwN0IFoHZF+joWqo7SSetMUnQ1cCiBGSMo6IeqpyNR8wRldKJVN5agVAYxWiHWUD+JfMvCrzYMAtzPqvbOk8QfgdJDTWQMgrEZJRRtAfcK2bw4DXUmrDmJ6XaHDzEflMk656hMtQE9xzCmEmVpCtJYGSiVNA36CG7BEn0wUKZNQb76vb1caWn6wEIJVIXzhAtUl9WYwbppO4Vgjpq1G26FpqJo5oHmoxrHwI1OByssxlduHUSA/wjSEAsVgD5F+VLOiGMxhZOtnxzRrZoGuBk0BnInpLgXAsgG40aZPIeFss6xDIpY5RO88BzyRSYMtrdpzbCyxuNL2sypAndVrzNa4mWM7L9hJrTm2J+o0eaezPFWG6VKPn4GEvN9VfwY8Gh1SZRo61Zamt7rY8GgkevCSb0305SjmRUw1ENzFsjV48jNQXwvB7lw+q8rBjpmF4IxivIKEPKLlhb4W9j0kC+FmrrP4S+E7hXtEitRoltzAohiitsZgDZaJtBg5d/r42uk1PREdyrB/kHxpkvJj5lqqfpPpAzXoJ1m/HaxMxw++eI5S0rGRvwA/pBn9L9DDqndMvIvgXk5HPF9ORF4UojxSX3JAt5VgMjJd95ohtq+l3H2hWnjRb+V6O6alPNjIPA5LcK5czQ1awtXNoQD99ZzR30wmym/imZ+y+RqaFWP0dreWr2I65yDiHps/ZbUOSPkU9nPHgiqTZBHQGivY187Ad2N6TIHVOTIV2O+EPmVHl+JXQwnFUu/rLg5tiDvDsigdCEWleRo8Y78d0tv3nvR8U1PHD+Hug5wyLD2dQyLft/W1XEkBlzPYvwxSoUvYuJ7xMd0YgXv7ONbERXGtqPZFtu8OW2qkDRP+eYhppMmzIKZ1IUftjcm6vjjCHWVjxVCeqbf+rMD/+0yUhovgbnnglhC9gg4zTtPYTHGkHHJ0K8OONt+S6fvF4K7/ysxLRl+XqTZPgdadmIakUwdWhe6BhXAJgzPSMss18XVBy7X4Z4eTmT1BcgzpAFNPDL7Hs/FI1iAIaALfpltq9mIjrz1mwNqcIkjY1zp20TMEt0JJFAeX2Smf3hNqvwazkbarpCZe8BZEK47XeXySfZoaoYpeIu1ZK9jXfiBYAfhJSDeVhCai+bq0NY11owlu80WDoXm/Howh8CM0pCN0GtRNNiyCKt2TnaI4nP0xgjv0OtmYCnhWTMR3+uXQCj9p2mz4gen/sVKUdHX3j2laq7kcoAZw3rVSoksu986hj0H9EHRoEAafFbNBA28N6N6kbjHDfqQnqJWsjGXQOQTRM4gnWU7kIHt3HIP8idzKTMcdPeEMrXAXUJcxqSvH9l6RzNRuhinK89j4FOwZUIFleqrzViNMgYbsIuihVWgvvQJnC216zPXs7wSPjmPuA1zi5w66xUpJy4fF6C4UtNrk70QC9Qjd7qNU1Nd6IrpzNKgZd4crXgy8XV/76vpXDy56ec+EOwBNbb0GQT73eUzk3eV+b6SA743QYcv5m60P2Ocdj4oaoxSdl6HCDL/8HrRUL19iHDE0MseC2Qp8s+ABSIP8COVC5tcZn5XqAb5AB2xtHG5je5WVdNbRwYEgluR45uxo8IcnRArdQaBiumSBe9n6LC+pyfz4jSB62nQ36H73JeSFB22yzzr90ZoK/HCBWwTU/0oJb9MRGs/Gt4h3CrmS3yENmPdZ11vwS6wbfGEopBfZhWg2GhWTa2dM2ZgxOjTX6XsY+JKfIZ34ieLUpGW1CWiibTQB7g3IIxrJmqXbQH0JMhnXyq2Onoe8ZObH5JV/TNUJ7UG+sjbjgQlxekk4uuXsFCfgJ5Fdrq3TLiPs5+hSKImleFimS26V9pWC7xPBIja1/cLXBAVZdZE2PZjhHGacb2BE6h6v9JB43uT7ZUxTNYvrHNQsL/6ZjjvDHHsSJg3dVWkNIyzAzu7r3BYbrALoSSCH2HsghbaG2yI2Yno+ahAYyV0inYy5+sMYjV9dWJodMtsJY6egl9izOkpdVTuPfN2pehA4v4NdVM4p3/M9LQivst6rZEQPaH1uJbRQmh0YP2vJnwaT0TPWi5VKsyupec9W3aV8G3WXxmVoFV7KKbmNjbVRuoHpLwrE2b3Zpg3hCNVXCH45Ww2BNlASy173M1bRGsGbFwZK3ELuIFKhXE0svi1JfwunEHm/8PEpV5MNr5cynz+nF92TQseXmGa4k2xhwHanQZ9aSd1OzACmWVM7MJ6tQ+ibevZ92f6Y7paVfwsrafd0UgW6JFL9V4VjgZ7gpAV/B8KJ42ARWBC0X9cpoVxniZO2Gmd6fkiT9ZOg9aydzgjOZqso5GtVDA5n1rifzRTPdapttVdb30iNXv0v4kkj+tAkmRf2Hzeq3gT7YZ5IS4ptZ7zg41yMKfhFWPvJ10x2e1vUFc9gKpHQ7n054zUm9mdt6T37F2+2rYWNIHt2WLeHRPYw0GCQJZhPzzwQmJmw+AD7ubdb7zjaR2lt+D2s/hDQNZAy90IfBp4Vkr62pkHNoB12OClOyLP1QDvYcz6PODsYgxVasOWbVLoUwt+Jqv+AXeD0xfNB3biuj3u+pdH6M/PzIB5CmlOkUpbhO+tYTqdzzgI7+BTTa22d04z7rrXTbqgcAj9Y1miOaXulBksjFta1Lwer3p5K/X4RtOqMCBfstOqdpK/hg/16auJEPw+MjWkkOtZXwbSnobhHmAuFWkPhpLaHuhFq721Th0TgkBXBvMX2uoS2ndbjViLr2kDL32LXBtJZdBwKicJQCqhr4IPXFheB9tWXz50wfdqxQ5mAT0GUhoARcVNS/RBC7qA9/nZtk1AbavxBhoMNubolTEexEDbYNYH7IIcLN7hh3AewxU0bbZnw35IPBym7UB3UiUWbLX4SvcAp53zwiI2o7/zBp/1ddK+jHahNJB7rojYAOqrB15qkCtDyH6bK21nL2/W4RHc353bG5VcgfChKlGaIrHMGgf2X9u9O2RLNyaZQ/bVse1IjrBJ3WTFdGOc1pndyMI0RbetsK+rrGtJPZL/VPZSyowVwCYE9JWQtkH5MI0oT+EMtVeOsMsv+0xihlmj6GOsAT/3C4czPVTp8ue1rrXKfZHct00MUaGvbn1p+ts67n4LK4nupr8dtcFgonWhhlTHlcPDmuK19++vtE7pHs47z17BODyGfurtqnACZKayURZns4aGL8e1SHU/4uPC1GNBPYgjk/erdwmFZouB2N22eRVc/c20np0zaM4RvaVPzpLBc2bIcXVnylzB6YkPgL2F1tXy3qNTqZPnB6YSd1dh8xp+RdgRTIlu4IdU76PlhDdEbNP/5PlqrrtOGdDcGX+mIKfYHQY6oyVKbLJ3KNXVvAEt/pQ9BsFBIP2OJ3GX5cQ04Xc27u8CLmZolg1mdUmE/0RnKQH4vHnDq1P3a2hfCfU6ZWiqpA/k5j7pp1JKkk64Vk/wIqYuFNZ7ykvSHk/Z8w17COJNHA0Nz8zVsKa0NFSDu4yjqXpNm89ywyTMmTlvYGncxz5dG71BKusqJS91o5E1Ay1g3GucIDME6uy/Jz8HfHqisRCtpzSQYiKUPsbGIdvoaQe6DrL1tP5yZLfrnqJkYt3bqyB38T4bV0fJvo9KAdJqcsswQLnFMWEYp032Y7SKIaQ0PHLOsozKmNdypqjHyfT9tNQRPBew+6EeLfoWf2IlpDDkbgD8CcYRpXxdnXcIaGR8EbkOHSJpiulZqn+a3livnfcCylrA2S1/ar0uSbUGJkczR+78lNIJTvVVlP6yT6gkZTtJp8/giS54UCW3pVZLlBNrpDmR6ntGCgg3YoDOgDpWYjv0mwLGrJ7NJT3b7ndNGzBoMWd7FpkpgOKZtrnMDYGJVIOxkiyPAsYCe7BtsTxOleN5yK5ingGhDlAa11bYQk6oyG77kWb6aCu4g/hAgV5apA8J4UvRTKIyDoux3SPpbzbxoUbbdYKxk6z3WLHrGrDJ+QLt2nVetntJdM34ewpkSwokeUFZGyazLZckypEcsyIRwjsADX2nNytV2lhbBNlX2Vw/ltEBYa6dCfQHXkyaLYU9mNQ3Q6s9JngqmLIekXOaXaTMdHhiqCvRjfPcQjNWt20ni/VgQs+stnSErlVa82DxRudkzksEYwEi/dJZn3KRKC9i4AJFpF2Ccc0hQd9zLZcr3C7ZiieNH59C5Aay+tj0/ptFfA849XPUG2OcFhsqiaPi/Qskuy/oDPjPzMdAN6VqmoSI4pUZEyn8PlOQUwgE/LKda+G9rfQZrLPTpotqPFeHYK7LqcmH44mun0is16PbNEqZH6tVdYNFjFSoNYeMtJ2JOoZa+dSvB4/K4ux/sT7X50E1hKixgeDriGY/L4n+EQIjpFqCCEAkHIexfwJISYdnmWG347F4YMDt1ILpXVgcttlL2b1impoPzgReWh4PQDCW14AyxGnpLCD5QzARNMhXKETqO3JXFO4SfpZWnQmv00KSAofIbkcyWGB76eQimh2GQnZIJTWLniwCvA+6brxLC8rGgNJCPa281I3SZAVF81xOvDjfXgDrmh7WQyZS7VJNUi+ZebXpB1GVRXn/niLB1oDHL58UKsd9+pz5HNLN6Q0cBVU2vBA0Le8ISvXL3bqbFl1FeoGCyTrYO/QgBjYXDjo4MN0Cg521+GC+34ObQQXboNp05TPzEmMD15UBxYURwVyAFSUh611mFfqRMm0/hjOcr3CsYF1VZeclYlgj/eNaA0eGw7t4z3SpWw23vQiAVlqdj/GcpjUDnA7b6sZ81BOhPNg4wmq68i1+MwnQ3RYGAFqd9QDNRjryxZtLcpV9BTUagKsY9prkppyrmY2fg2ZcaOkJvBZyWFdGMw4eG18q+VxMuvFWkjE3dE95ZqZIFno8wWLVjS9uOUXgePOxUyudzfU35YK40v3IKVMVAq3q9ltXldMIrcU05TcPlTwfGY4M7FewjUd8uej+ewuCPTGXlSb4mbMYCOgTNCe5cA2J1JunkGO1nLRkEzodsN0A/6+ApXbWp4JxivONNsTasf1OgDSzus/P+5tNqrRY69x62n4Ad1I9WA47BMlOrzZ3ByqeGO26kMZAKZJRDDwlGF0EJ5rIrfO3sc/SUMWZv8MOZmXa9vIQT3ejbFh6kREJDawIbOu/uZ/q5LBDl/4bzojzSwt52fk7x1owl1YzPAQ+Dn5z6fGlsArLw+9hP9EdF8LDgNTFeWki/9kx4IyvtI3o5bhHj5a3lJ6IKH9M+xbOWJ8Xvo+RV2uRVfOXTYhP+CHlnp1vVc0L8LPt/yHPnBMlhENNhXj0vjDxTVrj4JVcneAjbw7+X0IWrQp+F7CKRqFSttes4RPAD90sEGmCqpdMIvG568mtZ+nrQSHE4mWIYitAiAu9mpGAw04cjZhu6fZmQcQl8oaozhgEN4Xugblou/y2S8YD9WyhT58XVwjrMsYSOvL9YAcocSEZXZy3zSVGmmVoH9tjgOQ5pfV85MOG91R5bybbuwQlIHwrjWVCr7I/Mz6gnUHfW6RDfRflqd6xfAWaUcbzchHk63oRLCi2rNpwKAC1BY1TNrZNn39IAOBnyrUIXeEKSfl2Y1TIy/vr4+ZChEfpXFhVdEdDY/iuLt4FvbAN3JdMWYjPhc/9Y2Lrf2HV9MKyNP0i8Ogp8p1UT1Jnyi/BUnCsg8Bqru1ZZ9Tmas0Zr8Gdh1fNaeNX/72jhy+HIMAwnPX2vcR/EavHiHGnMwgt5voGtBP/OtJyc39lawRyUlGUXmqYm1Qak5WhcOWHOjfMm9IcM7XxGhI/kCesm8ONHwJ0uf6KGzELQOdMweKMIJ15NvPNPOBzL0k1uicSXrM7hyiIdKXI2xL0BTwPcaaZdzsDVISKgpy931tn2PvCxUyDYTVqn7fkyXFEYviCYF15k1IF5jcQTId7CCQ6G4DHK2wz9o3k3hfZTGoNljFOv1XHS1yhyS6WTXTanOmn9ywzbusc5YISLZHZeMiDNvlsc3tAqXul+CjRjVQhaBqpc3bPfTQf8UE20oSlBG0arz9iZ5HaKeZP/a4YjZwkqsPQ71tGvmiYi32Rnxpe4rzHmovyxOXTgKgYD6KiNnnVEhOOPJPDf5J2emgWv0M6FjLOEnAXJxD1xb+VTVjhrB0aHtPRL1VF7RGBJjubyizZu/ztck1sECdF2D9zWyeeO6Jp8eyadt0stM3ycUs5W2+iN7nimyaPUewzYjDzX1tkJy808OWM/54V+et2ukhUaJx2FFxiA9Q8JmkLatN6X3dDZBTt8BR22H6ew3dvMf5Sr8Lh294lAMTiVyDOgJgrrpEyZKQzssXkA0ACb3rlalSnj60i16qM5550B7/eBIT8yCXfvd2lQdRaNcvBF9qKTVCXI10Xs55yqqZEATf0hIz3Qj2ut/8hyHkJLN+L3yHk84n5kKZ1zOyIFM7xY1nJwDMaIEATogp/dRitUGzAXWdgGkjltgMVlQCUOLWZ6Sk8SsOXq3JQcDj6UCJWn4+FDk3zUxvWAOEt5VNgiVIIrQy0xTxTvcqvxdqlTTVEg3xmkInBpIF38hmV8ivIkcEO1F8/PCmiPvsEJ+2YNaJtpzrZ8qxAmMYS0gdZb9pXBinn9y36J64bbQ9qmg2hfMWvqseemHXtu+kuAZ6wMl4l73STyPw4M45Qd6EV7Ki8OAe4A6xf2A8eAtoNRjgtE2P2Mwk8udWGFzKPpXTJK7bWYDlbPAHtcwF8TrtIk88Pfu3gHemqBXXWQ7d95m3tWAp0w+DcQXV26EWOBPD/2d/CoDr4rKexnaAv02vqccHyUlbcQl9kpgzba5m7knldzuU1vgBORdgV8mUgF3Yul7VEsW/vkJawugMLpur5l2oNNgLuxJLcNPKDq7ck2Gdc5X3bxSnRR8/K0jOqYHgaKIN4ILTHcmTDjkFEd+B/ggj3N3WIYBPX5pw6vxMrAHqY6EPB44DhbDuLkKbRRmxloNNL/c8B0kI10Ld0v+SDLrNuHgvmBrnDmSSqTeBYCmp1Hc+r0k1UjBPL4ADWC+QJYG3okg0c5f4a8VUKR33kFUGNmSgYbHc5WXNfKy2xH6oW4EfPKt0XD1qVWterO+Ul/vrt7D/AzTHeOVs9czo/7GZPB7cI0HrAK83HMkjiMwlQSfeMsqmfRiyV6QmAG83VAvTP9rFag1tgRL3P5NZ5ICU7F5eVAu64I90W8TNwVC898HnRJddRRb1skE7aPazhb7GBxTGN5Jj/Ob+VdsMNaKQcibURz0IRZsz7bvHIAaEv8Wl0wcm6ahfwq+2IrqEtk4DsacB+LF8fiWUs8Nh6/Zy/xSTxUDg3RlGmWg1lIvINjVhHkBD41jSLw7YeyUsBktIKk00pv4VA7GXiA0vH7JX5n82N5XmIhJQILyb6aksGhWtovcuhdptrpzsEcutHhnUq3531sG3bsjJc3095vp0y9aUZjBu0hOozBmRzBV26ip1lyCbbnd6286LrBeZ2pEFu0qMyRN9nmLpsiObA6ekS9LNxkpp4FfoLtuPJl8BNt7oqmW/CMMD0q6FG9mCNu8CTzPYiVkMyuhKif0xq0WAT9XP3+NwPf/HPuBe+iGjU/B+zHmEonBtrZWGRxL/bpkpEez7bNeVq0YJ5qOQyG4cx3BS5D2fhTRlqpvmzGdNSoCmik8pwBOLNOmxRrkm5c1znFY2IzrQqI+HbxpXlpnG3g2p58IkiV6DytUTs6URgY4mbNHsv0AjpjO8XxB2fEEFS/hD+KRY59FKtUI7TimWwwVzay1YbfiKX0drOn3rkG/4u4hHQT8qPaN622v094FdH8s+FERiEY3yC+jPIcnzRgmhM9P1PWZ7YB932h3XA2fCFSrRNtwOaPBLaFRFZb8MSGDqLtzXZrXZ7WupDzwW6rydlW9/NRrhjRHReAn9sW3N/65xzwoV5bdwL5lcjkfsGXyFynLTx9Jdy70XomQqU1Tkjg4yDrIKmtWiQ5MDOPx7zsQF5GvbmY0Sjg4oQFhXkXAoWs6a1+tMVMo/+NteNurO2FOIoz+C91xsp/fvZkeaD7VfT5VOdnpt/wCbvO3gZqN+KL+RGnV6DfIlRHuibCgTGet3w5ws+qNCiNaX7Jo2xttNPsFtk8EcSP2V72haAqsI1Y+GOD/g55jw6TaGe0fN+CFinex5iNchxn6SJGlZ7NQpzqXICRx/ZPPbZ/BWB/fdWf8UkkndXOdQukjnIezRjFXUCf8XSZyNgGRoKcoxDT1jYhjgSweDUevwqSut6s/gs6k/SLHEgmtM5pg2V/y9swBUZPiZ9HdYmzsttCVGwyUjZQDXLfOTVzmgAHQr7ooI2YIbnnZkgnpTVW+jk3wyTU2tbcvBuwbVK0BWzVkT4XzjWo7pDo5dznBLKktcmb5sF84KmBs7Y0HbKG5/s6vKnlH2fmF8G5ghJf9tReO0g4TPKsTCOI5hMy2MXxcv4VcvpIt4ujKla7+ZugdwI/sr0PV8oaHUrUw3IEwoLwyHHoJIObstMwTHo5NcPURzlOMmeYVIvkOu8uCNZK1T35VU7KvkMC/sW8N0R70HmNre81SjXjC2NuCX/Pp+20o3yp6GJyeil8DeUm7duGgfu0wFk3YKTTX3S+y5dBe2I4lpgDyuCAn9Ua6CYwRzH+TkK/ztOdui3QPDB1IaYFdDkKIIdB3cuKhlJCzG6BU/N2u6koDZ7j7Gb11fIY3V3rtAnaRehTR0nHhbFb2znAFvLrYN8CQP9OP4/Ld0G4oChyAD9FewWr80DmVqk3s+xSTlqP23kzYAjnuadFEt6kc+7ToIaHPD+gm2A1fGyG3Sh7wR9bePnduDDEbxbkB6fvL0xDs02J+arZduju+hb0ziKEXi3rRrFdS9IoZ/gk2Nn2ozNlJGbp1J7xKPYtFHLdNhrnjIhjMZw30pZbq50YtqEwDhN+oh+oSUDS53979fvZ6Y/idr7fzstuHYsdcKWd1OIW1cp2xEqH1Sr2LkgPMFRRM7t2Rl3NrKlTJs+rnVFYgpC8SH4IO/G7zvTSTXgCRta3Pt9/x6W2MMEm6BEt6hoG44/MZo245mYma9ixQzMmT549exFEopg5JmIvAnx0odidofRJn/cv3bcIrP367uHH9tZNHle7EOL0koMLr4LOTTfBVXBgye0XwuD8CTs0b/jsnAlTtaZwuYoEfoeRF1oX3ju3HZwPVlzbSn9pASwOSDnqxptr5mDUMy61OwG486+EgkfntIfUJXScDAJnPix2De5Xe+vcn8Gnv7t3A1C3gppGWsa25En3VqhVphxxy/ivKBrF7ztEjPmg6jDYDSZuOQ92z7vPfK+JYFjGZbdMq7118q96Ond2ingLQM3EYCn8fXROZ0yVOzNxh2ebA2rrJk+b+wNG+etbM8yFoOag7cxCdyHUmg4Pnzv1BhHlebclrVvBd+eD8yvssAKWTv231M6eRjGO9I4EF0HS/JDdLgBv1NUG08pZBPuDrtn31rmzptbceh59ri9KhhaCX5pTsHDuQji4EM6Egs7LD9S+/MCHyVRRvZ5hVV1Eav09KYPZWxagUw87ZMgYM2FazdQJup5+dAZGdPmSxil28TXLpNGY7vovR9nDju2feGz/+AmLIdPL/ozlSak+Zfvihz7rthDcxTrBh40R2iDMrvlTDxJIRsJ/gvpGBPss2/XkAqAfIzmUonCzMbxmztxZv0GUu2MEF2xpWx3s8e677xZMzEtlkqN9gFG2tvBR9cAG77dWVi10AADas6H5ikKSWUleFnfDG4Ntw5df7KSEwx+1V4HlYNQC7y1UE26G31q8o/X9wQ0UxRbaW2T8X7gkcCd954T/bg6V8IZ8F9It+odZJ+RR09TDG0A2AmdMVqeudHiR5w4F3uHxoSD9U5KrJuXxOoxMMVK1G9bKe+r5y4Lg0Xm4HvkDVAc2NN8rDUGnlqqQLtI8H1dnOA+vDC5kunuVUwQFSIf9qPxc0F728b6iOA3J4dlGJk5i7NwOe4rQGDDvZlzQXbYCOmZEmp92tSXCd3sp/apOb7Rke6S8KarzbXxQ1V81LnSpxJWWC6lZT1eHXmd1Y+b//nrl2wklduXz0UlJfi+h06IO2hU3qFpAut/nHlIlN+bjJuApXfDmFqF+U3EicDtJK8BTK5B2Y0FzcwFkGJ8AD93IHULcIcdsSXI0eF7bTF693Ny8CtcA3y5p2yB6bKxdDBHaQZktU8FqoHvJnAC0oZ/qA4DzDZ4AfnQVhM5Bz+J0aNpGx14ktY92yhZaj63y4zNAtW8KUi6q91c9U5Ry1YyN0moADBdga1aHwPRzDoE7vAkr4HS44dNo6PgqagHyTJsDUyBw62zjCNIJOBc+u3GJ4VGRq7SYyI/o7aQX3uKnLruXjY8I3801FrOq3ehoR8nbPRqumboxy/QkVS+QgwAj7fBZE38GebVDX85TAslYp0t4FtwOKyy5it6sDf3OpXo1t84yPhHcFnhzmD9JuuH4s1bKmr3xVhklevHKUC/H+lrQ7tG8mY3oZW20b/m8kgvBeZbbYqEvub32hFKMwTg+yjsgVXgHOw9s1BlXnJ41CLUSPrBRnJohx0jf/wXdMdI6Cmrexg72eaYDFMg11DbT/I9OtIDi5+5cr+Lkm/HisLi5bbpvuAvNR+cRpgdNPuPEMgWqPjJAh0GHR3vfRtoGVN+epiPt28hTkW8EP+tGoLg0xqx1b5A0l4IaRzbyrBI6h2rFjtHBFPGV/dQtGyvkZYK2rFO3Md2jUtFWq/lNNi4T+DAZD7InT9tqwUbWhaXnlvJwCFzdZHxdkNIlFndI2U/akyRNm69et4tN+KIgbalVBsj/AJ2fy2XS+GKVNV5Sqxl8TVGqq1CLNpodML+f44vRYPdzRImIy3vADx4F11FYki1fAnVAM6AQ1MGNwztgix6S24y0M1h8BdaUJj/y1facVI65Vopih/7IFj1AHHUDnSS9hIEJjr3KUX9t7CB/FPSvXrwfAnGxOFwdPwnByJ2fQAoS3fkHpZawoMkl/H7DR7GUsTNX6mC3sANvcoxiqNAIKN4vTUpFizKlZVE3zVEerRGfzfnS/BYwBjxYr7dAFQPPt2b9mheqduTuO9ocXwrO+o3okMgHw7zU2UlFoVAfiSGtU5R3HQyJp1pRyDIeCtJyDOWupZXIx4PUEvgIG13QPqKT3Vq1lQXfFKUld6ohQDgLfPgxflGbysa7cLd8MpiS1i+gxkRDtB6coWARqi6b5CErr9rxIWV12kTjiIqLaIsp9fk/NxqD0XcP7ctMZam5UlDvpEaqPRXllaRo06AovhH1ww8BvcD2cigkrUFJb2QOrFepopNRMHRZfqjkPEdLCvPyy9UGcJS1yZKfgLI3OXIRK2eT2RZbLmJNF3cRY9t31thaQOjlHtxC2t92yk23XHOxKoQAVXXmHKkTm1A9NzW3LqTgQVa99KNWMU1Zp04yRDCD8TGDzMH5Z7HXBrqU+R+T/1xrXV0USyn3rXtpmkyYrTJRy4bUVRm2TAfzwJ+sZNDsyynL9rzQtRLfIbmcacBCfoiNtxjzL4KUULM27ZGt2Q+bSDM5Fvp+7Z/iQscPp9Ixm9dEffl576TZMQTWvttj3mhIOHoBZwy+VvoRnYrUHcYCLei4C0y5WpObd4L8xxIvZKTal7enOAVylFkIuI3wW9vRJzdZsgWqjza1L6TQVJvEgND7d6h825btpZ/ZGmk4KkuKrfESZ5d62aP3omVSHRP/hgaq7hTaL7EjTqGqYReFes7r1ybVaTmo05sq2mLBxet8XMz8Zb7sTREdDrONi++kH7izNskdrxNmiyhelKtCm0fJb9g3jmhlBoeIX5FvJVOXuEFJc4Tuesrb3GzKn0QsXI7Oc+yeZYp7AU20WwxcgtoD1M64saY/OP8w3UjOAKAlBhfKbP9uYL2c93WSOZC9+Qt5NauczZZ8tmNKDMUeDPYoyxoyX7XY3H48ti7qnGflv9CxxHysaKSRyuVL9FTw7oQcxapys7RWrdZop6lRL/gZ4zZWNxOFhm/O6hCXfHldKOXKuBS/mGr45uafzC02dojwXPa9V8PUVJ+ZdeQLpkRG4AtGuwX3zjteARyQ/AUv15xt802rvAHuSZHK0YR6jKhK4EAhX5Ma8EL7sds2ZxpNOrBvfoLzsAiypqJKrIS6cJpHbM7zcFa28RByf+ADJfwQJoxUrBb8nI3gNkkEW27U6WfzcHeKpEcyHcPBu0EekknRxk+FzxB3cNTSzc1n5O2slm1u/tALjnTw52L7L+Es29w2XDWN9XuOTEWXbW4vH2P9JO2Gt3h4qqdRafOBzcFK28PLuADkNu22juTne7wxTgtD+G3oL/vFpzerOSx86rGeBve/KBWYEaJzwaP3T39rw9H7P6vf9dltz9yoQcd4X+hmDFddhks46ohjOso83bG6wHgJzIWX/u//IrwLjK+hCg1dqoIVxir1JVW6yKvUM9ggy5AP2kl51ad+SrRwuBf4hj64WzHPBHzzc061Ux0bm5u34gXXh2MjaL1weoasvr4qaRwuj7bBEwNwpo37G8wX1uP9Yb5Eqn8291NrWDjNm++SKf8PnXiDkk8xbS2OcU0mvgoJ3UwS3ArMcnZagV9m13+7Xl+yh8FfrR8zsTaD17c0NoNtZFZbg8EdvfCMd9vebbS8vdN65RnviBKNzZ/KMlbc6JzCPNn4gJMOlchyVkZjnrZGUbrEMvqDOydoHXC5A6geLeGnjOn4NPNBpm1bc+0xF1l7mQohGSwEw896aKOavdhSPRubEdKhes8at190HYoPXcMTMyhzED24jLM2WLrN9278HHf10dWYJC9dp8pWQL2fecdm1b+e1kGK+zV+ITN0nmpU54X2EEyt1r+FKfo1SRPl8Amzxk2+is0rVxpDtrU634p/TDKBdeVKZ2GjMRfozWxjbBVVgGuu4REgPuuIq9qqRXqkO+TdrBY35steoOpnw4X2iZBwnZXuu+vpYI/Au+txRX8vMFpQyjO+3OwsaaT8wfRjS0qWiSWNfHgwD4REzhow0+uxS5svio/bK1vIj4D/nuPJ+/heSHlvdg+rqkbLbqNl4hLP++wTfonxytul7m9tBgbbyStLUnJEo6oAqUbo6cwsJLNeqJGNh+yWapQJ6ZIrGttH8PgqHCnoVyGWtTSD7Gc/InL4KIs+QG934yezaVfL3oCtEts5UJfpne1NC6EViJmaZTNPdz4feJVx1jAuR3W4caTWWfWMHuiMMY7Vs+Mhhc81Wro3qecbTVxgmaUc4X7dx7lnUu1TYt7FhOUUb23H2uMY4PG2ejX8WE7OXXTANoaB9V4y9zj+0ErWaRs8R81ZQBcodcWPjWtpj/SopUFLHP6xkXZUzMH2OmlYOSe0lNfw1CGjovpjpyygtobdFOKfBvAQn7dNxH8GyqaQSuvZreGWAndZ9OpokW4U9wYNHafjXwymxcsLdfyd8qsMZ37bMu/Dlavb1SWKdOkqImtiI+R2Vi+/R94rJqRisZw7wfG2qDshksztBezhMPQDTwNvaqJAyNgkcW//2/0GfGmD02UVbtbGqPq1xo5jsegQpAdcQHYzuevwZPRjR+fk11j92dghH4tWSNwVIBNEUS/t0FV0S4dCcm+RKral+RTZGXSvabwVRBxIWzz+4Gr6nlT7LXVyE6iLtmyHlh30SzoxiJ232B7EquOWv9tiXlTyNVv4bVu88IAcQ6NYJzJr9BI1dstBPB8Zt8UpEu7l4FsPEA1aaXoNeEm1/JjVxC2WdoFqUivIv3HLh5h2J2+x5DdOIvicViKD79ENaRVd3hhct0CWt4iXqWlb4GonFZu+pXkCzvYzzD6Wn9iHbDvuN6G41K1i5xatk9+c5boxP3YvuQrkKTOt6OgSaqGt3zVOz4V0p02mQU9m+jmPgdsWcJifnSpuL5O5WmAYa9xWypq4Rr235eZCCufYvjwMqtYh4yndMVtB8MQy1SJgYtZC8/vlI430gPvmRWZ0Q5Mcsw5UoMkKFEm66SoVbALjEHOPOfQ8qFBTbvKiUFOU5litwdC53bkYafsO4xWZCRqv4SZLK7DymhwZkirS9IVIcWgTUBiNnZKKBGeRijZZ//tjWLgC1A0LXS9+uE8O/eel1ZG6QRC8baEYOd/apWJN7aS/UGU0lXJ+vOnOQKo4s8nRLJxR5mf+BHS7Yy5FTyaV1lM/oU36ZGW2ArwlVxzXlmoobah2pigcE7qAP4kWTfla39GVfs5qcDtLnPRtRjj3upxw4dTQeSsGezc6/w25XSV9MjrwVgMi8oMNsaNZ1ZtM3ThyxKXpfvsM7YJ82t/b3WdEzOvuwlMbVVVTrhd+i/mZmHgVdIjTFxxsJxmuoR5jnUtW4VjsItg6MFtd0eR0ES6/JNSVTY5xHaurmrqTN8aE8xE/sBppkzQU0ue95bOOuqbJidF/QV3XlC93srp+JKYMdRBoWsA8qOPRTNZ0CRdDHxqf58ldGSWvsVMaqA6K29RQ8zkbX89Ks2wJWL6WSkUurSYezX5s8NTi/FjGlK6l1pQVotOBvogAy+sz1Dwxy0x58+9y54m4MUCqTU35ThdJA4XDjql37H5WW5qcpTlpzv4a3HO3GTchTulHJ6tbiJ/pRFZVOagmuBFT8S3anDax2mCTTK5VTU1zeIIXwa2A33NMvAiBNdJ+QiZaeOnkPOFeLOnpa4PXyP6Cw24dxLTldi6Wdvsmtb3J0gDZxDFrKAQO32EXx2PpgepByXI4JJ3WwEHt8JwbnRcA95Tjnp7GIOEHqoBPOO4gkcl3St+9CNSQO8kZK1W9jhLPZ3Az+6HLwC2S7hBJ/4xzCiTubnOgqHqIxqhj6fLKYLk0EiH6Aymb1Dds+v4Q+KFjetAkIR+31Yd6Mdp+bwf1UZNjtwV1qql5lLOYscGW2+81TjU5xRC6j2nXVOe2BvxK4IkkL7mB3mFeceXIbC2oN7dWPF0GP2I6XOOM+TunxGrlHo6eM4uk8rZ2iOIeR0W2vuvkuxtf6XEuEGY9sdjWfPmxrTKyIXWxim8dzvuFytx6qIuAzLc24OibfXGlI59m+pIC7QDXV9jtNAf+QdUOiDsbeAWonK3NjjObtbOWW9jI2bo/N3XRsIu4l9LnOsNBYKJNX0xwbmScA86NtpcdbIPaiSS34ghLtAyIZQUiuZXG5bXFdl2i3D6aH0i1WVclsiKZxi1a9jaRuAVkr51WjVJ5W5t72drm5JBabQcoQR6dXMROCNHESSc4MHl/3nl4Ol9Vb8G4XACs89BVlYlUxjP5ZI/cSvM0dNrnavtubgN8zBXnGL9qaRwF4zGXlHVv9vlB+VvNDjIkJA+RrpDqWRB7nMraGcf21s0+tveyrlg9J9LhKIhrQrwvZHVF7mYl8TaJdBzUI1ILirTaAv/VRJ0zxN0CbeYsEKOuShxrFoLGbhV/i6y2b+/rnArL71xRC+69oMZvlWZrUGO3Wm0xsgj83OEcXPQXHck2FwHLMC3iQE5c0MhEUQpHJlLBxdKsAT6D/H8FsZyF0iwCrlpCY1aoIhBXJ86qDVvbWat0kjsOxtNbc3Ih7eVfgS/bKbtha3Mf+6T0cm51E5QZs3rImL8bnZO3q+Nq7dbpgdul/RE767Y+k5mKnJjJU9hZv1XndfMaxr9Zrd96hXkz4Amm+1F/vYeTihwyvs1wCPiQVOu2dsVFS8R0oEaHXwgYBGLT1iXxsiPbtBWXYDF4FHkjdIUVB084EGN9SH0Y+jD1YenD1oejj4A+XH0E9RHSh3IgGnYgRTQVg3EQTWhcn1S520vHrI5XvmKTsewB97jI365sar5LPXhP+5FlfVK89Efbow5DSi+Cjk8VdrYAX+z0HZQ9eY+KtIJUtNfOdiO1LsEIj2hn2DNNynd1T28p4dw6a6ESlTvVBy1SNHZadNsiaTVIVWrkYjccWYQplyzQH29RH+xqWyX0M94+AxyjWObBzcY6L1W2EW8aoCqRVA+rebYav6dNruhqYdtiTLBhkpy2R3m8R/VzTqoL97bNhWJjUQQcQAh0scQ/96ognY0L01CXu3epZfe3G7mk75Khr+TTQGeNtaZzqq3LioDW3q8UdcNNhjvXVNeHtsSxmPT7XJX5YPsqbR9g8RbkjpZntOAWUlGkeRiXQkrHUItMRXGYgZ4xVV2DR3611fRwM6iH9ra5ood2TN9MDnuCZtGn26j6IlOPpHkRoNHWT6jfmuPq0n0XjRmLqcjcbF4chbiethnW3xq0T4VpqfV5UN0R/Vq9+XD7K6ox1e1rRR7Xl3d2IOgCLs1FE/j9h/UKu+E6thabNGWT6mOQymX95Ose7ZjANVH+M19DStQ8qsfrpn5CuhmqO6iv97cbWYtnv48zVblJ3JmpNTL/F6y+W8092GHkc3pptTMGUNzwM7pYBQJvanMe0+K2gyqHW1sfmvqteaCyH2+n71zzL5OvC3kt/u4xDRrU4xozra2kdlepM33UpkNdZU47XHiJ3Fsu9SSaDnX0YEFm8/VKZLHYgs4beZ9r0PwGUUAFOunRiJv8rMu60rUPZNDLi65QQAZUX2TMb42X/ivWoe9b+oowYHmpb87tTF23F7EsLtXX2ICsfZjfiTovvQIkSNp333gF0gBD7fW9Jpc6Qr4hP+tCA/bLQpgrQIFhwLX/9eO1HT3zGsg3aXRHL4qVSzR1FJj6xqF8co/nZwPevtsratiyMaYxoED7w/qdvPs+uvtAga3uWySCwKd2ZgRKoab9gFTCplbg8XDoahsw6UE/8yTgzMWl4Jg39KdnOukv02cgS2HtEkjlOAYEsn/OplXoR57E+ywIxPUAAQPmDeKNfT0Xscilvl+ss8DdAKkOrtHYSLG1VALi7tlei43oWY9D16ABBxq8yFCgH7v1lRAStuFpI9U1ZECk1I/+jZ7SS1ebITUhuxW0VEZhf7J7ar554VYQCde7njZfw8MKmneQb/8OXfQWPEr8xRL1JLk85xUvcsngqHiZjD//PUlnDRAdXyYvqy1QgV6nC/I0+cYZoE7Uig2dEVD/5ibhSRfs/xIPTy/DVEEMjPdJz82gd3Vv+QsG6Mqbz5OfGerkmcMgeJpo/YEi03zum0G68oEPaNS41qmbrbxfeMsfnjUDiyy69Sq9m9rp2afJ/JH4HfLinSD4KvERisdfpf5S+4qtvXx4lPystd353qu9wBdg7SV+odzL/AKi7tPEQ/p4bimMdA0I4h7Br9/lJUohEnwB6Xr07Y43ddK9ppLoHhqgaxp6nvjp7V6o92dwfj6v2O5lTYBWIec94u+RjulNBr282HtedjfoqKL/0Yo9nXTpwufJ537khbtBm3BUf8h67ja6E/gA4Hnir/v4gSWm+S/tY7uBlzsSCsh5SvJnZFCXUQAGiLDwc6/sSBMezxdmdT8vZxTL89IXGYJDwjdEFwfSD3DPi2nm40WsWl+MQfCNyZARGCwiIgt4QamRMHxYup4+fkwO1KJjaK7Tq7JAxlsbXCH88Cqi+7bJbIOWTS4FaPEaBacLniBYDhHThbtUUNhoaRpvj/Yi/3Sia8GzKkgON3j1j15eLZhTRH8LLBxpcK3w3dfRs/tD1J4tfPt6iMTjOg0+wHWGl/8FFtjOCINvE/RNJbWiwGTjM5l6qGUDBHcJ3iJ44Diyn8oPxBoML+KSnG2MEJq1gWzL8zaLAZoC7hHBTw8WrwrWJ/mkHXW+654AOkPchYZqRgTXGby0ySvoCNHgBUHvZNLDKP4U+NAI/DozcMQw/hR8WviRq4l+6l0UClUa+LZBDxrUDQZobqjTgstP06mtUbXkBO8e56nVWKDi+qY3TvFvQ7ycF3WDvCBYsBfug3tDKfhB8COb6H7OpA8Er/53O1HdtNTPrHz6zazwgNnq/FVDK+gRiIpzgid1IynNp2z5i6BP2BfvCnoJaQlR0RMF7PdCuhpoIKqPbeBfD9PDNoWEho58RtA0pKOCTNN4QmDE5MwZvjgo1KsCvMKvhDAdT/zZX6T+V5vfq3RtrDOCX0gN1A3D6mpmBLo6S8UxTp8wwbrcoOUoTwlfmOwVvYdR6xvhy/tNL+9NEbF+EzQdeACWaxeRM8sUgn06gj6+IPU8bF2D6U+Y+proAHr99I/gX4SXsQioB5j/CCruo5FTfAgKnMwy8OJ/xSnGGhZO9FGTr3X87CdNr1U3iAZymNuzLGTWKdNhvmaFn9CqMN+hV03rnMwwPhesC5wEr/VMahMMfgj4g8mtJDusfjCBnkFOk9SXSG+BK6kFmtLiz8lrcwW9gg1/C76qgv6hqHYxfpA9VQrWpw7DcAHMvwmO646gfT/A//4x+xv8P0J0ui0=", "saves/ankiian_t5.state": "...", "saves/ankiian_t6.state": "...", "saves/ankiian_t7.state": "..."}

for path, b64 in BLOB.items():
    with open(os.path.join(ROOT, path), "wb") as f: f.write(zlib.decompress(base64.b64decode(b64)))
print("wrote", len(BLOB), "files:", ", ".join(sorted(BLOB)))

## 2. Loading saves

In [ ]:
import importlib, re, glob, collections
import state_v123 as S

def load_save(raw):
    """Decode a .state file (bytes) into the JSON layout the engine reads."""
    st = S.read_state(S.decompress(raw)); h = st["initial"]["header"]; cur = st["current"]["header"]
    return {"gameVersion": st["version"], "mapWidth": h["MapWidth"], "mapHeight": h["MapHeight"], "currentTurn": cur["CurrentTurn"],
            "header": {"MapHeaderInput": h}, "currentHeader": cur,
            "initialTiles": st["initial"]["tiles"], "initialPlayers": [p for p in st["initial"]["players"] if p["PlayerId"] != 255],
            "currentTiles": st["current"]["tiles"], "currentPlayers": [p for p in st["current"]["players"] if p["PlayerId"] != 255],
            "actions": st["actions"], "actionsProblem": st["actions_problem"], "unparsedBytes": st["unparsed_bytes"]}

def read_any(path):
    """Raw .state, or the iPhone text export (first line = device path, then base64)."""
    b = open(path, "rb").read()
    try:
        lines = b.decode().strip().splitlines()
        if lines and lines[0].strip().endswith(".state"): return base64.b64decode("".join(l.strip() for l in lines[1:]))
    except UnicodeDecodeError: pass
    return b

GAMES = {}
def add(path):
    d = load_save(read_any(path)); name = os.path.splitext(os.path.basename(path))[0]
    jp = os.path.join(ROOT, "saves", name + ".json"); json.dump(d, open(jp, "w")); GAMES[name] = jp
    ok = d["actionsProblem"] is None and d["unparsedBytes"] == 0
    print(f"{name:28s} '{d['header']['MapHeaderInput']['MapName']}' {d['mapWidth']}x{d['mapHeight']} turn {d['currentTurn']:3d} "
          f"actions {len(d['actions']):5d}  decoded {'fully' if ok else 'PARTLY: ' + str(d['actionsProblem'])}")

for p in sorted(glob.glob(os.path.join(ROOT, "saves", "*.state"))): add(p)

### Upload your own saves (optional)
Run this cell and pick one or more files: the iPhone text exports or `.state` files. Each one is decoded and added to `GAMES`.

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    for name, data in up.items():
        p = os.path.join(ROOT, "saves", name); open(p, "wb").write(data); add(p)
except ImportError:
    print("Not running in Colab: copy files into", os.path.join(ROOT, "saves"), "and call add(path)")

## 3. Inspect a game
Players (tribe, stars, score, cities, kills and losses) and what happened in the move history.

In [ ]:
E = json.load(open(os.path.join(ROOT, "verify", "game_enums.json")))
def summary(name):
    d = json.load(open(GAMES[name])); T = E["TribeType"]
    print(d["header"]["MapHeaderInput"]["MapName"], "| turn", d["currentTurn"], "| difficulty", d["header"]["MapHeaderInput"]["Difficulty"])
    print(f"{'player':>6} {'tribe':10s} {'you?':5s} {'stars':>5} {'score':>6} {'cities':>6} {'kills':>5} {'losses':>6}")
    for p in d["currentPlayers"]:
        print(f"{p['PlayerId']:>6} {T.get(str(p['Tribe']), str(p['Tribe'])):10s} {'YOU' if p['AutoPlay'] == 0 else '':5s} {p['Currency']:>5} {p['Score']:>6} {p['NumCities']:>6} {p['Kills']:>5} {p['Losses']:>6}")
    print("actions:", dict(collections.Counter(a["name"] for a in d["actions"]).most_common()))
summary("botwar_t15")

## 4. Engine replay vs checkpoints
The engine replays the turn-15 save's full move history from turn 0, stopping at each earlier export (turns 5 and 11) to compare every unit's HP, type, position and owner with what the game saved.

In [ ]:
import replay as RP; importlib.reload(RP)
st, reps = RP.run(GAMES["botwar_t15"], [GAMES["botwar_t5"], GAMES["botwar_t11"]])
for path, n, res, bad in reps:
    tot = sum(v for k, v in res.items() if k in ("HP exact", "HP differs"))
    print(f"{os.path.basename(path):18s} after {n:5d} actions: exact HP {res.get('HP exact', 0)}/{tot} units  | {dict(res)}")

### Turn-by-turn replay: Ankiian Mountain (Bardur, Massive, 16 players, exported every turn 1–7)
With a checkpoint every turn, any mismatch points to one turn's actions.

In [ ]:
importlib.reload(RP)
st2, reps2 = RP.run(GAMES["ankiian_t7"], [GAMES[f"ankiian_t{t}"] for t in range(1, 7)])
for path, n, res, bad in reps2:
    tot = sum(v for k, v in res.items() if k in ("HP exact", "HP differs"))
    print(f"turn {os.path.basename(path).split('_t')[-1][:-5]:>2}: exact HP {res.get('HP exact', 0)}/{tot}  | {dict(res)}" + (f"  <- {bad}" if bad else ""))

### Rule A/B tests
Flip one rule and rerun. The version that reproduces more of the saved HP is the real game rule. This is how the counter-attack question was settled.

In [ ]:
def score(**rules):
    importlib.reload(RP); RP.RULES.update(rules)
    _, reps = RP.run(GAMES["botwar_t15"], [GAMES["botwar_t5"], GAMES["botwar_t11"]])
    return [r[2].get("HP exact", 0) for r in reps]
print("current rules                     ", score())
print("counter-attack WITHOUT def. bonus ", score(retaliation_uses_bonus=False))
print("no move-in after a melee kill     ", score(melee_moves_in_on_kill=False))
importlib.reload(RP);

## 5. Combat calculator
The confirmed formula: damage = round(aF/(aF+dF) × attack × 4.5), with aF = attack × hp/max and dF = defense × hp/max × bonus (1.5 for a city with Fortify, forest + Archery or mountain + Climbing; 4 behind walls). The counter-attack uses the same dF.

In [ ]:
from stats import STATS
def battle(attacker, defender, a_hp=None, d_hp=None, bonus=1.0, a_vet=False, d_vet=False, distance=1):
    a, d = STATS[attacker], STATS[defender]
    amax, dmax = a["hp"] + 5 * a_vet, d["hp"] + 5 * d_vet; ah, dh = a_hp or amax, d_hp or dmax
    af = a["atk"] * ah / amax; df = d["def"] * dh / dmax * bonus; tot = af + df
    dmg = RP.rnd(af / tot * a["atk"] * 4.5); left = dh - dmg; ret = 0
    if left > 0 and "stiff" not in d["skills"] and "surprise" not in a["skills"] and distance <= d.get("range", 1):
        ret = RP.rnd(df / tot * d["def"] * 4.5)
    print(f"{attacker} ({ah}/{amax}) -> {defender} ({dh}/{dmax}, bonus {bonus}): deals {dmg}, defender left {max(left, 0)}"
          + (f", hits back {ret}, attacker left {ah - ret}" if left > 0 else " (killed)"))

battle("Warrior", "Warrior")
battle("Knight", "Archer", bonus=1.5)
battle("Swordsman", "Defender", bonus=4)          # defender behind city walls
battle("Catapult", "Warrior", distance=3)         # out of range: no counter-attack
battle("Shark", "Mermaid")
print("\nunits known:", ", ".join(sorted(STATS)))

## 6. Map generator
The save-first generator: AquaCrop, fixed capital pairs, real capital rings, per-tribe terrain and per-type village counts, all measured from 27 real saves. Set `MODE = 'wiki'` for the wiki-only version.

In [ ]:
import wiki_mapgen as W; importlib.reload(W)
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
COL = {"Field": "#c9d98a", "Forest": "#4f8a3c", "Mountain": "#8c8c8c", "Water": "#7fb8e6", "Ocean": "#2f6fae"}
GLY = {"AquaCrop": ("d", "#2ec4b6"), "Fruit": ("o", "#d6336c"), "Crop": ("s", "#e8b923"), "Game": ("^", "#7a4a1f"), "Metal": ("D", "#e0e0e0"),
       "Fish": ("o", "#ffffff"), "Starfish": ("*", "#ffd43b"), "Spores": ("s", "#9b59b6")}
def draw(g, ax, title=""):
    n = g["size"]
    for t in g["tiles"].values():
        x, y = t["x"], n - 1 - t["y"]; ax.add_patch(Rectangle((x, y), 1, 1, color=COL.get(t["terrain"], "#cccccc")))
        if t["resource"] in GLY: m, c = GLY[t["resource"]]; ax.plot(x + .5, y + .5, m, color=c, ms=4, mec="black", mew=0.4)
        if t["improvement"] == "Village": ax.add_patch(Rectangle((x + .15, y + .15), .7, .7, fill=False, ec="red" if t["capital_of"] else "black", lw=1.5))
        elif t["improvement"] == "Ruin": ax.text(x + .5, y + .5, "R", ha="center", va="center", fontsize=6, weight="bold")
    ax.set_xlim(0, n); ax.set_ylim(0, n); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=10)

MODE, SIZE, SEED = "measured", 16, 7
TRIBES = ["Imperius", "Bardur", "XinXi", "Kickoo"]
fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for ax, mt in zip(axes.flat, ["lakes", "drylands", "continents", "pangea", "archipelago", "waterworld"]):
    g = W.generate(mt, SIZE, TRIBES, seed=SEED, mode=MODE)
    land = sum(t["terrain"] not in ("Water", "Ocean") for t in g["tiles"].values()) / SIZE ** 2
    draw(g, ax, f"{mt} (land {land:.0%})")
fig.suptitle(f"Generated maps ({MODE} mode), {SIZE}x{SIZE}, seed {SEED}. Red = capital, black = village, R = ruin, teal = AquaCrop")
plt.tight_layout(); plt.show()

### Compare with a real map
The real turn-0 map from a save, drawn the same way.

In [ ]:
def real_map(name):
    d = json.load(open(GAMES[name])); ter = {int(k): v for k, v in E["TerrainType"].items()}; res = {int(k): v for k, v in E["ResourceType"].items()}
    n = d["mapWidth"]; tiles = {}
    for y, row in enumerate(d["initialTiles"]):
        for x, t in enumerate(row):
            imp = t["ImprovementType"] if t["ImprovementExists"] else None
            tiles[y * n + x] = {"x": x, "y": y, "terrain": ter.get(t["Terrain"]), "resource": res.get(t["ResourceType"]) if t["ResourceExists"] else None,
                                "improvement": {1: "Village", 2: "Ruin"}.get(imp), "capital_of": t["Capital"]}
    return {"size": n, "tiles": tiles}
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
draw(real_map("giants_of_shaza_t6"), axes[0], "REAL: Giants of Shaza (Normal Lakes)")
draw(W.generate("lakes", 16, ["Imperius", "Bardur", "Luxidoor", "Aquarion"], seed=3), axes[1], "GENERATED: Lakes, same tribes")
plt.tight_layout(); plt.show()

## Notes
- **Not done yet:** this is a checker, not a playable engine. It doesn't generate legal moves or simulate bots.
- **Where the replay stands:** turn 5 matches almost exactly; later turns drift, because of rare mechanics not modelled yet (ruin spawns, boats, healing, poison, action type 18).
- **Where things live:** every confirmed rule, with its evidence, is in the project ledger (`claude/ledger.json`).